# 1. Dependencies

## CUDA

In [1]:
use_cuda = True

In [2]:
import os
import sys

In [3]:
if use_cuda:
    _cuda_root = os.path.join(sys.prefix, "targets", "x86_64-linux")
    if os.path.isdir(os.path.join(_cuda_root, "include")):
        os.environ.setdefault("CUDA_PATH", _cuda_root)

    %load_ext cuml.accel

cuML: Accelerator installed.


## Common Libraries

In [4]:
import gc
import glob
import time
from contextlib import contextmanager
from typing import cast, Any

In [5]:
import json
import joblib

In [6]:
from joblib import Parallel, delayed, parallel_config
from joblib import parallel_config

In [7]:
import math
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs

## Plotting

In [8]:
import matplotlib.pyplot as plt

## Pre-Processing

In [9]:

from sklearn.preprocessing import LabelEncoder

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA

# from imblearn.over_sampling import SMOTE

## Models

### KNN

In [11]:
if use_cuda:
    from cuml.neighbors import KNeighborsClassifier, NearestNeighbors
else:
    from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

### SVM

In [12]:
if use_cuda:
    from cuml.svm import SVC
else:
    from sklearn.svm import SVC

### Random Forest

In [13]:
if use_cuda:
    from cuml.ensemble import RandomForestClassifier
else:
    from sklearn.ensemble import RandomForestClassifier

### Logistic Regression (Softmax)

In [14]:
import tensorflow as tf
from tensorflow import keras

# TensorFlow claims the whole card on its first allocation, which would leave nothing
# for the cuML models above. Growth has to be enabled before any GPU is initialised,
# so this belongs here rather than next to the model.
if use_cuda:
    for _gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(_gpu, True)
else:
    tf.config.set_visible_devices([], "GPU")

I0000 00:00:1789736131.742384    2210 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789736132.043692    2210 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789736133.811597    2210 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1789736134.225034    6573 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that

### XGBoost

In [15]:
from xgboost import XGBClassifier

# XGBoost ships one class for both backends - the card is chosen with the `device`
# constructor argument rather than by importing from somewhere else - so there is no
# `use_cuda` fork here the way there is for the three models above; `build_xgb` passes
# `device="cuda"` instead. That argument needs xgboost >= 2.0. `cuml.accel` does not
# patch XGBoost either, so this is the upstream estimator and not a proxy over it.

## Evaluation

In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

## Global Variables

### Path

In [17]:
PATH_CACHE = "cache"

In [18]:
PATH_FOLDER_CSV = "cse-cic-ids2018"
PATH_FOLDER_RAW = "data-raw"

In [19]:
PATH_FOLDER_NO_INF = "data-no-inf"

In [20]:
PATH_FOLDER_SPLIT_DATA = "data-split-feature"
PATH_FOLDER_SPLIT_LABEL = "data-split-label"

In [21]:
PATH_FOLDER_ENCODED_LABEL = "data-encoded-label"
PATH_LABEL_ENCODER = os.path.join(PATH_CACHE,"label-encoder.pkl")

In [22]:
PATH_IMPUTER = os.path.join(PATH_CACHE,"median-imputer.json")
PATH_FOLDER_IMPUTED = "data-imputed"

In [23]:
PATH_SCALER = os.path.join(PATH_CACHE,"standard-scaler.pkl")
PATH_FOLDER_SCALED = "data-scaled"

In [24]:
PATH_FOLDER_IPCA_TRANSFORMER = os.path.join(PATH_CACHE,"pca")
PATH_IPCA = "ipca-transformer.pkl"
PATH_FOLDER_IPCA = "data-ipca"

In [25]:
PATH_FOLDER_SMOTE_TRANSFORMER = os.path.join(PATH_CACHE, "smote")
PATH_SMOTE = "smote-transformer.pkl"

In [26]:
PATH_FOLDER_SMOTE = "data-smote"

In [27]:
PATH_FOLDER_MODEL = "trained-model"

In [28]:
PATH_FOLDER_PREDICTION_RESULT = "prediction-result"

In [29]:
PATH_FOLDER_SEARCH_RESULT = "hyperparameter-search"

### Others

In [30]:
LABEL_COLUMN = "Label"
PARQUET_FLOAT_DTYPE = "float32"

In [31]:
PREDICT_BY_CHUNK = 10_000
SEARCH_DEV_MIN_PER_CLASS = 200

## Helper Functions

## Shared Helpers for Model Selection

Both hyperparameter searches below need the same conversion helpers, the same
subsampler and the same `SEARCH_DEV_MIN_PER_CLASS` floor, so all of it lives here
rather than being duplicated per model. Anything a search reads as a *default argument*
in particular has to be defined above both searches: a default is evaluated when the
`def` cell runs, so a constant parked in the KNN section makes the SVM search fail with
a `NameError` unless the KNN section happens to have been run first.

`stratified_subsample` exists because neither model can be searched over the full
SMOTE'd train split. A brute-force KNN query costs `O(n_query * n_train)` and kernel
SVM training is between `O(n^2)` and `O(n^3)` in the number of training rows, so at
~11M train rows a single hyperparameter combination would take longer than the whole
study. Sampling *proportionally per class*, with a floor of `min_per_class` rows,
keeps the rare attack classes (e.g. SQL Injection) present in the search sample
instead of letting uniform random sampling drop them entirely.

The floor matters more than it looks. Purely proportional sampling gives a class that
holds 3 of ~100k dev rows exactly *one* row in the subsample, and macro F1 - the
selection criterion - then swings by a full 1/n_classes depending on whether that
single row happens to be classified correctly. Both searches therefore pass
`SEARCH_DEV_MIN_PER_CLASS` when subsampling dev, so every class contributes a stable
per-class score rather than a coin flip.

Note that the subsample is only used for **model selection**. The final chosen model
is refit and every reported metric is computed on the full, untouched dev/test splits,
so the subsampling never enters the numbers that go into the results chapter.

In [32]:
def to_features(df) -> np.ndarray:
    """polars DataFrame -> C-contiguous float32 array for cuML."""
    if isinstance(df, (pl.DataFrame, pl.Series)):
        df = df.to_numpy()
    return np.ascontiguousarray(df, dtype=np.float32)


def to_labels(s) -> np.ndarray:
    """polars Series -> int32 labels (cuML classifiers reject float labels)."""
    if isinstance(s, (pl.DataFrame, pl.Series)):
        s = s.to_numpy()
    return np.asarray(s).ravel().astype(np.int32)

In [33]:
def to_numpy(a):
    if hasattr(a, "to_numpy"):
        return np.asarray(a.to_numpy()).ravel()
    if hasattr(a, "get"):
        return a.get().ravel()
    return np.asarray(a).ravel()

In [34]:
def to_numpy_2d(a) -> np.ndarray:
    """cupy / cuDF / polars 2-D result -> 2-D numpy array.

    `to_numpy` above ravels its input, which is correct for a prediction vector but
    destroys the (n_query, k) shape of a `kneighbors` result.
    """
    if hasattr(a, "to_numpy"):
        return np.asarray(a.to_numpy())
    if hasattr(a, "get"):
        return np.asarray(a.get())
    return np.asarray(a)

In [35]:
def stratified_subsample(
    x,
    y,
    n_samples: int | None,
    random_state: int = 42,
    min_per_class: int = 1,
) -> tuple[np.ndarray, np.ndarray]:
    """Class-proportional row subsample as (float32 features, int32 labels).

    Passing `n_samples=None` (or a value at least as large as the split) returns the
    whole split unchanged, so the same call site works for both search and final fit.
    """
    labels = to_labels(y)
    total = labels.shape[0]

    if n_samples is None or n_samples >= total:
        return to_features(x), labels

    rng = np.random.default_rng(random_state)
    classes, counts = np.unique(labels, return_counts=True)

    quota = np.floor(counts / total * n_samples).astype(np.int64)
    quota = np.maximum(quota, np.minimum(min_per_class, counts))
    quota = np.minimum(quota, counts)

    selected = []
    for class_id, class_quota in zip(classes, quota):
        class_index = np.flatnonzero(labels == class_id)
        if class_quota < class_index.size:
            class_index = rng.choice(class_index, size=int(class_quota), replace=False)
        selected.append(class_index)

    selected = np.sort(np.concatenate(selected))

    mask = np.zeros(total, dtype=bool)
    mask[selected] = True
    subset = x.filter(pl.Series(mask)) if isinstance(x, pl.DataFrame) else x[mask]

    print(
        f"Subsampled {total:,} -> {int(mask.sum()):,} rows across {classes.size} classes"
    )
    return to_features(subset), labels[mask]

In [36]:
def dump_search_results(
    results: pd.DataFrame, name: str, folder: str = PATH_FOLDER_SEARCH_RESULT
) -> str:
    os.makedirs(folder, exist_ok=True)
    file_path = os.path.join(folder, name)
    results.to_csv(file_path, index=False)
    print(f"Saved search results to: {file_path}")
    return file_path

In [37]:
def get_best_hyperparameter(
    results: pd.DataFrame, score_column: str = "f1_macro"
) -> dict:
    """Pick the top row by `score_column`.

    macro F1 is the selection criterion rather than accuracy: the dev split keeps its
    real class balance (Benign dominates), so accuracy would be maximised by a model
    that predicts the rare attack classes badly.
    """
    best = results.sort_values(score_column, ascending=False).iloc[0]
    print(f"Best by {score_column}:")
    for key, value in best.items():
        print(f"  {key:>16} = {value}")
    return best.to_dict()

In [38]:
def get_all_file_names(source_folder_name: str, file_format: str = ""):
    """List file names in source_folder_name that end with `.{file_format}`, sorted alphabetically.

    Note: if file_format is left empty, this returns an empty list (files are only ever
    appended when a format is given), so a format should always be supplied.
    """
    files = []
    for file in os.listdir(source_folder_name):
        if len(file_format) > 0:
            if file.endswith(f".{file_format}"):
                files.append(file)
    return sorted(files)

In [39]:
def filter_file_names(list_file_names: list, filter_word: str):
    """Return only the file names that contain filter_word as a substring (e.g. a date like "2018-02-14")."""
    filtered_file_names = []
    for file_name in list_file_names:
        if filter_word in file_name:
            filtered_file_names.append(file_name)
    return filtered_file_names

In [40]:
def to_pandas_dataframe(data, template_column: list) -> pd.DataFrame:
    """Build a DataFrame from `data` and reindex its columns to match `template_column`."""
    df = pd.DataFrame(data)
    df = df.reindex(columns=template_column)
    return df

In [41]:
def get_split_parquet_files(source_folder_name: str, split_name: str) -> list[str]:
    """Return sorted paths to all Parquet files under `source_folder_name/split_name` (e.g. .../train)."""
    files = sorted(glob.glob(os.path.join(source_folder_name, split_name, "*.parquet")))
    if not files:
        raise FileNotFoundError(
            f"No Parquet files found in {os.path.join(source_folder_name, split_name)}"
        )
    print(f"Found {len(files)} {split_name} files")
    return files

In [42]:
def get_polars_data_frame_with_label(
    source_folder_name: str,
    split_name: str = "train",
    label_column: str = LABEL_COLUMN,
    downcast_to_float32: bool = True,
) -> tuple[pl.DataFrame, pl.Series]:

    label_column = label_column.lower()
    files = get_split_parquet_files(source_folder_name, split_name)

    lazy_frame = pl.concat(pl.scan_parquet(file) for file in files)
    if downcast_to_float32:
        lazy_frame = lazy_frame.with_columns(cs.numeric().cast(pl.Float32))

    dataframe = lazy_frame.collect(engine="streaming")
    if label_column not in dataframe.columns:
        raise ValueError(f"Label column '{label_column}' not found in dataframe.")
    x = dataframe.drop(label_column)
    y = dataframe[label_column]
    return x, y

In [43]:
def get_polars_data_frame_without_label(
    source_folder_name: str, split_name: str = "", downcast_to_float32: bool = True
) -> pl.DataFrame:
    if len(split_name) > 0:
        files = get_split_parquet_files(source_folder_name, split_name)
    else:
        files = get_all_file_names(source_folder_name)
    total = len(files)

    lazy_frames = []
    for i, file in enumerate(files, start=1):
        print(f"Processing [{i}/{total}] {file}...", end="\r", flush=True)
        lazy_frames.append(pl.scan_parquet(file))
    lazy_frame = pl.concat(lazy_frames, how="diagonal_relaxed")

    if downcast_to_float32:
        lazy_frame = lazy_frame.with_columns(cs.numeric().cast(pl.Float32))

    dataframe = lazy_frame.collect(engine="streaming")

    return dataframe

### Model Training

In [44]:
def check_openmp_threads() -> int:
    """Report the thread count that actually governs the KNN search.

    Once sklearn dispatches a brute-force search to `ArgKmin.compute()` it
    threads with OpenMP and ignores `n_jobs` entirely. If this prints 1 the
    search runs single-threaded and will take roughly `n_cores` times longer.
    """
    from sklearn.utils._openmp_helpers import _openmp_effective_n_threads

    n_threads = _openmp_effective_n_threads()
    print(f"OpenMP effective threads: {n_threads}")
    try:
        import threadpoolctl

        for info in threadpoolctl.threadpool_info():
            print(
                f"  {info['user_api']:>8} / {info['internal_api']:<12}"
                f" threads={info['num_threads']}"
            )
    except ImportError:
        print("  (pip install threadpoolctl for per-library detail)")
    return n_threads

In [45]:
def dump_trained_model(
    model, name: str, subfolder: str, folder: str = PATH_FOLDER_MODEL
):
    target_folder = os.path.join(folder, subfolder)
    os.makedirs(target_folder, exist_ok=True)
    file_path = os.path.join(target_folder, name)
    joblib.dump(model, file_path)
    return file_path

In [46]:
def free_gpu_memory() -> None:
    """Release the device memory a finished cuML model was holding.

    cuML allocates through RMM/CuPy pools that only hand memory back once the Python
    objects are actually collected, so `del model` alone is not enough: measured over
    the 9-combination grid the card grew by roughly 850 MiB per fit
    (970 -> 1,826 -> 2,682 -> ... -> 6,424 MiB) and never came down. Collecting and
    then draining the pools returns it to ~106 MiB after every combination.

    This is about staying inside the 8 GiB card on longer grids and bigger
    subsamples. It is *not* the fix for the "Working set has already been
    initialized!" failure - that one was reproduced with the card at 166 MiB, so it
    is a solver bug, not exhaustion. The search guards against it separately.
    """
    gc.collect()
    if not use_cuda:
        return
    try:
        import cupy as cp

        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        pass

In [47]:
@contextmanager
def step(message: str):
    """Announce a blocking stage, then tick it off when it returns.

    `SVC.fit` is one opaque call, so there is nothing inside it to count. Printing
    `message ...` *before* entering it and completing the line afterwards is enough to
    tell which stage is running and which stages are finished - no timer and no
    background thread, just the two things that are actually known.
    """
    print(f"  {message} ...", end="", flush=True)
    try:
        yield
    except BaseException:
        print(" failed")
        raise
    print(" done")

In [48]:
def progress_bar(done: int, total: int, label: str = "", width: int = 30) -> None:
    """Redraw a one-line bar. Driven by the caller's own loop, so no thread is needed.

    Ends the line on the final update so whatever prints next starts cleanly.
    """
    filled = width if total <= 0 else int(width * done / total)
    line = f"  [{'#' * filled}{'-' * (width - filled)}] {done:,}/{total:,}"
    if label:
        line += f" {label}"
    print(line.ljust(100), end="\n" if done >= total else "\r", flush=True)

### Model Evaluation

In [49]:
def _unique_int_labels(true, pred) -> list[int]:
    return sorted({int(x) for x in np.unique(true)} | {int(x) for x in np.unique(pred)})

In [50]:
def evaluate(pred, true, labels=None) -> pd.DataFrame:
    if labels is None:
        labels = _unique_int_labels(true, pred)
    return pd.DataFrame(
        [
            {
                "accuracy": accuracy_score(true, pred),
                "precision_macro": precision_score(
                    true, pred, labels=labels, average="macro", zero_division=0
                ),
                "recall_macro": recall_score(
                    true, pred, labels=labels, average="macro", zero_division=0
                ),
                "f1_macro": f1_score(
                    true, pred, labels=labels, average="macro", zero_division=0
                ),
            }
        ]
    )

In [51]:
def get_classification_report(
    pred, true, label_encoder_path: str = PATH_LABEL_ENCODER
) -> pd.DataFrame:
    labels = _unique_int_labels(true, pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        true, pred, labels=labels, average=None, zero_division=0
    )
    return pd.DataFrame(
        {
            "class": decode_labels(labels, label_encoder_path),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
    )

In [52]:
def get_confusion_matrix(
    pred, true, label_encoder_path: str = PATH_LABEL_ENCODER
) -> pd.DataFrame:
    labels = _unique_int_labels(true, pred)
    class_names = decode_labels(labels, label_encoder_path)
    cm = confusion_matrix(true, pred, labels=labels)
    return pd.DataFrame(cm, index=class_names, columns=class_names)

# 2. Pre-Processing

## 2.1. Change CSV to Parquet

In [ ]:
CHUNK_SIZE = 100000
COLUMNS_TO_DROP = {
    "flow id",
    "src ip",
    "source ip",
    "src port",
    "source port",
    "dst ip",
    "destination ip",
    "timestamp",
}

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.str.strip().str.lower()
    return df

In [ ]:
def drop_unwanted_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=COLUMNS_TO_DROP, errors="ignore")

In [ ]:
def dataframe_to_parquet(
    chunk: pd.DataFrame,
    target_folder_name: str,
    file_name: str,
    template_columns: list[str] | None = None,
):
    chunk = normalize_columns(chunk)
    chunk = drop_unwanted_columns(chunk)

    if template_columns is not None:
        chunk = chunk.reindex(columns=template_columns)

    label = LABEL_COLUMN.lower()
    for col in chunk.columns:
        if col != label:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float64")

    chunk = chunk[chunk[label].str.lower() != label]
    chunk = chunk.dropna(subset=[label])
    output_file = os.path.join(target_folder_name, file_name)
    chunk.to_parquet(output_file, engine="pyarrow", compression="snappy", index=False)

In [ ]:
def create_template_columns(files: list, source_folder_name: str):
    template_columns = []
    seen = set()
    for file_name in files:
        source_file = os.path.join(source_folder_name, file_name)
        cols = (
            pd.read_csv(source_file, nrows=0).columns.str.strip().str.lower().tolist()
        )
        for c in cols:
            if c not in seen and c not in COLUMNS_TO_DROP:
                seen.add(c)
                template_columns.append(c)
    return template_columns

In [ ]:
def get_template_columns(
    source_folder_name, template_column_path: str = "dataframe-index.pkl"
):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files, source_folder_name)
    joblib.dump(template_columns, template_column_path)

In [ ]:
def _convert_single_csv_file(
    source_folder_name: str,
    target_folder_name: str,
    file_name: str,
    template_columns: list[str],
    chunk_size: int,
):
    source_file = os.path.join(source_folder_name, file_name)
    base_name = os.path.splitext(file_name)[0]

    for chunk_number, chunk in enumerate(
        pd.read_csv(source_file, chunksize=chunk_size, low_memory=False), start=1
    ):
        dataframe_to_parquet(
            chunk,
            target_folder_name,
            f"{base_name}_{chunk_number:05d}.parquet",
            template_columns,
        )

In [ ]:
def convert_all_file_to_parquet(
    source_folder_name: str, target_folder_name: str, chunk_size: int = CHUNK_SIZE
):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files, source_folder_name)

    total = len(csv_files)
    os.makedirs(target_folder_name, exist_ok=True)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_convert_single_csv_file)(
                source_folder_name,
                target_folder_name,
                file_name,
                template_columns,
                chunk_size,
            )
            for file_name in csv_files
        )

    print(f"Finished processing {total} files.")

In [ ]:
convert_all_file_to_parquet(PATH_FOLDER_CSV, PATH_FOLDER_RAW, CHUNK_SIZE)

## 2.2 Exploratory Data Analysis (EDA)

## 2.3. Change Infinite Values to NaN


Infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) are converted to `NaN` values to ensure that they can be handled consistently during the subsequent missing-value imputation process. This transformation allows both originally missing values and invalid infinite values to be processed using the same imputation method.

In [ ]:
def calculate_inf_values(source_folder_name: str):
    parquet_files = get_all_file_names(source_folder_name, "parquet")
    total = len(parquet_files)
    count = 0
    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        df = pd.read_parquet(source_file)
        inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
        count += inf_count
    print("\n")
    print(f"Completed. Processed {total} files.")
    print(f"Found {count} inf values")

In [ ]:
def _change_inf_to_nan_single_file(
    source_folder_name: str, target_folder_name: str, file_name: str
):
    source_file = os.path.join(source_folder_name, file_name)
    target_file = os.path.join(target_folder_name, file_name)

    df = pd.read_parquet(source_file)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.to_parquet(target_file, index=False)


def change_inf_to_nan(source_folder_name: str, target_folder_name: str):
    os.makedirs(target_folder_name, exist_ok=True)

    parquet_files = get_all_file_names(source_folder_name, "parquet")
    total = len(parquet_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_change_inf_to_nan_single_file)(
                source_folder_name, target_folder_name, file_name
            )
            for file_name in parquet_files
        )

    print(f"Completed. Processed {total} files.")

In [ ]:
change_inf_to_nan(PATH_FOLDER_RAW, PATH_FOLDER_NO_INF)

### Evaluation

The dataset was cleaned by converting both positive infinite  and negative infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) to `NaN`. A total of **121,886 infinite values** were identified across **168 Parquet files** before the cleaning process. After the transformation, no infinite values remained in the dataset.

**Before cleaning:**

In [ ]:
calculate_inf_values(PATH_FOLDER_RAW)

**After cleaning:**

In [ ]:
calculate_inf_values(PATH_FOLDER_NO_INF)

This ensures that all infinite values are handled as missing values and can subsequently be processed during the missing-value imputation stage.


## 2.4. Train/Validation/Test Split

The CSE-CIC-IDS2018 dataset requires careful consideration when dividing the data into training, validation, and test sets. This is because the attack classes are not uniformly distributed across the dataset; instead, specific attack scenarios were conducted on particular dates. As a result, directly splitting the dataset based on individual days may cause some attack classes to be absent from one or more subsets.

The distribution of attack scenarios across the data collection dates is presented below:

| Date  | Attack(s)                           |
|-------|-------------------------------------|
| 14-02 | FTP-BruteForce, SSH-Bruteforce      |
| 15-02 | DoS-GoldenEye, DoS-Slowloris        |
| 16-02 | DoS-SlowHTTPTest, DoS-Hulk          |
| 20-02 | DDoS-LOIC-HTTP, DDoS-LOIC-UDP       |
| 21-02 | DDoS-LOIC-UDP, DDoS-HOIC            |
| 22-02 | Web Brute Force, XSS, SQL Injection |
| 23-02 | Web Brute Force, XSS, SQL Injection |
| 28-02 | Infiltration                        |
| 01-03 | Infiltration                        |
| 02-03 | Bot                                 |

This distribution indicates that several attack classes are associated with only one or a small number of collection dates. Therefore, assigning entire dates directly to the training, validation, or test set could result in certain attack classes being completely absent from the training data. Such a split would make the experiment evaluate unseen attack-class generalization rather than the intended robustness of the ML-IDS against input disturbances.

Therefore, the primary experiment uses a **stratified train/validation/test split based on the attack label**, ensuring that the attack classes are represented across the three subsets. The validation and test sets are kept separate from the training data to prevent information leakage during model development and final evaluation.

A separate day- or scenario-based split may subsequently be used as an additional experiment to evaluate the model's ability to generalize to traffic collected under different attack scenarios.


In [ ]:
bruteforce = "2018-02-14"
dos_golden = "2018-02-15"
dos_hulk = "2018-02-16"
ddos_http = "2018-02-20"
ddos_udp = "2018-02-21"
web_first = "2018-02-22"
web_second = "2018-02-23"
infiltration_first = "2018-02-28"
infiltration_second = "2018-03-01"
botnet = "2018-03-02"

In [ ]:
def create_train_dev_test_folder(
    source_folder_name: str,
    features_folder_name: str,
    labels_folder_name: str,
    target_day,
):

    parquet_files = get_all_file_names(source_folder_name, "parquet")
    parquet_files = filter_file_names(parquet_files, target_day)

    if not parquet_files:
        print(f"No Parquet files found for {target_day}")
        raise ValueError(f"No Parquet files found for {target_day}")
    print(f"Found {len(parquet_files)} files for {target_day}")

    split_names = ("train", "dev", "test")
    feature_folders = {}
    label_folders = {}
    for split_name in split_names:
        feature_folder = os.path.join(features_folder_name, split_name)
        label_folder = os.path.join(labels_folder_name, split_name)
        os.makedirs(feature_folder, exist_ok=True)
        os.makedirs(label_folder, exist_ok=True)
        feature_folders[split_name] = feature_folder
        label_folders[split_name] = label_folder

    return parquet_files, feature_folders, label_folders

In [ ]:
def _split_single_parquet_file(
    source_folder_name: str,
    file_name: str,
    feature_folders: dict[str, str],
    label_folders: dict[str, str],
    label_column: str,
    dev_size: float,
    test_size: float,
    random_state: int,
):
    source_file = os.path.join(source_folder_name, file_name)
    df = pd.read_parquet(source_file)

    train_df, temp_df = train_test_split(
        df,
        test_size=dev_size + test_size,
        random_state=random_state,
        stratify=df[label_column],
    )
    dev_df, test_df = train_test_split(
        temp_df,
        test_size=test_size / (dev_size + test_size),
        random_state=random_state,
        stratify=temp_df[label_column],
    )

    for split_name, split_df in (
        ("train", train_df),
        ("dev", dev_df),
        ("test", test_df),
    ):
        labels = split_df[[label_column]]
        features = split_df.drop(columns=[label_column])

        features.to_parquet(
            os.path.join(feature_folders[split_name], file_name), index=False
        )
        labels.to_parquet(
            os.path.join(label_folders[split_name], file_name), index=False
        )

In [ ]:
def split_parquet_files(
    source_folder_name: str,
    parquet_files: list[str],
    feature_folders: dict[str, str],
    label_folders: dict[str, str],
    label_column: str,
    dev_size: float,
    test_size: float,
    random_state: int,
):
    total = len(parquet_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_split_single_parquet_file)(
                source_folder_name,
                file_name,
                feature_folders,
                label_folders,
                label_column,
                dev_size,
                test_size,
                random_state,
            )
            for file_name in parquet_files
        )

    print(f"Completed splitting {total} files.")

In [ ]:
def split_a_single_day(
    source_folder_name: str,
    features_folder_name: str,
    labels_folder_name: str,
    target_day: str,
    train_size: float = 0.60,
    dev_size: float = 0.20,
    test_size: float = 0.20,
    label_column: str = LABEL_COLUMN.lower(),
    random_state: int = 42,
):
    if min(train_size, dev_size, test_size) < 0.0:
        raise ValueError("train_size, dev_size, test_size must be non-negative")
    if not math.isclose(train_size + dev_size + test_size, 1.0, abs_tol=1e-9):
        raise ValueError("train_size + dev_size + test_size must equal 1.0")

    parquet_files, feature_folders, label_folders = create_train_dev_test_folder(
        source_folder_name, features_folder_name, labels_folder_name, target_day
    )

    split_parquet_files(
        source_folder_name,
        parquet_files,
        feature_folders,
        label_folders,
        label_column,
        dev_size,
        test_size,
        random_state,
    )

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, bruteforce)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, dos_golden)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, dos_hulk)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, ddos_http)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, ddos_udp)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, web_first)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, web_second)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, infiltration_first)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, infiltration_second)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, botnet)

Files are successfully split.

## 2.5. Imputation

Missing values are filled using **per-column medians computed from the training split only**, then applied to train/dev/test. Using the median (rather than the mean) avoids distortion from the heavy outliers typical of network flow features (e.g. flow duration, byte counts). Fitting the medians on the training split only and reusing them for dev/test prevents information leakage from the validation/test sets into preprocessing.

### Create Imputation

imputation for NaN

In [ ]:
def get_numeric_columns(combined: pl.DataFrame) -> list[str]:
    schema = combined.collect_schema()

    numeric_cols = []
    for column, dtype in zip(schema.names(), schema.dtypes()):
        if dtype.is_numeric():
            numeric_cols.append(column)

    print(f"Found {len(numeric_cols)} numeric columns")
    return numeric_cols

In [ ]:
def compute_medians(combined: pl.DataFrame, numeric_cols: list[str]) -> pl.DataFrame:
    median_exprs = []
    total = len(numeric_cols)

    for i, column in enumerate(numeric_cols, start=1):
        print(f"Processing [{i}/{total}] {column}...", end="\r", flush=True)
        expr = pl.col(column).median().alias(column)
        median_exprs.append(expr)

    medians = combined.select(median_exprs)
    return medians

In [ ]:
def build_median_dict(
    medians: pl.DataFrame, numeric_cols: list[str]
) -> dict[str, float]:
    median_dict = {}
    null_columns = []

    for c in numeric_cols:
        val = medians[c][0]
        if val is None:
            null_columns.append(c)
            median_dict[c] = 0.0
        else:
            median_dict[c] = float(val)

    if null_columns:
        print(
            f"Warning: {len(null_columns)} columns had no non-null values, defaulted to 0.0: {null_columns}"
        )

    return median_dict

In [ ]:
def save_median_dict(median_dict: dict[str, float], output_file: str):
    with open(output_file, "w") as file:
        json.dump(median_dict, file, indent=4)
    print(f"Saved medians to: {output_file}")

In [ ]:
def get_median_imputation(
    source_folder_name: str, output_file: str
) -> dict[str, float]:
    df = get_polars_data_frame_without_label(source_folder_name, "train")
    numeric_cols = get_numeric_columns(df)
    medians = compute_medians(df, numeric_cols)
    median_dict = build_median_dict(medians, numeric_cols)
    save_median_dict(median_dict, output_file)
    return median_dict

In [ ]:
median = get_median_imputation(PATH_FOLDER_SPLIT_DATA, PATH_IMPUTER)
print(json.dumps(median, indent=4))

### Impute Training Data

In [ ]:
def impute_dataframe(df: pd.DataFrame, medians):
    for column, median in medians.items():
        if column in df.columns:
            df[column] = df[column].fillna(median)
    return df

In [ ]:
def _impute_single_file(
    source_file: str, output_split_folder: str, medians: dict[str, float]
):
    df = pd.read_parquet(source_file)
    df = impute_dataframe(df, medians)
    output_file = os.path.join(output_split_folder, os.path.basename(source_file))
    df.to_parquet(output_file, engine="pyarrow", compression="snappy", index=False)


def impute_all_split_files(
    source_folder_name: str, output_folder_name: str, medians_path: str
):
    with open(medians_path, "r") as f:
        medians = json.load(f)

    for split_name in ("train", "dev", "test"):
        files = get_split_parquet_files(source_folder_name, split_name)
        total = len(files)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)

        with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_impute_single_file)(file, output_split_folder, medians)
                for file in files
            )

        print(f"Finished imputing {total} {split_name} files.")

In [ ]:
impute_all_split_files(PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_IMPUTED, PATH_IMPUTER)

## 2.6. Label Encoding

### Create Label Encoder

The string attack labels written by the train/dev/test split (`PATH_FOLDER_SPLIT_LABEL`) are mapped to integer class ids with `LabelEncoder`. As with every other transformer in this pipeline the encoder is fit on the **train split only** and then reused for dev/test, so the class-id mapping never depends on data the model is evaluated on. The fitted encoder is pickled to `PATH_LABEL_ENCODER` so predictions can later be turned back into human-readable labels with `decode_labels` / `encoder.inverse_transform`.

In [ ]:
def create_label_encoder(
    labels_folder_name: str, split_name: str = "train"
) -> LabelEncoder:
    labels = get_polars_data_frame_without_label(
        labels_folder_name, split_name
    ).to_series()

    encoder = LabelEncoder()
    encoder.fit(labels.to_numpy())

    print(
        f"Fitted LabelEncoder on {labels.len()} '{split_name}' rows; {len(encoder.classes_)} classes:"
    )
    for class_id, class_name in enumerate(encoder.classes_):
        print(f"  {class_id:>2} -> {class_name}")
    return encoder

In [ ]:
def dump_label_encoder(encoder: LabelEncoder, output_path: str = PATH_LABEL_ENCODER):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    joblib.dump(encoder, output_path)
    print(f"Saved fitted LabelEncoder to: {output_path}")

In [ ]:
def get_label_encoder(
    labels_folder_name: str, output_path: str = PATH_LABEL_ENCODER
) -> LabelEncoder:
    encoder = create_label_encoder(labels_folder_name)
    dump_label_encoder(encoder, output_path)
    return encoder

In [ ]:
label_encoder = get_label_encoder(PATH_FOLDER_SPLIT_LABEL)

### Encode Labels

The fitted encoder is applied to every split, writing one integer-label Parquet file per input file into `PATH_FOLDER_ENCODED_LABEL/{train,dev,test}`. The `label` column name is kept unchanged, so the encoded folder is a drop-in replacement for `PATH_FOLDER_SPLIT_LABEL` in the downstream data-loading helpers.

In [176]:
def load_label_encoder(path: str = PATH_LABEL_ENCODER) -> LabelEncoder:
    return joblib.load(path)

In [76]:
def _encode_single_label_file(
    file_path: str, encoder: LabelEncoder, output_split_folder: str
):
    label_column = LABEL_COLUMN.lower()
    labels = pd.read_parquet(file_path)[label_column]

    encoded_df = pd.DataFrame({label_column: encoder.transform(labels.to_numpy())})
    encoded_df.to_parquet(
        os.path.join(output_split_folder, os.path.basename(file_path)),
        engine="pyarrow",
        compression="snappy",
        index=False,
    )


In [77]:
def encode_split_labels(
    labels_folder_name: str, encoder_path: str, output_folder_name: str
):
    encoder = load_label_encoder(encoder_path)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(labels_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_encode_single_label_file)(
                    file_path, encoder, output_split_folder
                )
                for file_path in file_paths
            )

        print(f"Finished encoding {total} {split_name} label files.")

In [103]:
encode_split_labels(PATH_FOLDER_SPLIT_LABEL, PATH_LABEL_ENCODER, PATH_FOLDER_ENCODED_LABEL)

Found 168 train files


/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    2.6s


Finished encoding 168 train label files.
Found 168 dev files
Finished encoding 168 dev label files.
Found 168 test files
Finished encoding 168 test label files.


[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    2.8s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 155 out of 168 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 155 out of 168 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    0.1s finished


### Decode Predictions

`decode_labels` loads the pickled encoder and maps integer class ids (model predictions, or any encoded label column) back to their original human-readable strings.

In [86]:
def decode_labels(encoded, encoder_path: str = PATH_LABEL_ENCODER) -> np.ndarray:
    encoder = load_label_encoder(encoder_path)
    return encoder.inverse_transform(np.asarray(encoded))

## 2.7. Normalized or Feature Scaling

Features are standardized (zero mean, unit variance) using `StandardScaler`. As with imputation, the scaler is fit with `partial_fit` on the **train split only** (processed file-by-file to avoid loading the full ~16M-row dataset into memory at once), then reused to transform train/dev/test. This keeps dev/test statistically unseen during fitting and puts every feature on a comparable scale, which PCA and distance/gradient-based models both depend on.

### Create Scaler

In [ ]:
def fit_scaler_on_files(
    scaler: StandardScaler, file_paths: list[str]
) -> StandardScaler:
    total = len(file_paths)
    for i, file_path in enumerate(file_paths, start=1):
        print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
        df = pd.read_parquet(file_path)

        if df.empty:
            print(f"Empty file found: {file_path}")
            continue

        scaler.partial_fit(df)
    return scaler

In [ ]:
def create_standard_scaler(
    source_folder_name: str, split_name: str = "train"
) -> StandardScaler:
    file_paths = get_split_parquet_files(source_folder_name, split_name)
    scaler = StandardScaler()
    scaler = fit_scaler_on_files(scaler, file_paths)

    if not hasattr(scaler, "mean_"):
        raise ValueError("The scaler could not be fitted because all files were empty.")

    print(" " * 100, end="\r")
    print(
        f"Successfully fitted scaler using {len(file_paths)} Parquet file(s) from '{split_name}'."
    )

    return scaler

In [ ]:
def dump_standard_scaler(scaler: StandardScaler, output_path: str):
    joblib.dump(scaler, output_path)

In [ ]:
def get_standard_scaler(source_folder_name: str, output_path: str):
    scaler = create_standard_scaler(source_folder_name)
    dump_standard_scaler(scaler, output_path)

In [ ]:
get_standard_scaler(PATH_FOLDER_IMPUTED, PATH_SCALER)

### Scale Training Data

In [79]:
def load_standard_scaler(path: str) -> StandardScaler:
    return joblib.load(path)

In [ ]:
def _scale_single_file(
    file_path: str, scaler: StandardScaler, output_split_folder: str
):
    df = pd.read_parquet(file_path)
    columns = df.columns.tolist()

    data = scaler.transform(df).astype(PARQUET_FLOAT_DTYPE, copy=False)
    scaled_df = pd.DataFrame(data, columns=columns)

    scaled_df.to_parquet(
        os.path.join(output_split_folder, os.path.basename(file_path)),
        engine="pyarrow",
        compression="snappy",
        index=False,
    )

In [ ]:
def scale_split_files(
    source_folder_name: str, scaler_path: str, output_folder_name: str
):
    scaler = load_standard_scaler(scaler_path)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(source_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        with parallel_config(backend="loky", inner_max_num_threads=1):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_scale_single_file)(file_path, scaler, output_split_folder)
                for file_path in file_paths
            )

        print(f"Finished scaling {total} {split_name} files.")

In [ ]:
scale_split_files(PATH_FOLDER_IMPUTED, PATH_SCALER, PATH_FOLDER_SCALED)

## 2.8. Principal Component Analysis (PCA)

`IncrementalPCA` is used instead of standard `PCA` because it supports `partial_fit` on mini-batches, so the scaled dataset never needs to be fully loaded into memory. Transformers are fit for a range of component counts (5-40) so the explained-variance-vs-components trade-off can be inspected before committing to a final value.

### Creating Incremental Principal Component Analysis

In [ ]:
LIST_PC_COMPONENTS = [5,10,15,20,25,30,35,40]

In [ ]:
def fit_ipca_on_files(ipca: IncrementalPCA, file_paths: list[str]) -> IncrementalPCA:
    total = len(file_paths)
    for i, file_path in enumerate(file_paths, start=1):
        print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
        df = pd.read_parquet(file_path)

        if df.empty:
            print(f"Empty file found: {file_path}")
            continue

        ipca.partial_fit(df)
    print(" " * 100, end="\r")
    return ipca

In [ ]:
def create_ipca(
    n_components, source_folder_name: str, split_name: str = "train"
) -> IncrementalPCA:
    print(f"Creating IPCA transfromer with {n_components} pc")
    file_paths = get_split_parquet_files(source_folder_name, split_name)
    ipca = IncrementalPCA(n_components=n_components)
    ipca = fit_ipca_on_files(ipca, file_paths)
    print(f"Successfully fitted ipca with {n_components} pc from '{split_name}'.")

    return ipca

In [107]:
def get_ipca_transformer_path(pc: int, ipca_path: str) -> str:
    return f"{pc}-pc-{ipca_path}"

In [ ]:
def dump_ipca(
    ipca: IncrementalPCA,
    transformer_folder: str = PATH_FOLDER_IPCA_TRANSFORMER,
    ipca_path: str = PATH_IPCA,
):
    filename = get_ipca_transformer_path(ipca.n_components_, ipca_path)
    joblib.dump(ipca, os.path.join(transformer_folder, filename))

In [ ]:
def get_ipca(list_pc_components: list, source_folder_name: str, output_path: str):
    os.makedirs(PATH_FOLDER_IPCA_TRANSFORMER, exist_ok=True)
    for pc in list_pc_components:
        ipca = create_ipca(pc, source_folder_name)
        dump_ipca(ipca, PATH_FOLDER_IPCA_TRANSFORMER, output_path)

In [ ]:
get_ipca(LIST_PC_COMPONENTS,PATH_FOLDER_SCALED, PATH_IPCA)

### Evaluate Incremental Principal Component Analysis

In [93]:
def load_ipca(
    pc: int,
    transformer_folder: str = PATH_FOLDER_IPCA_TRANSFORMER,
    ipca_path: str = PATH_IPCA,
) -> IncrementalPCA:
    filename = get_ipca_transformer_path(pc, ipca_path)
    return joblib.load(os.path.join(transformer_folder, filename))

In [ ]:
def evaluate_ipca_variance(list_pc_components: list) -> pd.DataFrame:
    rows = []
    for pc in list_pc_components:
        ipca = load_ipca(pc)

        cum_var = np.cumsum(ipca.explained_variance_ratio_)
        rows.append(
            {
                "n_components": pc,
                "total_explained_variance": cum_var[-1],
                "explained_variance_ratio": ipca.explained_variance_ratio_,
            }
        )

    return pd.DataFrame(rows)

In [ ]:
def plot_ipca_variance(evaluation_results: pd.DataFrame):
    plt.figure(figsize=(8, 5))

    x = evaluation_results["n_components"]
    y = evaluation_results["total_explained_variance"]
    plt.plot(x, y, marker="o", color="black")
    for xi, yi in zip(x, y):
        plt.annotate(
            f"{yi:.3f}",
            (xi, yi),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
        )

    plt.axhline(0.95, color="red", linestyle="--", label="95% threshold")
    plt.xlabel("Principal Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("IPCA: Variance Retained vs. Principal Components")
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
ipca_variance_results = evaluate_ipca_variance(LIST_PC_COMPONENTS)
plot_ipca_variance(ipca_variance_results)

### Implement Incremental Principal Component Analysis

`IPCA_PC_USED = 25` was chosen from the variance plot above as the smallest evaluated component count that clears the 95% cumulative explained-variance threshold.

In [65]:
IPCA_PC_USED = 25

In [ ]:
def _ipca_transform_single_file(
    file_path: str, ipca: IncrementalPCA, output_split_folder: str
):
    df = pd.read_parquet(file_path)

    data = ipca.transform(df).astype(PARQUET_FLOAT_DTYPE, copy=False)
    pc_columns = [f"pc{i+1}" for i in range(data.shape[1])]
    scaled_df = pd.DataFrame(data, columns=pc_columns)

    scaled_df.to_parquet(
        os.path.join(output_split_folder, os.path.basename(file_path)),
        engine="pyarrow",
        compression="snappy",
        index=False,
    )

In [ ]:
def ipca_split_files(source_folder_name: str, ipca_path: str, output_folder_name: str):
    ipca = load_ipca(IPCA_PC_USED)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(source_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        with parallel_config(backend="loky", inner_max_num_threads=1):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_ipca_transform_single_file)(
                    file_path, ipca, output_split_folder
                )
                for file_path in file_paths
            )

        print(f"Finished scaling {total} {split_name} files.")

In [ ]:
ipca_split_files(PATH_FOLDER_SCALED, PATH_IPCA, PATH_FOLDER_IPCA)

## 2.9. Synthetic Minor Oversampling Technique

CSE-CIC-IDS2018 is heavily imbalanced (Benign traffic dominates; some attack classes are a small fraction of a percent). SMOTE is applied to the **training split only** (never dev/test, so evaluation still reflects real-world class balance) to synthesize minority-class samples and reduce the model's bias toward the majority class.

SMOTE resamples the **PCA-reduced train features** (`PATH_FOLDER_IPCA`, paired with the original `PATH_FOLDER_SPLIT_LABEL` labels), since that's the final feature representation a downstream model would train on. Fitting concatenates every train file via polars (`load_train_features_and_labels`), since imbalanced-learn needs the full split in memory to compute global class counts. Once fitted, the transformer is applied **one file at a time** (`load_smote`, `resample_with_smote`, `save_smote_train_data`) using pandas instead - matching the streaming approach used for IPCA/normalization elsewhere - so resampling never needs the whole split in memory at once; imbalanced-learn only accepts numpy arrays, so the conversion happens right at the `fit_resample` call and the result is wrapped back into a `pd.DataFrame`/`pd.Series`.

Because some attack classes (e.g. SQL Injection) have very few samples in this dataset, `fit_smote` clamps `k_neighbors` down to what the smallest class can support instead of letting `fit_resample` raise on the default `k_neighbors=5`. Fitting is kept separate from resampling: `fit_smote` fits and `dump_smote` persists the fitted transformer via joblib right away, and `resample_with_smote` performs the actual oversampling per file afterwards. Two things can still go wrong at the per-file level that the global fit can't see: (1) a class that's fine globally can be scarce in one specific chunk (e.g. one file has only 5 `Benign` rows against 59,995 `DDOS attack-HOIC` rows), so `resample_with_smote` re-checks `k_neighbors` against that file's own smallest class and clamps further if needed; (2) since each day's capture is chunked in original time order, most individual chunks fall entirely outside the attack window and end up **100% Benign** (123 of the 168 train files, in practice) - `fit_resample` requires at least 2 classes, so `resample_with_smote` detects a single-class file and passes it through unresampled rather than erroring.

### Create SMOTE

In [ ]:
def load_train_features_and_labels(
    features_folder_name: str, labels_folder_name: str, split_name: str = "train"
) -> tuple[pl.DataFrame, pl.Series]:
    feature_files = get_split_parquet_files(features_folder_name, split_name)
    label_files = get_split_parquet_files(labels_folder_name, split_name)

    print(
        f"Loading {len(feature_files)} '{split_name}' feature file(s)...",
        end="",
        flush=True,
    )
    features_df = pl.concat([pl.scan_parquet(f) for f in feature_files]).collect()
    labels_df = (
        pl.concat([pl.scan_parquet(f) for f in label_files])
        .select(LABEL_COLUMN.lower())
        .collect()
    )

    print(f"Loaded {features_df.height} rows for '{split_name}'.")
    return features_df, labels_df.to_series()

In [ ]:
def get_class_count(labels: pl.Series) -> dict:
    counts = labels.value_counts()
    return dict(zip(counts[labels.name].to_list(), counts["count"].to_list()))

In [ ]:
def fit_smote(
    features: pl.DataFrame, labels: pl.Series, random_state: int = 42
) -> SMOTE:
    class_counts = get_class_count(labels)
    smallest_class_count = min(class_counts.values())
    k_neighbors = max(1, min(5, smallest_class_count - 1))

    if k_neighbors < 5:
        print(
            f"Warning: smallest class has {smallest_class_count} samples; reducing k_neighbors to {k_neighbors}"
        )

    smote = SMOTE(random_state=random_state, k_neighbors=k_neighbors)
    with parallel_config(n_jobs=-1):
        smote.fit(features.to_numpy(), labels.to_numpy())
    return smote

In [ ]:
def dump_smote(
    smote: SMOTE,
    transformer_folder: str = PATH_FOLDER_SMOTE_TRANSFORMER,
    smote_path: str = PATH_SMOTE,
):
    os.makedirs(transformer_folder, exist_ok=True)
    output_path = os.path.join(transformer_folder, smote_path)
    joblib.dump(smote, output_path)
    print(f"Saved fitted SMOTE transformer to: {output_path}")

In [ ]:
def get_smote_transformer(features_folder_name: str, labels_folder_name: str):
    features, labels = load_train_features_and_labels(
        features_folder_name, labels_folder_name
    )
    smote = fit_smote(features, labels)
    dump_smote(smote)

In [ ]:
get_smote_transformer(PATH_FOLDER_IPCA, PATH_FOLDER_ENCODED_LABEL)

### Implement SMOTE

In [ ]:
def load_smote(
    transformer_folder: str = PATH_FOLDER_SMOTE_TRANSFORMER,
    smote_path: str = PATH_SMOTE,
) -> SMOTE:
    return joblib.load(os.path.join(transformer_folder, smote_path))

In [ ]:
def resample_with_smote(
    smote: SMOTE, features: pd.DataFrame, labels: pd.Series
) -> tuple[pd.DataFrame, pd.Series]:
    if labels.nunique() < 2:
        return features, labels

    smallest_class_count = labels.value_counts().min()
    k_neighbors = max(1, min(smote.k_neighbors, smallest_class_count - 1))

    if k_neighbors < smote.k_neighbors:
        print(
            f"Warning: smallest class in this file has {smallest_class_count} samples; reducing k_neighbors to {k_neighbors}"
        )
        smote = SMOTE(
            random_state=smote.random_state,
            k_neighbors=k_neighbors,
            sampling_strategy=smote.sampling_strategy,
        )

    features_resampled, labels_resampled = cast(
        tuple[np.ndarray, np.ndarray],
        smote.fit_resample(features.to_numpy(), labels.to_numpy()),
    )
    resampled_features = pd.DataFrame(
        features_resampled, columns=features.columns
    ).astype(PARQUET_FLOAT_DTYPE)
    return resampled_features, pd.Series(labels_resampled, name=labels.name)

In [ ]:
def save_smote_train_data(
    features: pd.DataFrame,
    labels: pd.Series,
    file_name: str,
    output_folder_name: str = PATH_FOLDER_SMOTE,
):
    output_split_folder = os.path.join(output_folder_name, "train")
    os.makedirs(output_split_folder, exist_ok=True)

    output_df = features.assign(**{LABEL_COLUMN.lower(): labels.to_numpy()})
    output_df.to_parquet(
        os.path.join(output_split_folder, file_name),
        engine="pyarrow",
        compression="snappy",
        index=False,
    )

In [ ]:
def _smote_resample_single_file(
    feature_file_path: str, label_file_path: str, smote: SMOTE, output_folder_name: str
):
    features = pd.read_parquet(feature_file_path)
    labels = pd.read_parquet(label_file_path)[LABEL_COLUMN.lower()]

    features_resampled, labels_resampled = resample_with_smote(smote, features, labels)
    save_smote_train_data(
        features_resampled,
        labels_resampled,
        os.path.basename(feature_file_path),
        output_folder_name,
    )

In [ ]:
def smote_resample_files(
    features_folder_name: str,
    labels_folder_name: str,
    output_folder_name: str = PATH_FOLDER_SMOTE,
):
    smote = load_smote()

    feature_files = get_split_parquet_files(features_folder_name, "train")
    label_files = get_split_parquet_files(labels_folder_name, "train")
    total = len(feature_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_smote_resample_single_file)(
                feature_file_path, label_file_path, smote, output_folder_name
            )
            for feature_file_path, label_file_path in zip(feature_files, label_files)
        )

    print(f"Finished SMOTE-resampling {total} train file(s).")

In [ ]:
smote_resample_files(PATH_FOLDER_IPCA, PATH_FOLDER_ENCODED_LABEL, PATH_FOLDER_SMOTE)

### Evaluate SMOTE

In [ ]:
def get_class_count_from_label(source_folder_path: str):
    label_files = get_split_parquet_files(source_folder_path, "train")
    labels = (
        pl.concat([pl.scan_parquet(f) for f in label_files])
        .select(LABEL_COLUMN.lower())
        .collect()
    )
    return get_class_count(labels.to_series()), len(labels)

In [ ]:
before_class_count, before_row_count = get_class_count_from_label(
    PATH_FOLDER_ENCODED_LABEL
)
print(f"Row count: {before_row_count}")
print(f"Class count:\n{json.dumps(before_class_count, indent=4)}")

In [ ]:
after_class_count, after_row_count = get_class_count_from_label(PATH_FOLDER_SMOTE)
print(f"Row count: {after_row_count}")
print(f"Class count:\n{json.dumps(after_class_count, indent=4)}")

In [ ]:
def plot_smote_comparison(
    before_class_count: dict[str, int],
    after_class_count: dict[str, int],
) -> None:
    classes = sorted(set(before_class_count) | set(after_class_count))
    before = []
    for class_name in classes:
        before.append(before_class_count.get(class_name, 0))

    after = []
    for class_name in classes:
        after.append(after_class_count.get(class_name, 0))

    df = pd.DataFrame(
        {
            "Class": classes,
            "Before SMOTE": before,
            "After SMOTE": after,
        }
    )

    ax = df.set_index("Class").plot(
        kind="bar",
        figsize=(14, 7),
        width=0.8,
    )

    ax.set_title("Class Distribution Before and After SMOTE")
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of Samples")

    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Dataset")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_smote_comparison(before_class_count, after_class_count)

# 3. Load Datasets

## Load Training Data

In [87]:
train_x, train_y = get_polars_data_frame_with_label(PATH_FOLDER_SMOTE)

Found 168 train files


## Load Validation Data

In [88]:
dev_x = get_polars_data_frame_without_label(PATH_FOLDER_IPCA,"dev")
dev_y = get_polars_data_frame_without_label(PATH_FOLDER_ENCODED_LABEL,"dev").to_series()

Found 168 dev files
Found 168 dev files] data-ipca/dev/2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......


## Load Test Data

In [110]:
test_x = get_polars_data_frame_without_label(PATH_FOLDER_IPCA,"test")
test_y = get_polars_data_frame_without_label(PATH_FOLDER_ENCODED_LABEL,"test").to_series()

Found 168 test files
Found 168 test files data-ipca/test/2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......


# 4. K-Nearest Neighbor

In [139]:
def get_knn_name(n,metric,weights):
    return f"knn-k-{n}-m-{metric}-w-{weights}.pkl"

## Hyperparameter Search for KNN

The grid is `k x metric x weighting` = 7 x 3 x 2 = 42 combinations. Fitting a KNN is
cheap (it only builds an index); the cost is entirely in the dev-set query, so a naive
loop would run 42 full queries.

`search_knn` runs **one query per metric** instead. For a fixed metric, `kneighbors`
returns neighbours already sorted by distance, so the k nearest for any smaller k are
just the leading columns of the same result, and both weighting schemes are re-votes
over those same columns. That is 3 queries instead of 42 for identical results.

The vote itself is done with `np.bincount` over a flattened `(row, class)` index rather
than a Python loop over query rows, which is what makes re-voting 42 times essentially
free compared to the query.

In [ ]:
list_of_n_components = [i for i in range(1, 15, 2)]
list_of_distance_metrics = ["manhattan", "euclidean", "cosine"]
list_of_weighing = ["uniform", "distance"]

In [ ]:
KNN_SEARCH_TRAIN_SAMPLES = 500_000
KNN_SEARCH_DEV_SAMPLES = 200_000
KNN_SEARCH_QUERY_CHUNK = 20_000
KNN_SEARCH_RANDOM_STATE = 42

In [ ]:
def _knn_vote(
    neighbor_labels: np.ndarray,
    neighbor_distances: np.ndarray,
    n_classes: int,
    weighting: str,
) -> np.ndarray:
    """Majority vote over pre-computed neighbours.

    `neighbor_labels` and `neighbor_distances` are both (n_query, k).
    """
    n_query, n_neighbors = neighbor_labels.shape

    if weighting == "uniform":
        weights = np.ones((n_query, n_neighbors), dtype=np.float64)
    else:
        with np.errstate(divide="ignore"):
            weights = 1.0 / neighbor_distances.astype(np.float64)

        exact = ~np.isfinite(weights)
        exact_rows = exact.any(axis=1)
        if exact_rows.any():
            weights[exact_rows] = exact[exact_rows].astype(np.float64)

    row_offset = np.arange(n_query, dtype=np.int64)[:, None] * n_classes
    flat_index = (row_offset + neighbor_labels).ravel()
    scores = np.bincount(
        flat_index, weights=weights.ravel(), minlength=n_query * n_classes
    )
    return scores.reshape(n_query, n_classes).argmax(axis=1).astype(np.int32)

In [ ]:
def _kneighbors_by_chunk(
    index, query: np.ndarray, n_neighbors: int, chunk_size: int
) -> tuple[np.ndarray, np.ndarray]:
    total = query.shape[0]
    n_chunks = math.ceil(total / chunk_size)
    distances = []
    indices = []
    started = time.time()

    for i, start in enumerate(range(0, total, chunk_size), start=1):
        chunk_distance, chunk_index = index.kneighbors(
            query[start : start + chunk_size], n_neighbors=n_neighbors
        )
        distances.append(to_numpy_2d(chunk_distance).astype(np.float32, copy=False))
        indices.append(to_numpy_2d(chunk_index).astype(np.int64, copy=False))

        elapsed = time.time() - started
        eta = elapsed / i * (n_chunks - i)
        print(
            f"    neighbours [{i}/{n_chunks}] elapsed {elapsed:,.0f}s eta {eta:,.0f}s",
            end="\r",
            flush=True,
        )

    print(" " * 100, end="\r")
    return np.vstack(distances), np.vstack(indices)

In [ ]:
def search_knn(
    train_x,
    train_y,
    dev_x,
    dev_y,
    list_n_neighbors: list[int],
    list_distance_metrics: list[str],
    list_weighting: list[str],
    train_samples: int | None = KNN_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = KNN_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    chunk_size: int = KNN_SEARCH_QUERY_CHUNK,
    random_state: int = KNN_SEARCH_RANDOM_STATE,
) -> pd.DataFrame:
    
    print("Preparing search subsamples...")
    search_train_x, search_train_y = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    
    search_dev_x, search_dev_y = stratified_subsample(
        dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class
    )

    n_classes = int(max(search_train_y.max(), search_dev_y.max())) + 1
    max_neighbors = max(list_n_neighbors)
    rows = []

    for distance_metric in list_distance_metrics:
        print(f"Fitting neighbour index (metric={distance_metric})...")
        index = NearestNeighbors(n_neighbors=max_neighbors, metric=distance_metric)
        index.fit(search_train_x)

        neighbor_distances, neighbor_indices = _kneighbors_by_chunk(
            index, search_dev_x, max_neighbors, chunk_size
        )
        neighbor_labels = search_train_y[neighbor_indices]

        for n_neighbors in list_n_neighbors:
            for weighting in list_weighting:
                pred = _knn_vote(
                    neighbor_labels[:, :n_neighbors],
                    neighbor_distances[:, :n_neighbors],
                    n_classes,
                    weighting,
                )
                scores = evaluate(pred=pred, true=search_dev_y).iloc[0].to_dict()
                rows.append(
                    {
                        "n_neighbors": n_neighbors,
                        "metric": distance_metric,
                        "weights": weighting,
                        **scores,
                    }
                )
                print(
                    f"  k={n_neighbors:>3} metric={distance_metric:<10}"
                    f" weights={weighting:<8} f1_macro={scores['f1_macro']:.4f}"
                )

        del index, neighbor_distances, neighbor_indices, neighbor_labels

    return pd.DataFrame(rows).sort_values(
        "f1_macro", ascending=False, ignore_index=True
    )

In [ ]:
knn_search_results = search_knn(
    train_x,
    train_y,
    dev_x,
    dev_y,
    list_of_n_components,
    list_of_distance_metrics,
    list_of_weighing,
)
dump_search_results(knn_search_results, "knn-search.csv")
display(knn_search_results)

In [ ]:
best_knn = get_best_hyperparameter(knn_search_results)
KNN_BEST_N_COMPONENT = int(best_knn["n_neighbors"])
KNN_BEST_DISTANCE_METRIC = str(best_knn["metric"])
KNN_BEST_WEIGHING_METHOD = str(best_knn["weights"])

The `KNN_MODEL_*` constants in the *Evaluate KNN* section further down are still
hardcoded to `k=11 / manhattan / uniform`. Once this search has been run, set them from
`KNN_BEST_*` (and retrain / clear the cached prediction chunks) so the reported model is
the one the search actually selected.

## Train KNN

In [ ]:
def train_knn(
    train_x, train_y, n_neighbors, distance_metric, weighting
) -> KNeighborsClassifier:
    model = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        metric=distance_metric,
        weights=weighting,
    )
    model.fit(to_features(train_x), to_labels(train_y))
    dump_trained_model(
        model, get_knn_name(n_neighbors, distance_metric, weighting), "KNN"
    )

In [ ]:
train_knn(train_x,train_y,11,"manhattan","uniform")

## Evaluate KNN

In [144]:
KNN_MODEL_N_COMPONENT = 11
KNN_MODEL_WEIGHING_METHOD = "uniform"
KNN_MODEL_DISTANCE_METRIC = "manhattan"

In [134]:
def load_knn_model(
    model_path: str, folder_path: str = PATH_FOLDER_MODEL
) -> KNeighborsClassifier:
    return joblib.load(os.path.join(folder_path, "KNN", model_path))

In [133]:
def knn_predict(
    model_path,
    x,
    chunk_size: int = PREDICT_BY_CHUNK,
    cache_dir: str = PATH_FOLDER_PREDICTION_RESULT,
    cache_dir_subfolder: str = "",
):
    cache_dir = os.path.join(cache_dir, "KNN")
    if cache_dir_subfolder:
        cache_dir = os.path.join(cache_dir, cache_dir_subfolder)
    os.makedirs(cache_dir, exist_ok=True)

    model = load_knn_model(model_path)

    total = len(x)
    if total == 0:
        return np.empty(0)

    n_chunks = math.ceil(total / chunk_size)
    paths = []

    for i, start in enumerate(range(0, total, chunk_size), start=1):
        path = os.path.join(cache_dir, f"knn_{i:05d}.npy")
        paths.append(path)

        if os.path.exists(path):
            print(f"Cached [{i}/{n_chunks}]", end="\r", flush=True)
            continue

        if isinstance(x, (pl.DataFrame, pl.Series)):
            chunk = x.slice(start, chunk_size)
        else:
            chunk = x[start : start + chunk_size]

        pred = to_numpy(model.predict(to_features(chunk)))

        tmp = path + ".tmp"
        with open(tmp, "wb") as f:
            np.save(f, pred)
        os.replace(tmp, path)

        done = min(start + chunk_size, total)
        print(
            f"Predicting [{i}/{n_chunks}] {done:,}/{total:,} rows ",
            end="\r",
            flush=True,
        )

    print(f"Predicted {total:,} rows" + " " * 20)
    return np.concatenate([np.load(p) for p in paths])

In [ ]:
knn_predict(get_knn_name(KNN_MODEL_N_COMPONENT,KNN_MODEL_DISTANCE_METRIC,KNN_MODEL_WEIGHING_METHOD),dev_x,cache_dir_subfolder="dev")

In [ ]:
knn_predict(get_knn_name(KNN_MODEL_N_COMPONENT,KNN_MODEL_DISTANCE_METRIC,KNN_MODEL_WEIGHING_METHOD),test_x,cache_dir_subfolder="test")

### Report

In [117]:
def _load_chunks(folder: str) -> np.ndarray:
    if not os.path.isdir(folder):
        raise FileNotFoundError(folder)
    names = sorted(n for n in os.listdir(folder) if n.endswith(".npy"))
    if not names:
        raise FileNotFoundError(f"no .npy chunks in {folder}")
    return np.concatenate([np.load(os.path.join(folder, n)) for n in names])

In [118]:
def load_true_labels(labels_folder: str, split_name: str) -> np.ndarray:
    return to_numpy(
        get_polars_data_frame_without_label(labels_folder, split_name).to_series()
    )

In [115]:
def load_predictions_and_labels(
    folder_predict: str,
    split_name: str,
    labels_folder: str = PATH_FOLDER_ENCODED_LABEL,
) -> tuple[np.ndarray, np.ndarray]:
    pred = _load_chunks(folder_predict)
    true = load_true_labels(labels_folder, split_name)
    if len(pred) != len(true):
        raise ValueError(f"length mismatch: pred={len(pred):,}, true={len(true):,}")
    return pred, true

In [113]:
def get_evaluation_results(
    folder_predict: str,
    split_name: str,
    labels_folder: str = PATH_FOLDER_ENCODED_LABEL,
) -> pd.DataFrame:
    pred, true = load_predictions_and_labels(folder_predict, split_name, labels_folder)
    return evaluate(pred=pred, true=true)

In [72]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "KNN", "test"),
    "test",
)

Found 168 test files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.980034,0.761029,0.844731,0.74896


In [73]:
knn_test_pred, knn_test_true = load_predictions_and_labels(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "KNN", "test"), "test"
)

Found 168 test files


In [74]:
get_classification_report(pred=knn_test_pred, true=knn_test_true)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,class,precision,recall,f1,support
0,Benign,0.989467,0.997013,0.993226,2696950
1,Bot,0.999860,0.999790,0.999825,57238
2,Brute Force -Web,0.188849,0.860656,0.309735,122
3,Brute Force -XSS,0.716667,0.934783,0.811321,46
4,DDOS attack-HOIC,0.999985,1.000000,0.999993,137203
5,DDOS attack-LOIC-UDP,0.714876,1.000000,0.833735,346
6,DDoS attacks-LOIC-HTTP,0.999514,0.998672,0.999093,115237
7,DoS attacks-GoldenEye,0.997236,0.999518,0.998376,8301
8,DoS attacks-Hulk,0.999913,0.999924,0.999919,92382
9,DoS attacks-SlowHTTPTest,0.641026,0.000894,0.001785,27978


In [75]:
cm = get_confusion_matrix(pred=knn_test_pred, true=knn_test_true)
display(cm)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Benign,Bot,Brute Force -Web,Brute Force -XSS,DDOS attack-HOIC,DDOS attack-LOIC-UDP,DDoS attacks-LOIC-HTTP,DoS attacks-GoldenEye,DoS attacks-Hulk,DoS attacks-SlowHTTPTest,DoS attacks-Slowloris,FTP-BruteForce,Infilteration,SQL Injection,SSH-Bruteforce
Benign,2688894,8,435,16,2,0,54,19,5,0,10,2,7452,34,19
Bot,12,57226,0,0,0,0,0,0,0,0,0,0,0,0,0
Brute Force -Web,13,0,105,1,0,0,1,0,0,0,0,0,0,2,0
Brute Force -XSS,0,0,1,43,0,0,0,0,0,0,0,0,0,2,0
DDOS attack-HOIC,0,0,0,0,137203,0,0,0,0,0,0,0,0,0,0
DDOS attack-LOIC-UDP,0,0,0,0,0,346,0,0,0,0,0,0,0,0,0
DDoS attacks-LOIC-HTTP,5,0,10,0,0,138,115084,0,0,0,0,0,0,0,0
DoS attacks-GoldenEye,1,0,0,0,0,0,0,8297,3,0,0,0,0,0,0
DoS attacks-Hulk,4,0,0,0,0,0,0,3,92375,0,0,0,0,0,0
DoS attacks-SlowHTTPTest,0,0,0,0,0,0,0,0,0,25,0,27953,0,0,0


# 5. Support Vector Machine

Kernel SVM does not scale to this dataset the way the other classifiers do. Training
is between `O(n^2)` and `O(n^3)` in the number of rows, and prediction costs
`O(n_query * n_support_vectors)`, so the ~11M-row SMOTE'd train split is out of reach
regardless of GPU acceleration.

The SVM is therefore fit on a **class-proportional stratified subsample** of the train
split (`SVM_TRAIN_SAMPLES`), with a smaller subsample again for the hyperparameter
search (`SVM_SEARCH_TRAIN_SAMPLES`). Two points matter for interpreting the results:

1. Only *training* is subsampled. Dev and test predictions are computed over the full
   splits, so SVM metrics are directly comparable with the other four classifiers.
2. The subsample is drawn *after* SMOTE, from an already-balanced split, so
   proportional sampling preserves the balance SMOTE established.

This is a compute constraint rather than a methodological choice, and it should be
stated as such when the SVM results are reported.

Every slow step below reports progress, but by *stage*, not by clock. `SVC.fit` is a
single blocking call with nothing inside it to count, so each stage prints when it
starts and is completed in place when it returns (`step`), while the chunked prediction
loop draws a real bar from its own iterations (`progress_bar`). Neither uses a
background thread.

On the GPU this section uses `cuml.svm.SVC` directly rather than the `cuml.accel`
proxy. The proxy wraps sklearn's `SVC` but its `_gpu_fit` raises `UnsupportedOnGPU`
whenever `y` has more than two classes, so with 15 attack classes every fit silently
fell back to single-core CPU libsvm. `cuml.svm.SVC` routes multiclass through
`cuml.multiclass.MulticlassClassifier` and fits each pairwise binary SVM on the GPU.

In [156]:
def get_svm_name(kernel, c, gamma):
    return f"svm-k-{kernel}-c-{c}-g-{gamma}.pkl"

## Hyperparameter Search for SVM

`gamma` only affects the RBF kernel, so pairing it with `linear` would duplicate every
`C` value for no benefit; `build_svm_grid` pins it to `"scale"` there. With the defaults
below that gives `3 (linear) + 6 (rbf) = 9` combinations.

`fit_seconds` and `n_support` are recorded alongside the scores because they, not the
score, determine whether a combination is affordable at full scale: the support-vector
count sets the per-row cost of predicting the ~3.2M dev and test rows later.

cuML's GPU solver refuses some combinations outright. A high `C` on a hard pairwise
sub-problem raises `Working set has already been initialized!` from its C++ layer -
reproducibly, as the very first fit in a fresh process, with the card at ~106 MiB, so
it is a solver bug and not an out-of-memory condition. Which combination trips it
depends on the subsample.

Those combinations are **skipped and recorded** with NaN scores and the error text
rather than retried on the CPU: libsvm takes over 15 minutes on precisely the
combinations cuML rejects, against roughly 9 seconds for the ones it accepts, so a
retry would cost more than the entire rest of the grid. Check the `error` column
before reading the results - a `NaN` row is a combination that was never scored, not
one that scored badly, and if the grid's best model would have been in that region it
will not appear in the table at all.

In [50]:
SVM_TRAIN_SAMPLES = 300_000
SVM_SEARCH_TRAIN_SAMPLES = 50_000
SVM_SEARCH_DEV_SAMPLES = 100_000
SVM_RANDOM_STATE = 42
SVM_CACHE_SIZE = 2048
SVM_TOLERANCE = 1e-3
SVM_MAX_ITER = -1

In [51]:
list_of_kernels = ["rbf", "linear"]
list_of_c = [0.1, 1.0, 10.0]
list_of_gamma = ["scale", 0.1]

In [52]:
def build_svm_grid(
    kernels: list[str], list_c: list[float], list_gamma: list
) -> list[tuple]:
    combinations = []
    for kernel in kernels:
        for c in list_c:
            if kernel == "linear":
                combinations.append((kernel, c, "scale"))
                continue
            for gamma in list_gamma:
                combinations.append((kernel, c, gamma))
    return combinations

In [52]:
def build_svm(
    kernel,
    c: float,
    gamma,
    cache_size: int = SVM_CACHE_SIZE,
    tol: float = SVM_TOLERANCE,
    max_iter: int = SVM_MAX_ITER,
) -> SVC:
    return SVC(
        C=float(c),
        kernel=kernel,
        gamma=gamma,
        cache_size=cache_size,
        tol=tol,
        max_iter=max_iter,
    )

In [54]:
def _support_vector_count(model) -> float:
    """Total support vectors, or NaN when the estimator does not expose them.

    cuML's multiclass wrapper does not always surface `n_support_`, so this must not
    be allowed to break the search loop.
    """
    try:
        n_support = getattr(model, "n_support_", None)
        if n_support is not None:
            return float(np.sum(to_numpy_2d(n_support)))
        support = getattr(model, "support_", None)
        if support is not None:
            return float(np.size(to_numpy_2d(support)))
    except Exception:
        pass
    return float("nan")

In [55]:
def _fit_and_score_svm(
    kernel, c, gamma, train_x, train_y, dev_x, dev_y
) -> tuple[float, float, dict]:
    """Fit one combination, score it on dev, and hand the device memory back."""
    model = build_svm(kernel, c, gamma)

    fit_started = time.time()
    with step(f"fitting on {train_x.shape[0]:,} rows"):
        model.fit(train_x, train_y)
    fit_seconds = time.time() - fit_started

    with step(f"predicting {dev_x.shape[0]:,} dev rows"):
        pred = to_numpy(model.predict(dev_x))

    scores = evaluate(pred=pred, true=dev_y).iloc[0].to_dict()
    n_support = _support_vector_count(model)

    del model
    free_gpu_memory()
    return fit_seconds, n_support, scores

In [ ]:
def search_svm(
    train_x,
    train_y,
    dev_x,
    dev_y,
    combinations: list[tuple] | None = None,
    train_samples: int | None = SVM_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = SVM_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    random_state: int = SVM_RANDOM_STATE,
) -> pd.DataFrame:
    if combinations is None:
        combinations = build_svm_grid(list_of_kernels, list_of_c, list_of_gamma)

    total = len(combinations)
    print(f"SVM hyperparameter search over {total} combinations")
    free_gpu_memory()

    print("Preparing search subsamples...")
    search_train_x, search_train_y = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    search_dev_x, search_dev_y = stratified_subsample(
        dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class
    )

    n_train, n_features = search_train_x.shape
    n_dev = search_dev_x.shape[0]
    print(f"  search train {n_train:,} rows x {n_features} features, dev {n_dev:,} rows")

    rows = []
    for i, (kernel, c, gamma) in enumerate(combinations, start=1):
        print(f"[{i}/{total}] kernel={kernel:<7} C={c:<6} gamma={str(gamma):<6}")

        try:
            fit_seconds, n_support, scores = _fit_and_score_svm(
                kernel, c, gamma,
                search_train_x, search_train_y, search_dev_x, search_dev_y,
            )
        except Exception as error:
            reason = f"{type(error).__name__}: {str(error).splitlines()[0][:120]}"
            print(f"!! FAILED on GPU: {reason}")
            print("skipped; continuing with the remaining combinations")
            free_gpu_memory()
            rows.append(
                {
                    "kernel": kernel,
                    "C": c,
                    "gamma": gamma,
                    "fit_seconds": float("nan"),
                    "n_support": float("nan"),
                    "error": reason,
                }
            )
            continue

        rows.append(
            {
                "kernel": kernel,
                "C": c,
                "gamma": gamma,
                "fit_seconds": fit_seconds,
                "n_support": n_support,
                **scores,
            }
        )
        print(
            f"  f1_macro={scores['f1_macro']:.4f}"
            f" accuracy={scores['accuracy']:.4f}"
            f" ({i}/{total} combinations done)"
        )

    results = pd.DataFrame(rows)
    failed = int(results["error"].notna().sum()) if "error" in results else 0
    print(f"Search finished: {total - failed}/{total} combinations scored", end="")
    print(f", {failed} failed" if failed else "")

    if "f1_macro" not in results:
        return results
    return results.sort_values("f1_macro", ascending=False, ignore_index=True)

In [57]:
svm_search_results = search_svm(train_x, train_y, dev_x, dev_y)
dump_search_results(svm_search_results, "svm-search.csv")
display(svm_search_results)

SVM hyperparameter search over 9 combinations
Preparing search subsamples...
Subsampled 11,365,674 -> 49,992 rows across 15 classes
Subsampled 3,246,589 -> 100,497 rows across 15 classes
  search train 49,992 rows x 25 features, dev 100,497 rows
[1/9] kernel=rbf     C=0.1    gamma=scale 
  fitting on 49,992 rows ... done
  predicting 100,497 dev rows ... done
  f1_macro=0.5749 accuracy=0.9362 (1/9 combinations done)
[2/9] kernel=rbf     C=0.1    gamma=0.1   
  fitting on 49,992 rows ... done
  predicting 100,497 dev rows ... done
  f1_macro=0.6846 accuracy=0.9652 (2/9 combinations done)
[3/9] kernel=rbf     C=1.0    gamma=scale 
  fitting on 49,992 rows ... done
  predicting 100,497 dev rows ... done
  f1_macro=0.6877 accuracy=0.9641 (3/9 combinations done)
[4/9] kernel=rbf     C=1.0    gamma=0.1   
  fitting on 49,992 rows ... done
  predicting 100,497 dev rows ... done
  f1_macro=0.7615 accuracy=0.9776 (4/9 combinations done)
[5/9] kernel=rbf     C=10.0   gamma=scale 
  fitting on 49

,kernel,C,gamma,fit_seconds,n_support,accuracy,precision_macro,recall_macro,f1_macro,error
0,rbf,10.0,0.1,10.900169,12401.0,0.978815,0.836687,0.754204,0.770191,NaN
1,rbf,1.0,0.1,8.318841,17260.0,0.977631,0.861189,0.724085,0.761518,NaN
2,linear,1.0,scale,36.027224,11986.0,0.962039,0.872350,0.659094,0.699255,NaN
3,rbf,1.0,scale,8.030320,20458.0,0.964138,0.907165,0.631474,0.687740,NaN
4,rbf,0.1,0.1,10.494591,30330.0,0.965153,0.861869,0.621463,0.684619,NaN
5,linear,0.1,scale,10.144107,17655.0,0.956138,0.874304,0.603402,0.648513,NaN
6,rbf,0.1,scale,9.476785,40911.0,0.936177,0.741910,0.523855,0.574895,NaN
7,rbf,10.0,scale,NaN,NaN,NaN,NaN,NaN,NaN,RuntimeError: exception occurred! file=/tmp/co...
8,linear,10.0,scale,NaN,NaN,NaN,NaN,NaN,NaN,RuntimeError: exception occurred! file=/tmp/co...


In [58]:
best_svm = get_best_hyperparameter(svm_search_results)
SVM_MODEL_KERNEL = str(best_svm["kernel"])
SVM_MODEL_C = float(best_svm["C"])
SVM_MODEL_GAMMA = best_svm["gamma"]

Best by f1_macro:
            kernel = rbf
                 C = 10.0
             gamma = 0.1
       fit_seconds = 10.900169372558594
         n_support = 12401.0
          accuracy = 0.9788152880185478
   precision_macro = 0.8366866805375203
      recall_macro = 0.7542036077816197
          f1_macro = 0.7701906746235035
             error = nan


## Train SVM

In [151]:

SVM_MODEL_KERNEL = "rbf"
SVM_MODEL_C = 10.0
SVM_MODEL_GAMMA = 0.1

In [ ]:
def train_svm(
    train_x,
    train_y,
    kernel: str,
    c: float,
    gamma,
    train_samples: int | None = SVM_TRAIN_SAMPLES,
    random_state: int = SVM_RANDOM_STATE,
    subfolder: str = "SVM",
) -> SVC:
    print(f"Training SVM: kernel={kernel} C={c} gamma={gamma}")

    print("Preparing training subsample...")
    features, labels = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    n_rows, n_features = features.shape

    model = build_svm(kernel, c, gamma)
    with step(f"fitting on {n_rows:,} rows x {n_features} features"):
        model.fit(features, labels)
    print(f"  {_support_vector_count(model):,.0f} support vectors")

    with step("saving model"):
        file_path = dump_trained_model(model, get_svm_name(kernel, c, gamma), subfolder)
    print(f"Saved model to: {file_path}")

In [60]:
train_svm(train_x, train_y, SVM_MODEL_KERNEL, SVM_MODEL_C, SVM_MODEL_GAMMA)

Training SVM: kernel=rbf C=10.0 gamma=0.1
Preparing training subsample...
Subsampled 11,365,674 -> 299,994 rows across 15 classes
  fitting on 299,994 rows x 25 features ... done
  45,515 support vectors
  saving model ... done
Saved model to: trained-model/SVM/svm-k-rbf-c-10.0-g-0.1.pkl


SVC()

## Evaluate SVM

In [150]:
def load_svm_model(model_path: str, folder_path: str = PATH_FOLDER_MODEL) -> SVC:
    return joblib.load(os.path.join(folder_path, "SVM", model_path))

In [149]:
def svm_predict(
    model_path,
    x,
    chunk_size: int = PREDICT_BY_CHUNK,
    cache_dir: str = PATH_FOLDER_PREDICTION_RESULT,
    cache_dir_subfolder: str = "",
):
    cache_dir = os.path.join(cache_dir, "SVM")
    if cache_dir_subfolder:
        cache_dir = os.path.join(cache_dir, cache_dir_subfolder)
    os.makedirs(cache_dir, exist_ok=True)

    with step(f"loading model {model_path}"):
        model = load_svm_model(model_path)

    total = len(x)
    if total == 0:
        print("Nothing to predict (0 rows)")
        return np.empty(0)

    n_chunks = math.ceil(total / chunk_size)
    print(
        f"Predicting {total:,} rows in {n_chunks:,} chunks of {chunk_size:,}"
        f" ({_support_vector_count(model):,.0f} support vectors)"
    )
    print(f"  cache: {cache_dir}")

    paths = []
    computed = 0
    cached = 0

    for i, start in enumerate(range(0, total, chunk_size), start=1):
        path = os.path.join(cache_dir, f"svm_{i:05d}.npy")
        paths.append(path)

        if os.path.exists(path):
            cached += 1
        else:
            if isinstance(x, (pl.DataFrame, pl.Series)):
                chunk = x.slice(start, chunk_size)
            else:
                chunk = x[start : start + chunk_size]

            pred = to_numpy(model.predict(to_features(chunk)))

            tmp = path + ".tmp"
            with open(tmp, "wb") as f:
                np.save(f, pred)
            os.replace(tmp, path)
            computed += 1

        progress_bar(i, n_chunks, f"chunks ({computed:,} computed, {cached:,} cached)")

    del model
    free_gpu_memory()

    with step(f"loading {len(paths):,} prediction chunks"):
        result = np.concatenate([np.load(p) for p in paths])
    return result

In [66]:
svm_predict(
    get_svm_name(SVM_MODEL_KERNEL, SVM_MODEL_C, SVM_MODEL_GAMMA),
    dev_x,
    cache_dir_subfolder="dev",
)

  loading model svm-k-rbf-c-10.0-g-0.1.pkl ... done
Predicting 3,246,589 rows in 325 chunks of 10,000 (45,515 support vectors)
  cache: prediction-result/SVM/dev
  [##############################] 325/325 chunks (325 computed, 0 cached)                          
  loading 325 prediction chunks ... done


array([11, 11, 11, ...,  0,  0,  0], shape=(3246589,), dtype=int32)

In [69]:
svm_predict(
    get_svm_name(SVM_MODEL_KERNEL, SVM_MODEL_C, SVM_MODEL_GAMMA),
    test_x,
    cache_dir_subfolder="test",
)

  loading model svm-k-rbf-c-10.0-g-0.1.pkl ... done
Predicting 3,246,594 rows in 325 chunks of 10,000 (45,515 support vectors)
  cache: prediction-result/SVM/test
  [##############################] 325/325 chunks (325 computed, 0 cached)                          
  loading 325 prediction chunks ... done


array([11, 11, 11, ...,  0,  0,  0], shape=(3246594,), dtype=int32)

### Report

In [93]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "SVM", "dev"),
    "dev",
)

Found 168 dev files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.979212,0.737305,0.787156,0.699035


In [94]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "SVM", "test"),
    "test",
)

Found 168 test files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.979236,0.735212,0.786074,0.700878


In [95]:
svm_test_pred, svm_test_true = load_predictions_and_labels(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "SVM", "test"), "test"
)

Found 168 test files


In [104]:
get_classification_report(pred=svm_test_pred, true=svm_test_true)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,class,precision,recall,f1,support
0,Benign,0.988321,0.997111,0.992697,2696950
1,Bot,0.970477,0.997572,0.983838,57238
2,Brute Force -Web,0.703704,0.467213,0.561576,122
3,Brute Force -XSS,0.003384,0.673913,0.006733,46
4,DDOS attack-HOIC,0.999665,1.000000,0.999832,137203
5,DDOS attack-LOIC-UDP,0.703476,0.994220,0.823952,346
6,DDoS attacks-LOIC-HTTP,0.998701,0.940523,0.968739,115237
7,DoS attacks-GoldenEye,0.991086,0.830382,0.903644,8301
8,DoS attacks-Hulk,0.984528,0.999480,0.991948,92382
9,DoS attacks-SlowHTTPTest,0.759781,0.426192,0.546071,27978


In [105]:
cm = get_confusion_matrix(pred=svm_test_pred, true=svm_test_true)
display(cm)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Benign,Bot,Brute Force -Web,Brute Force -XSS,DDOS attack-HOIC,DDOS attack-LOIC-UDP,DDoS attacks-LOIC-HTTP,DoS attacks-GoldenEye,DoS attacks-Hulk,DoS attacks-SlowHTTPTest,DoS attacks-Slowloris,FTP-BruteForce,Infilteration,SQL Injection,SSH-Bruteforce
Benign,2689158,1736,23,2707,43,1,139,27,67,2,641,21,1400,956,29
Bot,139,57099,0,0,0,0,0,0,0,0,0,0,0,0,0
Brute Force -Web,40,0,57,14,0,0,0,0,0,0,0,0,0,11,0
Brute Force -XSS,15,0,0,31,0,0,0,0,0,0,0,0,0,0,0
DDOS attack-HOIC,0,0,0,0,137203,0,0,0,0,0,0,0,0,0,0
DDOS attack-LOIC-UDP,2,0,0,0,0,344,0,0,0,0,0,0,0,0,0
DDoS attacks-LOIC-HTTP,366,0,0,6344,0,144,108383,0,0,0,0,0,0,0,0
DoS attacks-GoldenEye,51,0,0,0,0,0,0,6893,1355,0,2,0,0,0,0
DoS attacks-Hulk,14,0,0,0,0,0,0,34,92334,0,0,0,0,0,0
DoS attacks-SlowHTTPTest,0,0,0,0,0,0,0,0,0,11924,0,16054,0,0,0


# 6. Random Forest

Random Forest is the first model in this study that does **not** need a training
subsample. Each tree costs about `O(n_rows log n_rows)` to build and the trees are
independent, so the full ~11.3M-row SMOTE'd train split is affordable where kernel SVM
was not: `train_rf` fits on all of it by default, the way `train_knn` does, and takes a
`train_samples` budget for the runs where it cannot. Only the *search* is subsampled
by default, and only to keep a 12-combination grid down to minutes rather than
hours.

As in the SVM section, `cuml.ensemble.RandomForestClassifier` is imported directly
rather than reached through the `cuml.accel` proxy over sklearn, so the estimator that
is built, pickled and reloaded here is unambiguous.

Two cuML-specific details matter when reading the results:

1. cuML splits on **quantiles** (`n_bins`, default 128) rather than on exact feature
   values the way sklearn does, so its trees are not identical to a CPU forest fit with
   the same hyperparameters.
2. `n_streams > 1` builds trees on several CUDA streams whose timing is not
   deterministic, so `random_state` alone does not pin the forest exactly. The default
   of 4 is kept here for speed; set `RF_N_STREAMS = 1` if an exactly reproducible
   forest is needed.

Also unlike sklearn, cuML does not support unlimited depth - `max_depth` must be a
positive integer (default 16) - so depth is an explicit search axis below rather than
something left to grow until the leaves are pure.

In [157]:
def get_rf_name(n_estimators, max_depth, max_features):
    return f"rf-n-{n_estimators}-d-{max_depth}-f-{max_features}.pkl"

## Hyperparameter Search for Random Forest

The grid is `n_estimators x max_depth x max_features` = 2 x 3 x 2 = 12 combinations.
`max_features` is the fraction of the 25 PCA components considered at each split -
`"sqrt"` gives 5 of them and `"log2"` about 4.6 - and it is what decorrelates the trees
from each other, which is the whole reason the ensemble beats a single deep tree.

`fit_seconds` and `predict_seconds` are recorded next to the scores for the same reason
`n_support` was recorded for the SVM: they, not the score, decide whether a combination
is affordable once it has to predict the ~3.2M dev and test rows. A forest of 300 deep
trees can win the search by a hair and still cost several times more per prediction
than the runner-up.

The search subsample is much larger than the SVM's (1M train rows against 50k) because
RF training is near-linear in the row count rather than quadratic, and a forest scored
on too few rows would pick a depth that does not generalise to the full split.

Each combination is wrapped in the same `try` / `free_gpu_memory` guard the SVM search
uses. The deepest, largest forests are the ones most likely to exhaust the 8 GiB card,
and a combination that dies there is recorded with NaN scores and its error text so the
remaining combinations still run. Check the `error` column before reading the table - a
NaN row is a combination that was never scored, not one that scored badly.

In [50]:
RF_SEARCH_TRAIN_SAMPLES = 1_000_000
RF_SEARCH_DEV_SAMPLES = 200_000
RF_TRAIN_SAMPLES = 1_000_000
RF_RANDOM_STATE = 42
RF_SPLIT_CRITERION = "gini"
RF_N_BINS = 128
RF_N_STREAMS = 4

In [51]:
list_of_n_estimators = [100, 300]
list_of_max_depth = [12, 16, 24]
list_of_max_features = ["sqrt", "log2"]

In [66]:
def build_rf_grid(
    list_n_estimators: list[int], list_max_depth: list[int], list_max_features: list
) -> list[tuple]:
    combinations = []
    for n_estimators in list_n_estimators:
        for max_depth in list_max_depth:
            for max_features in list_max_features:
                combinations.append((n_estimators, max_depth, max_features))
    return combinations

In [54]:
def build_rf(
    n_estimators: int,
    max_depth: int,
    max_features,
    split_criterion: str = RF_SPLIT_CRITERION,
    n_bins: int = RF_N_BINS,
    n_streams: int = RF_N_STREAMS,
    random_state: int = RF_RANDOM_STATE,
) -> RandomForestClassifier:
    """Construct the forest, translating the names the two backends disagree on.

    `split_criterion`, `n_bins` and `n_streams` are cuML-only: sklearn calls the first
    `criterion` and has no equivalent for the other two, so forwarding them unchanged
    would make the `use_cuda = False` path raise `TypeError`.
    """
    if use_cuda:
        return RandomForestClassifier(
            n_estimators=int(n_estimators),
            max_depth=int(max_depth),
            max_features=max_features,
            split_criterion=split_criterion,
            n_bins=n_bins,
            n_streams=n_streams,
            random_state=random_state,
        )
    return RandomForestClassifier(
        n_estimators=int(n_estimators),
        max_depth=int(max_depth),
        max_features=max_features,
        criterion=split_criterion,
        random_state=random_state,
        n_jobs=-1,
    )

In [60]:
def _fit_and_score_rf(
    n_estimators, max_depth, max_features, train_x, train_y, dev_x, dev_y
) -> tuple[float, float, dict]:
    """Fit one combination, score it on dev, and hand the device memory back."""
    model = build_rf(n_estimators, max_depth, max_features)

    fit_started = time.time()
    with step(f"fitting {n_estimators} trees on {train_x.shape[0]:,} rows"):
        model.fit(train_x, train_y)
    fit_seconds = time.time() - fit_started

    predict_started = time.time()
    with step(f"predicting {dev_x.shape[0]:,} dev rows"):
        pred = to_numpy(model.predict(dev_x)).astype(np.int32)
    predict_seconds = time.time() - predict_started

    scores = evaluate(pred=pred, true=dev_y).iloc[0].to_dict()

    del model
    free_gpu_memory()
    return fit_seconds, predict_seconds, scores

In [56]:
def search_rf(
    train_x,
    train_y,
    dev_x,
    dev_y,
    combinations: list[tuple] | None = None,
    train_samples: int | None = RF_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = RF_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    random_state: int = RF_RANDOM_STATE,
) -> pd.DataFrame:
    if combinations is None:
        combinations = build_rf_grid(
            list_of_n_estimators, list_of_max_depth, list_of_max_features
        )

    total = len(combinations)
    print(f"Random Forest hyperparameter search over {total} combinations")
    free_gpu_memory()

    print("Preparing search subsamples...")
    search_train_x, search_train_y = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    search_dev_x, search_dev_y = stratified_subsample(
        dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class
    )

    n_train, n_features = search_train_x.shape
    n_dev = search_dev_x.shape[0]
    print(f"  search train {n_train:,} rows x {n_features} features, dev {n_dev:,} rows")

    rows = []
    for i, (n_estimators, max_depth, max_features) in enumerate(combinations, start=1):
        print(
            f"[{i}/{total}] n_estimators={n_estimators:<5}"
            f" max_depth={max_depth:<4} max_features={str(max_features):<6}"
        )

        try:
            fit_seconds, predict_seconds, scores = _fit_and_score_rf(
                n_estimators, max_depth, max_features,
                search_train_x, search_train_y, search_dev_x, search_dev_y,
            )
        except Exception as error:
            reason = f"{type(error).__name__}: {str(error).splitlines()[0][:120]}"
            print(f"!! FAILED on GPU: {reason}")
            print("skipped; continuing with the remaining combinations")
            free_gpu_memory()
            rows.append(
                {
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "max_features": max_features,
                    "fit_seconds": float("nan"),
                    "predict_seconds": float("nan"),
                    "error": reason,
                }
            )
            continue

        rows.append(
            {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "max_features": max_features,
                "fit_seconds": fit_seconds,
                "predict_seconds": predict_seconds,
                **scores,
            }
        )
        print(
            f"  f1_macro={scores['f1_macro']:.4f}"
            f" accuracy={scores['accuracy']:.4f}"
            f" ({i}/{total} combinations done)"
        )

    results = pd.DataFrame(rows)
    failed = int(results["error"].notna().sum()) if "error" in results else 0
    print(f"Search finished: {total - failed}/{total} combinations scored", end="")
    print(f", {failed} failed" if failed else "")

    if "f1_macro" not in results:
        return results
    return results.sort_values("f1_macro", ascending=False, ignore_index=True)

In [57]:
rf_search_results = search_rf(train_x, train_y, dev_x, dev_y)
dump_search_results(rf_search_results, "rf-search.csv")
display(rf_search_results)

Random Forest hyperparameter search over 12 combinations
Preparing search subsamples...
Subsampled 11,365,674 -> 999,993 rows across 15 classes
Subsampled 3,246,589 -> 200,415 rows across 15 classes
  search train 999,993 rows x 25 features, dev 200,415 rows
[1/12] n_estimators=100   max_depth=12   max_features=sqrt  
  fitting 100 trees on 999,993 rows ... done
  predicting 200,415 dev rows ... done
  f1_macro=0.8307 accuracy=0.9834 (1/12 combinations done)
[2/12] n_estimators=100   max_depth=12   max_features=log2  
  fitting 100 trees on 999,993 rows ... done
  predicting 200,415 dev rows ... done
  f1_macro=0.8129 accuracy=0.9832 (2/12 combinations done)
[3/12] n_estimators=100   max_depth=16   max_features=sqrt  
  fitting 100 trees on 999,993 rows ... done
  predicting 200,415 dev rows ... done
  f1_macro=0.8456 accuracy=0.9836 (3/12 combinations done)
[4/12] n_estimators=100   max_depth=16   max_features=log2  
  fitting 100 trees on 999,993 rows ... done
  predicting 200,415 de

,n_estimators,max_depth,max_features,fit_seconds,predict_seconds,accuracy,precision_macro,recall_macro,f1_macro
0,300,24,sqrt,25.880617,2.864710,0.983370,0.876757,0.862078,0.861523
1,100,24,sqrt,4.621328,4.802711,0.983384,0.877159,0.861403,0.861346
2,100,24,log2,7.913444,1.163562,0.983419,0.873017,0.861307,0.858919
3,300,24,log2,21.438648,5.380967,0.983444,0.872478,0.861025,0.858532
4,300,16,sqrt,18.975945,0.931166,0.983589,0.861698,0.858372,0.848918
5,100,16,log2,6.424853,0.309633,0.983659,0.859764,0.857383,0.847040
6,300,16,log2,20.628134,0.831615,0.983639,0.859243,0.856937,0.846656
7,100,16,sqrt,6.680006,0.294816,0.983569,0.858990,0.854178,0.845640
8,300,12,sqrt,16.203462,0.325706,0.983340,0.845670,0.845541,0.831640
9,100,12,sqrt,6.457608,0.194547,0.983399,0.844936,0.844935,0.830721


In [58]:
best_rf = get_best_hyperparameter(rf_search_results)
RF_MODEL_N_ESTIMATORS = int(best_rf["n_estimators"])
RF_MODEL_MAX_DEPTH = int(best_rf["max_depth"])
RF_MODEL_MAX_FEATURES = best_rf["max_features"]

Best by f1_macro:
      n_estimators = 300
         max_depth = 24
      max_features = sqrt
       fit_seconds = 25.880617380142212
   predict_seconds = 2.8647100925445557
          accuracy = 0.9833695082703391
   precision_macro = 0.8767567138394117
      recall_macro = 0.8620778899980094
          f1_macro = 0.8615228839917856


## Train Random Forest

The final forest is refit with the winning combination on the **full** train split by
default rather than on the 1M-row search subsample: `train_samples` defaults to
`RF_TRAIN_SAMPLES = None`, and `stratified_subsample` passes a `None` budget straight
through to `to_features` / `to_labels` without dropping a row.

300 deep trees over 11.3M rows is the heaviest single fit in this notebook. If the
8 GiB card runs out of memory, either set `RF_N_STREAMS = 1` so fewer trees are built
concurrently, or pass a row budget to fit on a class-proportional subsample instead:

```python
train_rf(
    train_x,
    train_y,
    RF_MODEL_N_ESTIMATORS,
    RF_MODEL_MAX_DEPTH,
    RF_MODEL_MAX_FEATURES,
    train_samples=5_000_000,
)
```

Both fits are saved under the same `get_rf_name(...)` file name, so a subsampled forest
overwrites a full one trained with the same hyperparameters.

In [159]:
RF_MODEL_N_ESTIMATORS = 300
RF_MODEL_MAX_DEPTH = 24
RF_MODEL_MAX_FEATURES = "sqrt"

In [83]:
def train_rf(
    train_x,
    train_y,
    n_estimators: int,
    max_depth: int,
    max_features,
    train_samples: int | None = RF_TRAIN_SAMPLES,
    random_state: int = RF_RANDOM_STATE,
    subfolder: str = "RF",
) -> RandomForestClassifier:
    print(
        f"Training Random Forest: n_estimators={n_estimators}"
        f" max_depth={max_depth} max_features={max_features}"
    )

    print("Preparing training data...")
    features, labels = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    n_rows, n_features = features.shape
    
    model = build_rf(n_estimators, max_depth, max_features, random_state=random_state)
    with step(f"fitting {n_estimators} trees on {n_rows:,} rows x {n_features} features"):
        model.fit(features, labels)

    with step("saving model"):
        file_path = dump_trained_model(
            model, get_rf_name(n_estimators, max_depth, max_features), subfolder
        )
    print(f"Saved model to: {file_path}")
    return model

NameError: name 'RF_TRAIN_SAMPLES' is not defined

In [57]:
train_rf(
    train_x, train_y, RF_MODEL_N_ESTIMATORS, RF_MODEL_MAX_DEPTH, RF_MODEL_MAX_FEATURES
)

Training Random Forest: n_estimators=300 max_depth=24 max_features=sqrt
Preparing training data...
Subsampled 11,365,674 -> 999,993 rows across 15 classes
  fitting 300 trees on 999,993 rows x 25 features ... done
  saving model ... done
Saved model to: trained-model/RF/rf-n-300-d-24-f-sqrt.pkl


RandomForestClassifier()

## Evaluate Random Forest

In [161]:
def load_rf_model(
    model_path: str, folder_path: str = PATH_FOLDER_MODEL
) -> RandomForestClassifier:
    return joblib.load(os.path.join(folder_path, "RF", model_path))

In [160]:
def rf_predict(
    model_path,
    x,
    chunk_size: int = PREDICT_BY_CHUNK,
    cache_dir: str = PATH_FOLDER_PREDICTION_RESULT,
    cache_dir_subfolder: str = "",
):
    cache_dir = os.path.join(cache_dir, "RF")
    if cache_dir_subfolder:
        cache_dir = os.path.join(cache_dir, cache_dir_subfolder)
    os.makedirs(cache_dir, exist_ok=True)

    with step(f"loading model {model_path}"):
        model = load_rf_model(model_path)

    total = len(x)
    if total == 0:
        print("Nothing to predict (0 rows)")
        return np.empty(0)

    n_chunks = math.ceil(total / chunk_size)
    print(f"Predicting {total:,} rows in {n_chunks:,} chunks of {chunk_size:,}")
    print(f"  cache: {cache_dir}")

    paths = []
    computed = 0
    cached = 0

    for i, start in enumerate(range(0, total, chunk_size), start=1):
        path = os.path.join(cache_dir, f"rf_{i:05d}.npy")
        paths.append(path)

        if os.path.exists(path):
            cached += 1
        else:
            if isinstance(x, (pl.DataFrame, pl.Series)):
                chunk = x.slice(start, chunk_size)
            else:
                chunk = x[start : start + chunk_size]
            pred = to_numpy(model.predict(to_features(chunk))).astype(np.int32)

            tmp = path + ".tmp"
            with open(tmp, "wb") as f:
                np.save(f, pred)
            os.replace(tmp, path)
            computed += 1

        progress_bar(i, n_chunks, f"chunks ({computed:,} computed, {cached:,} cached)")

    del model
    free_gpu_memory()

    with step(f"loading {len(paths):,} prediction chunks"):
        result = np.concatenate([np.load(p) for p in paths])
    return result

In [62]:
rf_predict(
    get_rf_name(RF_MODEL_N_ESTIMATORS, RF_MODEL_MAX_DEPTH, RF_MODEL_MAX_FEATURES),
    dev_x,
    cache_dir_subfolder="dev",
)

  loading model rf-n-300-d-24-f-sqrt.pkl ... done
Predicting 3,246,589 rows in 325 chunks of 10,000
  cache: prediction-result/RF/dev
  [##############################] 325/325 chunks (325 computed, 0 cached)                          
  loading 325 prediction chunks ... done


array([11, 11, 11, ...,  0,  0,  0], shape=(3246589,), dtype=int32)

In [71]:
rf_predict(
    get_rf_name(RF_MODEL_N_ESTIMATORS, RF_MODEL_MAX_DEPTH, RF_MODEL_MAX_FEATURES),
    test_x,
    cache_dir_subfolder="test",
)

  loading model rf-n-300-d-24-f-sqrt.pkl ... done
Predicting 3,246,594 rows in 325 chunks of 10,000
  cache: prediction-result/RF/test
  [##############################] 325/325 chunks (325 computed, 0 cached)                          
  loading 325 prediction chunks ... done


array([11, 11, 11, ...,  0,  0,  0], shape=(3246594,), dtype=int32)

### Report

In [69]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "RF", "dev"),
    "dev",
)

Found 168 dev files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.983482,0.76634,0.862465,0.777635


In [72]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "RF", "test"),
    "test",
)

Found 168 test files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.983606,0.775959,0.862437,0.782463


In [73]:
rf_test_pred, rf_test_true = load_predictions_and_labels(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "RF", "test"), "test"
)

Found 168 test files


In [80]:
get_classification_report(pred=rf_test_pred, true=rf_test_true)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,class,precision,recall,f1,support
0,Benign,0.989378,0.997713,0.993528,2696950
1,Bot,0.999965,0.999004,0.999484,57238
2,Brute Force -Web,0.139912,0.778689,0.237203,122
3,Brute Force -XSS,0.759259,0.891304,0.820000,46
4,DDOS attack-HOIC,1.000000,0.999985,0.999993,137203
5,DDOS attack-LOIC-UDP,0.720833,1.000000,0.837772,346
6,DDoS attacks-LOIC-HTTP,0.999592,0.998247,0.998919,115237
7,DoS attacks-GoldenEye,0.997708,0.996266,0.996986,8301
8,DoS attacks-Hulk,0.999870,0.999773,0.999821,92382
9,DoS attacks-SlowHTTPTest,0.769637,0.509901,0.613407,27978


In [81]:
cm = get_confusion_matrix(pred=rf_test_pred, true=rf_test_true)
display(cm)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Benign,Bot,Brute Force -Web,Brute Force -XSS,DDOS attack-HOIC,DDOS attack-LOIC-UDP,DDoS attacks-LOIC-HTTP,DoS attacks-GoldenEye,DoS attacks-Hulk,DoS attacks-SlowHTTPTest,DoS attacks-Slowloris,FTP-BruteForce,Infilteration,SQL Injection,SSH-Bruteforce
Benign,2690781,2,560,11,0,0,45,8,8,2,79,6,5379,52,17
Bot,57,57181,0,0,0,0,0,0,0,0,0,0,0,0,0
Brute Force -Web,20,0,95,2,0,0,1,0,0,0,0,0,0,4,0
Brute Force -XSS,0,0,1,41,0,0,0,0,0,0,0,0,0,4,0
DDOS attack-HOIC,2,0,0,0,137201,0,0,0,0,0,0,0,0,0,0
DDOS attack-LOIC-UDP,0,0,0,0,0,346,0,0,0,0,0,0,0,0,0
DDoS attacks-LOIC-HTTP,49,0,19,0,0,134,115035,0,0,0,0,0,0,0,0
DoS attacks-GoldenEye,27,0,0,0,0,0,0,8270,3,0,1,0,0,0,0
DoS attacks-Hulk,10,0,0,0,0,0,0,11,92361,0,0,0,0,0,0
DoS attacks-SlowHTTPTest,0,0,0,0,0,0,0,0,0,14266,0,13708,0,0,4


# 7. Logistic Regression (Softmax)

Softmax regression is the linear baseline of this study. The model is a single
`Dense(n_classes, activation="softmax")` layer applied straight to the 25 PCA
components - **no hidden layer and no non-linearity between input and output** - so
what is being fit is exactly multinomial logistic regression,
$P(y = k \mid x) = \mathrm{softmax}(Wx + b)_k$, with $W$ of shape 15 x 25 and $b$ of
length 15: 390 parameters in total. A model with no hidden representation cannot
learn feature interactions, which is precisely what separates it from the neural
networks this thesis is not about. Keras is used here only as the optimiser and GPU
runtime, not as a way to build a deep model.

Keras rather than `sklearn.linear_model.LogisticRegression` or `cuml.linear_model`
because those solvers are full-batch: `lbfgs` holds the design matrix in memory and
takes a pass over all ~11.3M SMOTE'd rows per iteration. `model.fit` streams
mini-batches to the card instead, so the full train split is affordable the same way
it was for the Random Forest.

The objective is convex, so the optimiser only decides how quickly the single global
optimum is reached, not which model is found. That is why the search below varies the
*learning rate*, *batch size* and *L2 strength* rather than an architecture: the first
two govern convergence and the third is the only real capacity knob a linear model has.

Two data details carry over from the preprocessing chapter. Labels are integer-encoded,
so the loss is `sparse_categorical_crossentropy` rather than `categorical_crossentropy`
- no one-hot expansion of 11.3M x 15 is ever materialised. And no `class_weight` is
passed: the train split is already balanced by SMOTE, so weighting it a second time
would over-correct the rare attack classes.

In [162]:
def get_logreg_name(learning_rate, batch_size, l2):
    """Keras models are not picklable, so this section saves `.keras` archives rather
    than the `.pkl` files `dump_trained_model` writes for the other three models."""
    return f"logreg-lr-{learning_rate}-bs-{batch_size}-l2-{l2}.keras"

## Hyperparameter Search for Logistic Regression

The grid is `learning_rate x batch_size x l2` = 2 x 2 x 3 = 12 combinations, the same
size as the Random Forest grid.

`epochs` is deliberately *not* a search axis. It is not a property of the model, it is
how long the optimiser was allowed to run, so each combination instead trains up to
`LOGREG_MAX_EPOCHS` under an `EarlyStopping` callback watching dev loss and the epoch
count it actually stopped at is recorded as `epochs_run`. `restore_best_weights=True`
means the scored weights are the best epoch's, not the last one's - without it a
combination that started to overfit would be judged on weights it would never be
deployed with. The winning `epochs_run` is then what the final fit is given below.

`fit_seconds` and `predict_seconds` sit next to the scores for the same reason they did
for the forest, though the expectation is the opposite here: 390 parameters predict the
~3.2M dev and test rows far faster than 300 deep trees do, and if softmax regression
comes within a point or two of the forest's macro F1 that cost difference is the
interesting result.

The search subsample matches the Random Forest's 1M train rows so the two are selected
on comparable evidence. The dev subsample is what `EarlyStopping` monitors *and* what
the combination is scored on. That is acceptable because dev is the model-selection
split by construction - the test split is never touched until the report below - but it
does mean the search scores are mildly optimistic and should not be quoted as the
model's performance.

Each combination is wrapped in the same `try` / `free_gpu_memory` guard the SVM and RF
searches use, with `keras.backend.clear_session()` added: TensorFlow keeps the graph
and the optimiser slot variables alive per session, so `del model` alone leaks a little
device memory on every one of the 12 fits. Check the `error` column before reading the
table - a NaN row is a combination that was never scored, not one that scored badly.

In [178]:
LOGREG_SEARCH_TRAIN_SAMPLES = 1_000_000
LOGREG_SEARCH_DEV_SAMPLES = 200_000
LOGREG_TRAIN_SAMPLES = None
LOGREG_RANDOM_STATE = 42
LOGREG_N_CLASSES = len(load_label_encoder().classes_)
LOGREG_MAX_EPOCHS = 30
LOGREG_PATIENCE = 3
LOGREG_PREDICT_BATCH = 8192

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [123]:
list_of_learning_rates = [1e-2, 1e-3]
list_of_batch_sizes = [2048, 8192]
list_of_l2 = [0.0, 1e-5, 1e-4]

In [57]:
def build_logreg_grid(
    list_learning_rate: list[float], list_batch_size: list[int], list_l2: list[float]
) -> list[tuple]:
    combinations = []
    for learning_rate in list_learning_rate:
        for batch_size in list_batch_size:
            for l2 in list_l2:
                combinations.append((learning_rate, batch_size, l2))
    return combinations

In [58]:
def build_logreg(
    n_features: int,
    learning_rate: float,
    l2: float,
    n_classes: int = LOGREG_N_CLASSES,
    random_state: int = LOGREG_RANDOM_STATE,
) -> keras.Model:
    """One softmax layer over the input features - no hidden layer, nothing in between.

    `set_random_seed` seeds Python, NumPy and TensorFlow at once, which is what makes
    the weight initialisation and the per-epoch shuffling order repeatable between
    runs. Unlike the forest there is no `n_streams` caveat here: the fit is
    deterministic once the seed is fixed.
    """
    keras.utils.set_random_seed(random_state)

    model = keras.Sequential(
        [
            keras.Input(shape=(n_features,)),
            keras.layers.Dense(
                n_classes,
                activation="softmax",
                kernel_regularizer=keras.regularizers.L2(l2) if l2 else None,
                name="softmax",
            ),
        ],
        name="logistic_regression",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=float(learning_rate)),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [126]:
def _fit_and_score_logreg(
    learning_rate,
    batch_size,
    l2,
    train_x,
    train_y,
    dev_x,
    dev_y,
    max_epochs: int = LOGREG_MAX_EPOCHS,
    patience: int = LOGREG_PATIENCE,
) -> tuple[float, float, int, dict]:
    """Fit one combination, score it on dev, and hand the device memory back."""
    model = build_logreg(train_x.shape[1], learning_rate, l2)

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience, restore_best_weights=True
    )

    fit_started = time.time()
    with step(f"fitting {train_x.shape[0]:,} rows for up to {max_epochs} epochs"):
        history = model.fit(
            train_x,
            train_y,
            validation_data=(dev_x, dev_y),
            epochs=max_epochs,
            batch_size=int(batch_size),
            callbacks=[early_stopping],
            shuffle=True,
            verbose=0,
        )
    fit_seconds = time.time() - fit_started
    epochs_run = len(history.history["loss"])

    predict_started = time.time()
    with step(f"predicting {dev_x.shape[0]:,} dev rows"):
        probabilities = model.predict(
            dev_x, batch_size=LOGREG_PREDICT_BATCH, verbose=0
        )
        pred = probabilities.argmax(axis=1).astype(np.int32)
    predict_seconds = time.time() - predict_started

    scores = evaluate(pred=pred, true=dev_y).iloc[0].to_dict()

    del model
    keras.backend.clear_session()
    free_gpu_memory()
    return fit_seconds, predict_seconds, epochs_run, scores

In [127]:
def search_logreg(
    train_x,
    train_y,
    dev_x,
    dev_y,
    combinations: list[tuple] | None = None,
    train_samples: int | None = LOGREG_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = LOGREG_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    random_state: int = LOGREG_RANDOM_STATE,
) -> pd.DataFrame:
    if combinations is None:
        combinations = build_logreg_grid(
            list_of_learning_rates, list_of_batch_sizes, list_of_l2
        )

    total = len(combinations)
    print(f"Logistic Regression hyperparameter search over {total} combinations")
    free_gpu_memory()

    print("Preparing search subsamples...")
    search_train_x, search_train_y = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    search_dev_x, search_dev_y = stratified_subsample(
        dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class
    )

    n_train, n_features = search_train_x.shape
    n_dev = search_dev_x.shape[0]
    print(f"  search train {n_train:,} rows x {n_features} features, dev {n_dev:,} rows")

    rows = []
    for i, (learning_rate, batch_size, l2) in enumerate(combinations, start=1):
        print(
            f"[{i}/{total}] learning_rate={learning_rate:<8}"
            f" batch_size={batch_size:<6} l2={l2}"
        )

        try:
            fit_seconds, predict_seconds, epochs_run, scores = _fit_and_score_logreg(
                learning_rate, batch_size, l2,
                search_train_x, search_train_y, search_dev_x, search_dev_y,
            )
        except Exception as error:
            reason = f"{type(error).__name__}: {str(error).splitlines()[0][:120]}"
            print(f"!! FAILED on GPU: {reason}")
            print("skipped; continuing with the remaining combinations")
            keras.backend.clear_session()
            free_gpu_memory()
            rows.append(
                {
                    "learning_rate": learning_rate,
                    "batch_size": batch_size,
                    "l2": l2,
                    "epochs_run": 0,
                    "fit_seconds": float("nan"),
                    "predict_seconds": float("nan"),
                    "error": reason,
                }
            )
            continue

        rows.append(
            {
                "learning_rate": learning_rate,
                "batch_size": batch_size,
                "l2": l2,
                "epochs_run": epochs_run,
                "fit_seconds": fit_seconds,
                "predict_seconds": predict_seconds,
                **scores,
            }
        )
        print(
            f"  f1_macro={scores['f1_macro']:.4f}"
            f" accuracy={scores['accuracy']:.4f}"
            f" epochs={epochs_run}"
            f" ({i}/{total} combinations done)"
        )

    results = pd.DataFrame(rows)
    failed = int(results["error"].notna().sum()) if "error" in results else 0
    print(f"Search finished: {total - failed}/{total} combinations scored", end="")
    print(f", {failed} failed" if failed else "")

    if "f1_macro" not in results:
        return results
    return results.sort_values("f1_macro", ascending=False, ignore_index=True)

In [131]:
logreg_search_results = search_logreg(train_x, train_y, dev_x, dev_y)
dump_search_results(logreg_search_results, "logreg-search.csv")
display(logreg_search_results)

Logistic Regression hyperparameter search over 12 combinations
Preparing search subsamples...
Subsampled 11,365,674 -> 999,993 rows across 15 classes
Subsampled 3,246,589 -> 200,415 rows across 15 classes
  search train 999,993 rows x 25 features, dev 200,415 rows
[1/12] learning_rate=0.01     batch_size=2048   l2=0.0
  fitting 999,993 rows for up to 30 epochs ... done
  predicting 200,415 dev rows ... done
  f1_macro=0.6509 accuracy=0.9604 epochs=30 (1/12 combinations done)
[2/12] learning_rate=0.01     batch_size=2048   l2=1e-05
  fitting 999,993 rows for up to 30 epochs ... done
  predicting 200,415 dev rows ... done
  f1_macro=0.6265 accuracy=0.9557 epochs=30 (2/12 combinations done)
[3/12] learning_rate=0.01     batch_size=2048   l2=0.0001
  fitting 999,993 rows for up to 30 epochs ... done
  predicting 200,415 dev rows ... done
  f1_macro=0.6075 accuracy=0.9508 epochs=21 (3/12 combinations done)
[4/12] learning_rate=0.01     batch_size=8192   l2=0.0
  fitting 999,993 rows for up 

,learning_rate,batch_size,l2,epochs_run,fit_seconds,predict_seconds,accuracy,precision_macro,recall_macro,f1_macro
0,0.010,2048,0.00000,30,23.187312,0.121840,0.960357,0.730837,0.646063,0.650932
1,0.010,2048,0.00001,30,22.146403,0.114561,0.955707,0.731509,0.617519,0.626547
2,0.010,8192,0.00000,30,9.280601,0.097716,0.952524,0.710486,0.609594,0.620550
3,0.010,8192,0.00001,30,12.122184,0.091570,0.952040,0.716968,0.608356,0.618211
4,0.010,8192,0.00010,30,8.683969,0.089462,0.950358,0.676880,0.594945,0.610987
5,0.001,2048,0.00000,30,19.909465,0.096549,0.950488,0.719047,0.596531,0.609814
6,0.001,2048,0.00001,30,17.461799,0.096536,0.950188,0.714520,0.596273,0.609679
7,0.010,2048,0.00010,21,14.636083,0.105959,0.950767,0.752582,0.590356,0.607484
8,0.001,2048,0.00010,30,20.143674,0.096715,0.948392,0.677225,0.583221,0.602060
9,0.001,8192,0.00001,30,9.023938,0.097607,0.926138,0.651963,0.517741,0.533225


In [132]:
best_logreg = get_best_hyperparameter(logreg_search_results)
LOGREG_MODEL_LEARNING_RATE = float(best_logreg["learning_rate"])
LOGREG_MODEL_BATCH_SIZE = int(best_logreg["batch_size"])
LOGREG_MODEL_L2 = float(best_logreg["l2"])
LOGREG_MODEL_EPOCHS = int(best_logreg["epochs_run"])

Best by f1_macro:
     learning_rate = 0.01
        batch_size = 2048.0
                l2 = 0.0
        epochs_run = 30.0
       fit_seconds = 23.187312364578247
   predict_seconds = 0.12183952331542969
          accuracy = 0.960357258688222
   precision_macro = 0.7308371796470929
      recall_macro = 0.6460630738450399
          f1_macro = 0.6509317609424488


## Train Logistic Regression

The final model is refit with the winning combination on the **full** train split:
`train_samples` defaults to `LOGREG_TRAIN_SAMPLES = None`, which `stratified_subsample`
passes straight through to `to_features` / `to_labels` without dropping a row. At 390
parameters this is the cheapest fit in the notebook - the cost is dominated by moving
~11.3M x 25 float32 values through the card, not by the arithmetic - so unlike the SVM
there is no reason to subsample it.

The refit runs for a fixed `LOGREG_MODEL_EPOCHS` taken from the search's `epochs_run`
rather than early-stopping again, so no part of the dev split influences the final
weights; dev then stays a clean held-out split for the report alongside test.

`model.fit` is not wrapped in `step` here the way `RandomForestClassifier.fit` is.
`step` exists for calls with nothing inside them to count, and Keras reports its own
per-epoch progress, so `verbose=2` is used instead - one line per epoch, without the
per-batch bar that would flood the notebook over ~5,500 steps an epoch.

In [163]:
LOGREG_MODEL_LEARNING_RATE = 0.01
LOGREG_MODEL_BATCH_SIZE = 2048
LOGREG_MODEL_L2 = 0.0
LOGREG_MODEL_EPOCHS = 30

In [60]:
def dump_keras_model(model, name: str, subfolder: str, folder: str = PATH_FOLDER_MODEL):
    """`dump_trained_model` for Keras: `joblib.dump` cannot pickle a compiled graph, so
    the model is written as a `.keras` archive instead."""
    target_folder = os.path.join(folder, subfolder)
    os.makedirs(target_folder, exist_ok=True)
    file_path = os.path.join(target_folder, name)
    model.save(file_path)
    return file_path

In [61]:
def train_logreg(
    train_x,
    train_y,
    learning_rate: float,
    batch_size: int,
    l2: float,
    epochs: int,
    train_samples: int | None = LOGREG_TRAIN_SAMPLES,
    random_state: int = LOGREG_RANDOM_STATE,
    subfolder: str = "LOGREG",
):
    print(
        f"Training Logistic Regression: learning_rate={learning_rate}"
        f" batch_size={batch_size} l2={l2} epochs={epochs}"
    )

    print("Preparing training data...")
    features, labels = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    n_rows, n_features = features.shape

    model = build_logreg(n_features, learning_rate, l2, random_state=random_state)
    model.summary()

    print(f"Fitting {epochs} epochs on {n_rows:,} rows x {n_features} features")
    model.fit(
        features,
        labels,
        epochs=int(epochs),
        batch_size=int(batch_size),
        shuffle=True,
        verbose=2,
    )

    with step("saving model"):
        file_path = dump_keras_model(
            model, get_logreg_name(learning_rate, batch_size, l2), subfolder
        )
    print(f"Saved model to: {file_path}")

In [62]:
train_logreg(
    train_x,
    train_y,
    LOGREG_MODEL_LEARNING_RATE,
    LOGREG_MODEL_BATCH_SIZE,
    LOGREG_MODEL_L2,
    LOGREG_MODEL_EPOCHS,
)

Training Logistic Regression: learning_rate=0.01 batch_size=2048 l2=0.0 epochs=30
Preparing training data...


Model: "logistic_regression"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ softmax (Dense)                 │ (None, 15)             │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390 (1.52 KB)

 Trainable params: 390 (1.52 KB)

 Non-trainable params: 0 (0.00 B)

Fitting 30 epochs on 11,365,674 rows x 25 features
Epoch 1/30
5550/5550 - 6s - 1ms/step - accuracy: 0.9103 - loss: 0.3690
Epoch 2/30
5550/5550 - 5s - 930us/step - accuracy: 0.9319 - loss: 0.2603
Epoch 3/30
5550/5550 - 5s - 965us/step - accuracy: 0.9361 - loss: 0.2454
Epoch 4/30
5550/5550 - 5s - 941us/step - accuracy: 0.9372 - loss: 0.2380
Epoch 5/30
5550/5550 - 5s - 936us/step - accuracy: 0.9368 - loss: 0.2353
Epoch 6/30
5550/5550 - 5s - 927us/step - accuracy: 0.9362 - loss: 0.2315
Epoch 7/30
5550/5550 - 5s - 949us/step - accuracy: 0.9360 - loss: 0.2318
Epoch 8/30
5550/5550 - 5s - 949us/step - accuracy: 0.9360 - loss: 0.2298
Epoch 9/30
5550/5550 - 5s - 937us/step - accuracy: 0.9361 - loss: 0.2279
Epoch 10/30
5550/5550 - 5s - 978us/step - accuracy: 0.9363 - loss: 0.2271
Epoch 11/30
5550/5550 - 5s - 955us/step - accuracy: 0.9365 - loss: 0.2257
Epoch 12/30
5550/5550 - 5s - 950us/step - accuracy: 0.9367 - loss: 0.2255
Epoch 13/30
5550/5550 - 5s - 894us/step - accuracy: 0.9369 - loss: 0.227

## Evaluate Logistic Regression

In [164]:
def load_logreg_model(
    model_path: str, folder_path: str = PATH_FOLDER_MODEL
) -> keras.Model:
    return keras.models.load_model(os.path.join(folder_path, "LOGREG", model_path))

In [180]:
def logreg_predict(
    model_path,
    x,
    chunk_size: int = PREDICT_BY_CHUNK,
    cache_dir: str = PATH_FOLDER_PREDICTION_RESULT,
    cache_dir_subfolder: str = "",
    batch_size: int = LOGREG_PREDICT_BATCH,
):
    cache_dir = os.path.join(cache_dir, "LOGREG")
    if cache_dir_subfolder:
        cache_dir = os.path.join(cache_dir, cache_dir_subfolder)
    os.makedirs(cache_dir, exist_ok=True)

    with step(f"loading model {model_path}"):
        model = load_logreg_model(model_path)

    total = len(x)
    if total == 0:
        print("Nothing to predict (0 rows)")
        return np.empty(0)

    n_chunks = math.ceil(total / chunk_size)
    print(f"Predicting {total:,} rows in {n_chunks:,} chunks of {chunk_size:,}")
    print(f"  cache: {cache_dir}")

    paths = []
    computed = 0
    cached = 0

    for i, start in enumerate(range(0, total, chunk_size), start=1):
        path = os.path.join(cache_dir, f"logreg_{i:05d}.npy")
        paths.append(path)

        if os.path.exists(path):
            cached += 1
        else:
            if isinstance(x, (pl.DataFrame, pl.Series)):
                chunk = x.slice(start, chunk_size)
            else:
                chunk = x[start : start + chunk_size]

            # `predict` returns a (chunk, 15) matrix of class probabilities; only the
            # argmax is cached, so these files stay comparable with the other models'.
            probabilities = model.predict(
                to_features(chunk), batch_size=batch_size, verbose=0
            )
            pred = probabilities.argmax(axis=1).astype(np.int32)

            tmp = path + ".tmp"
            with open(tmp, "wb") as f:
                np.save(f, pred)
            os.replace(tmp, path)
            computed += 1

        progress_bar(i, n_chunks, f"chunks ({computed:,} computed, {cached:,} cached)")

    del model
    keras.backend.clear_session()
    free_gpu_memory()

    with step(f"loading {len(paths):,} prediction chunks"):
        result = np.concatenate([np.load(p) for p in paths])
    return result

In [66]:
logreg_predict(
    get_logreg_name(
        LOGREG_MODEL_LEARNING_RATE, LOGREG_MODEL_BATCH_SIZE, LOGREG_MODEL_L2
    ),
    dev_x,
    cache_dir_subfolder="dev",
)

  loading model logreg-lr-0.01-bs-2048-l2-0.0.keras ... done
Predicting 3,246,589 rows in 325 chunks of 10,000
  cache: prediction-result/LOGREG/dev
  [##############################] 325/325 chunks (325 computed, 0 cached)                          
  loading 325 prediction chunks ... done


array([11, 11, 11, ...,  0,  0,  0], shape=(3246589,), dtype=int32)

In [79]:
logreg_predict(
    get_logreg_name(
        LOGREG_MODEL_LEARNING_RATE, LOGREG_MODEL_BATCH_SIZE, LOGREG_MODEL_L2
    ),
    test_x,
    cache_dir_subfolder="test",
)

  loading model logreg-lr-0.01-bs-2048-l2-0.0.keras ... done
Predicting 3,246,594 rows in 325 chunks of 10,000
  cache: prediction-result/LOGREG/test
  [##############################] 325/325 chunks (325 computed, 0 cached)                          
  loading 325 prediction chunks ... done


array([11, 11, 11, ...,  0,  0,  0], shape=(3246594,), dtype=int32)

### Report

In [76]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "LOGREG", "dev"),
    "dev",
)

Found 168 dev files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.963266,0.659582,0.68562,0.635082


In [80]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "LOGREG", "test"),
    "test",
)

Found 168 test files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.963301,0.659458,0.681453,0.632795


In [81]:
logreg_test_pred, logreg_test_true = load_predictions_and_labels(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "LOGREG", "test"), "test"
)

Found 168 test files


In [82]:
get_classification_report(pred=logreg_test_pred, true=logreg_test_true)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,class,precision,recall,f1,support
0,Benign,0.976119,0.991525,0.983762,2696950
1,Bot,0.710082,0.483193,0.575067,57238
2,Brute Force -Web,0.025886,0.377049,0.048447,122
3,Brute Force -XSS,0.358491,0.413043,0.383838,46
4,DDOS attack-HOIC,0.975092,0.999198,0.986998,137203
5,DDOS attack-LOIC-UDP,0.713684,0.979769,0.825822,346
6,DDoS attacks-LOIC-HTTP,0.959191,0.964152,0.961665,115237
7,DoS attacks-GoldenEye,0.973925,0.742441,0.842573,8301
8,DoS attacks-Hulk,0.977814,0.996634,0.987134,92382
9,DoS attacks-SlowHTTPTest,0.504434,0.215491,0.301978,27978


In [83]:
cm = get_confusion_matrix(pred=logreg_test_pred, true=logreg_test_true)
display(cm)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Benign,Bot,Brute Force -Web,Brute Force -XSS,DDOS attack-HOIC,DDOS attack-LOIC-UDP,DDoS attacks-LOIC-HTTP,DoS attacks-GoldenEye,DoS attacks-Hulk,DoS attacks-SlowHTTPTest,DoS attacks-Slowloris,FTP-BruteForce,Infilteration,SQL Injection,SSH-Bruteforce
Benign,2674094,10477,1695,32,3478,0,4698,132,206,40,591,2,827,621,57
Bot,29581,27657,0,0,0,0,0,0,0,0,0,0,0,0,0
Brute Force -Web,74,2,46,0,0,0,0,0,0,0,0,0,0,0,0
Brute Force -XSS,27,0,0,19,0,0,0,0,0,0,0,0,0,0,0
DDOS attack-HOIC,110,0,0,0,137093,0,0,0,0,0,0,0,0,0,0
DDOS attack-LOIC-UDP,1,0,0,0,0,339,6,0,0,0,0,0,0,0,0
DDoS attacks-LOIC-HTTP,3412,577,4,2,0,136,111106,0,0,0,0,0,0,0,0
DoS attacks-GoldenEye,301,0,0,0,0,0,0,6163,1817,0,20,0,0,0,0
DoS attacks-Hulk,265,0,0,0,0,0,0,27,92071,0,17,0,0,2,0
DoS attacks-SlowHTTPTest,0,0,0,0,0,0,0,0,0,6029,0,21949,0,0,0


# 8. XGBoost

Gradient boosting is the counterpart to the Random Forest above: both are ensembles of
decision trees, but a forest averages a few hundred *independent* deep trees, while a
booster fits shallow trees *sequentially*, each one on the error the ones before it
left behind. That is why the two sections search different axes. A forest's error is
dominated by how decorrelated its trees are, which is what `max_features` controls; a
booster's is dominated by how large a step each tree is allowed to take
(`learning_rate`) and how many steps it is given.

cuML has no boosting implementation, so this section uses XGBoost's own
`XGBClassifier`. The card is selected with `device="cuda"` and `tree_method="hist"` -
the histogram builder is the only one with a CUDA implementation, and asking for
`"exact"` on the GPU silently falls back to the CPU. Histogram building is the same
quantile-bucketing idea `RF_N_BINS` controls for the forest, so `XGB_MAX_BIN` is set to
the same 128 buckets to keep the two comparable.

Because `cuml.accel` does not intercept XGBoost, `free_gpu_memory`'s CuPy pool drain is
not what releases a finished booster - the `gc.collect()` inside that helper, after
`del model`, is. It is still called at the same points as in the sections above so that
whatever cuML and TensorFlow are holding is returned between combinations.

The 15-class objective makes this the most device-memory-hungry fit per row in the
notebook: boosting keeps a gradient and a hessian *per class per row*. The training
section below has the numbers and the budget to fall back on if the 8 GiB card cannot
take the full split.

XGBoost is not part of the RAPIDS environment by default. Install it with
`pip install "xgboost>=2.0"` (or `conda install -c conda-forge xgboost`) before running
this section.

In [166]:
def get_xgb_name(max_depth, learning_rate, colsample_bytree):
    """XGBoost is saved as a native `.json` archive rather than as the `.pkl` files
    `dump_trained_model` writes for KNN, SVM and the forest: a pickled booster is only
    guaranteed to reload in the XGBoost version that wrote it, while the JSON model
    format is stable across versions."""
    return f"xgb-d-{max_depth}-lr-{learning_rate}-c-{colsample_bytree}.json"

## Hyperparameter Search for XGBoost

The grid is `max_depth x learning_rate x colsample_bytree` = 3 x 2 x 2 = 12
combinations, the same size as the Random Forest and Logistic Regression grids.

`n_estimators` is deliberately **not** a search axis, for the same reason `epochs` was
not one for the softmax regression: it is not a property of the model but how long the
fit was allowed to run, and it trades off directly against `learning_rate` - a grid
over both would mostly rediscover that a smaller step size needs more steps. Each
combination is instead given a budget of `XGB_MAX_ROUNDS` boosting rounds with
`early_stopping_rounds` watching dev `mlogloss`, and the round it stopped improving at
is recorded as `n_trees_used`. The winning `n_trees_used` is what the final fit below
is given.

`colsample_bytree` is the boosting analogue of the forest's `max_features`: the
fraction of the 25 PCA components each tree may split on. `0.6` gives 15 of them and
`1.0` all 25, so the axis measures whether decorrelating the trees still buys anything
once they are fit sequentially rather than independently.

`max_depth` runs 6/8/10 rather than the forest's 12/16/24. A boosted tree is a
correction rather than a standalone predictor, so it is conventionally much shallower,
and depth is the axis that costs the most - the histogram builder's work per round
roughly doubles per level.

`fit_seconds` and `predict_seconds` sit next to the scores for the same reason they do
in the two sections above. A booster of 400 shallow trees can be *cheaper* to predict
the ~3.2M dev and test rows with than a forest of 300 deep ones, and that comparison is
a result in its own right.

Each combination is wrapped in the same `try` / `free_gpu_memory` guard the SVM, RF and
logistic regression searches use, so a combination that exhausts the card is recorded
with NaN scores and its error text while the rest still run. Check the `error` column
before reading the table - a NaN row is a combination that was never scored, not one
that scored badly.

In [90]:
XGB_SEARCH_TRAIN_SAMPLES = 1_000_000
XGB_SEARCH_DEV_SAMPLES = 200_000
XGB_TRAIN_SAMPLES = None
XGB_RANDOM_STATE = 42
XGB_DEVICE = "cuda" if use_cuda else "cpu"
XGB_TREE_METHOD = "hist"
XGB_MAX_ROUNDS = 400
XGB_EARLY_STOPPING_ROUNDS = 20
XGB_MAX_BIN = 128
XGB_SUBSAMPLE = 0.8
XGB_MIN_CHILD_WEIGHT = 1.0

In [91]:
# Prefixed with `xgb_` rather than reusing the bare `list_of_max_depth` the Random
# Forest section defines: `search_rf` reads that name as a default, so shadowing it
# here would silently change the forest search on a later re-run.
list_of_xgb_max_depth = [6, 8, 10]
list_of_xgb_learning_rates = [0.1, 0.3]
list_of_xgb_colsample_bytree = [0.6, 1.0]

In [92]:
def build_xgb_grid(
    list_max_depth: list[int],
    list_learning_rate: list[float],
    list_colsample_bytree: list[float],
) -> list[tuple]:
    combinations = []
    for max_depth in list_max_depth:
        for learning_rate in list_learning_rate:
            for colsample_bytree in list_colsample_bytree:
                combinations.append((max_depth, learning_rate, colsample_bytree))
    return combinations

In [93]:
def build_xgb(
    max_depth: int,
    learning_rate: float,
    colsample_bytree: float,
    n_estimators: int = XGB_MAX_ROUNDS,
    early_stopping_rounds: int | None = None,
    subsample: float = XGB_SUBSAMPLE,
    min_child_weight: float = XGB_MIN_CHILD_WEIGHT,
    max_bin: int = XGB_MAX_BIN,
    tree_method: str = XGB_TREE_METHOD,
    device: str = XGB_DEVICE,
    random_state: int = XGB_RANDOM_STATE,
) -> XGBClassifier:
    """Construct the booster.

    Unlike `build_rf` there is no `use_cuda` fork: the same class takes `device="cuda"`
    or `device="cpu"` and every other argument means the same thing on both, so the
    only difference between the two paths is where the histograms are built.

    `early_stopping_rounds` is a *constructor* argument in the sklearn wrapper - it was
    moved out of `fit` in XGBoost 1.6 - so the search passes it here and the final
    refit below leaves it at `None`.
    """
    return XGBClassifier(
        n_estimators=int(n_estimators),
        max_depth=int(max_depth),
        learning_rate=float(learning_rate),
        colsample_bytree=float(colsample_bytree),
        subsample=float(subsample),
        min_child_weight=float(min_child_weight),
        max_bin=int(max_bin),
        tree_method=tree_method,
        device=device,
        objective="multi:softprob",
        eval_metric="mlogloss",
        early_stopping_rounds=early_stopping_rounds,
        random_state=random_state,
    )

In [94]:
def to_xgb_features(x):
    """float32 matrix on whichever device the booster is on.

    `predict` has a fast in-place path that needs its input where the model lives.
    Handing a NumPy array to a `device="cuda"` booster still returns the right answer,
    but it falls back to building a `DMatrix` and warns about the mismatched device on
    *every* call - once per 10k-row chunk, so ~320 times over a 3.2M-row split. `fit`
    is left on host arrays, where XGBoost does the transfer itself while binning.
    """
    features = to_features(x)
    if not use_cuda:
        return features

    import cupy as cp

    return cp.asarray(features)

In [95]:
def _fit_and_score_xgb(
    max_depth,
    learning_rate,
    colsample_bytree,
    train_x,
    train_y,
    dev_x,
    dev_y,
    max_rounds: int = XGB_MAX_ROUNDS,
    early_stopping_rounds: int = XGB_EARLY_STOPPING_ROUNDS,
) -> tuple[float, float, int, dict]:
    """Fit one combination, score it on dev, and hand the device memory back."""
    model = build_xgb(
        max_depth,
        learning_rate,
        colsample_bytree,
        n_estimators=max_rounds,
        early_stopping_rounds=early_stopping_rounds,
    )

    fit_started = time.time()
    with step(f"boosting up to {max_rounds} rounds on {train_x.shape[0]:,} rows"):
        model.fit(train_x, train_y, eval_set=[(dev_x, dev_y)], verbose=False)
    fit_seconds = time.time() - fit_started

    # `best_iteration` is a 0-based round index, so the tree count is one more than it.
    # It is only set when early stopping is configured; falling back to the full budget
    # keeps the column meaningful if a future run drops the callback.
    best_iteration = getattr(model, "best_iteration", None)
    n_trees_used = max_rounds if best_iteration is None else int(best_iteration) + 1

    predict_started = time.time()
    with step(f"predicting {dev_x.shape[0]:,} dev rows"):
        pred = to_numpy(model.predict(to_xgb_features(dev_x))).astype(np.int32)
    predict_seconds = time.time() - predict_started

    scores = evaluate(pred=pred, true=dev_y).iloc[0].to_dict()

    del model
    free_gpu_memory()
    return fit_seconds, predict_seconds, n_trees_used, scores

In [96]:
def search_xgb(
    train_x,
    train_y,
    dev_x,
    dev_y,
    combinations: list[tuple] | None = None,
    train_samples: int | None = XGB_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = XGB_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    random_state: int = XGB_RANDOM_STATE,
) -> pd.DataFrame:
    if combinations is None:
        combinations = build_xgb_grid(
            list_of_xgb_max_depth,
            list_of_xgb_learning_rates,
            list_of_xgb_colsample_bytree,
        )

    total = len(combinations)
    print(f"XGBoost hyperparameter search over {total} combinations on {XGB_DEVICE}")
    free_gpu_memory()

    print("Preparing search subsamples...")
    search_train_x, search_train_y = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    search_dev_x, search_dev_y = stratified_subsample(
        dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class
    )

    n_train, n_features = search_train_x.shape
    n_dev = search_dev_x.shape[0]
    print(f"  search train {n_train:,} rows x {n_features} features, dev {n_dev:,} rows")

    rows = []
    for i, (max_depth, learning_rate, colsample_bytree) in enumerate(
        combinations, start=1
    ):
        print(
            f"[{i}/{total}] max_depth={max_depth:<4}"
            f" learning_rate={learning_rate:<6} colsample_bytree={colsample_bytree}"
        )

        try:
            fit_seconds, predict_seconds, n_trees_used, scores = _fit_and_score_xgb(
                max_depth, learning_rate, colsample_bytree,
                search_train_x, search_train_y, search_dev_x, search_dev_y,
            )
        except Exception as error:
            reason = f"{type(error).__name__}: {str(error).splitlines()[0][:120]}"
            print(f"!! FAILED on GPU: {reason}")
            print("skipped; continuing with the remaining combinations")
            free_gpu_memory()
            rows.append(
                {
                    "max_depth": max_depth,
                    "learning_rate": learning_rate,
                    "colsample_bytree": colsample_bytree,
                    "n_trees_used": 0,
                    "fit_seconds": float("nan"),
                    "predict_seconds": float("nan"),
                    "error": reason,
                }
            )
            continue

        rows.append(
            {
                "max_depth": max_depth,
                "learning_rate": learning_rate,
                "colsample_bytree": colsample_bytree,
                "n_trees_used": n_trees_used,
                "fit_seconds": fit_seconds,
                "predict_seconds": predict_seconds,
                **scores,
            }
        )
        print(
            f"  f1_macro={scores['f1_macro']:.4f}"
            f" accuracy={scores['accuracy']:.4f}"
            f" trees={n_trees_used}"
            f" ({i}/{total} combinations done)"
        )

    results = pd.DataFrame(rows)
    failed = int(results["error"].notna().sum()) if "error" in results else 0
    print(f"Search finished: {total - failed}/{total} combinations scored", end="")
    print(f", {failed} failed" if failed else "")

    if "f1_macro" not in results:
        return results
    return results.sort_values("f1_macro", ascending=False, ignore_index=True)

In [97]:
xgb_search_results = search_xgb(train_x, train_y, dev_x, dev_y)
dump_search_results(xgb_search_results, "xgb-search.csv")
display(xgb_search_results)

XGBoost hyperparameter search over 12 combinations on cuda
Preparing search subsamples...
Subsampled 11,365,674 -> 999,993 rows across 15 classes
Subsampled 3,246,589 -> 200,415 rows across 15 classes
  search train 999,993 rows x 25 features, dev 200,415 rows
[1/12] max_depth=6    learning_rate=0.1    colsample_bytree=0.6
  boosting up to 400 rounds on 999,993 rows ... done
  predicting 200,415 dev rows ... done
  f1_macro=0.8605 accuracy=0.9836 trees=343 (1/12 combinations done)
[2/12] max_depth=6    learning_rate=0.1    colsample_bytree=1.0
  boosting up to 400 rounds on 999,993 rows ... done
  predicting 200,415 dev rows ... done
  f1_macro=0.8555 accuracy=0.9836 trees=349 (2/12 combinations done)
[3/12] max_depth=6    learning_rate=0.3    colsample_bytree=0.6
  boosting up to 400 rounds on 999,993 rows ... done
  predicting 200,415 dev rows ... done
  f1_macro=0.7903 accuracy=0.9801 trees=8 (3/12 combinations done)
[4/12] max_depth=6    learning_rate=0.3    colsample_bytree=1.0
  

,max_depth,learning_rate,colsample_bytree,n_trees_used,fit_seconds,predict_seconds,accuracy,precision_macro,recall_macro,f1_macro
0,6,0.1,0.6,343,30.902259,0.663241,0.983624,0.875056,0.865261,0.860534
1,10,0.1,1.0,126,15.316569,0.129531,0.983569,0.871859,0.866785,0.860256
2,8,0.1,0.6,217,24.514732,0.201191,0.983674,0.873737,0.863178,0.859075
3,10,0.1,0.6,146,21.575544,0.137183,0.983584,0.871670,0.862649,0.858132
4,8,0.1,1.0,193,18.713591,0.156307,0.983529,0.869426,0.862893,0.857178
5,6,0.1,1.0,349,29.995047,0.339945,0.983609,0.868573,0.861629,0.855468
6,10,0.3,1.0,1,2.424760,0.019499,0.981803,0.794357,0.853859,0.802937
7,6,0.3,0.6,8,2.536117,0.033180,0.980061,0.793093,0.836087,0.790298
8,8,0.3,0.6,4,2.579453,0.036031,0.981049,0.774878,0.830901,0.778313
9,8,0.3,1.0,1,2.214997,0.020921,0.980351,0.770692,0.854843,0.777710


In [98]:
best_xgb = get_best_hyperparameter(xgb_search_results)
XGB_MODEL_MAX_DEPTH = int(best_xgb["max_depth"])
XGB_MODEL_LEARNING_RATE = float(best_xgb["learning_rate"])
XGB_MODEL_COLSAMPLE_BYTREE = float(best_xgb["colsample_bytree"])
XGB_MODEL_N_ESTIMATORS = int(best_xgb["n_trees_used"])

Best by f1_macro:
         max_depth = 6.0
     learning_rate = 0.1
  colsample_bytree = 0.6
      n_trees_used = 343.0
       fit_seconds = 30.90225887298584
   predict_seconds = 0.663240909576416
          accuracy = 0.983623980241
   precision_macro = 0.8750560475235908
      recall_macro = 0.8652607726356969
          f1_macro = 0.8605342246454897


## Train XGBoost

The final booster is refit with the winning combination on the **full** train split:
`train_samples` defaults to `XGB_TRAIN_SAMPLES = None`, which `stratified_subsample`
passes straight through to `to_features` / `to_labels` without dropping a row.

It runs for a fixed `XGB_MODEL_N_ESTIMATORS` rounds taken from the search's
`n_trees_used`, with early stopping switched off, so no part of the dev split
influences the final trees - the same arrangement the softmax regression uses for its
epoch count. Dev then stays a clean held-out split for the report alongside test.

This is the fit most likely to exhaust the 8 GiB card. A 15-class objective keeps a
gradient and a hessian *per class per row* on the device: at ~11.3M rows that is about
1.4 GiB, on top of ~0.7 GiB of cached margins and ~0.3 GiB for the binned feature
matrix. If it runs out, pass a row budget rather than cutting the tree count, which
would change the model the search selected:

```python
train_xgb(
    train_x,
    train_y,
    XGB_MODEL_MAX_DEPTH,
    XGB_MODEL_LEARNING_RATE,
    XGB_MODEL_COLSAMPLE_BYTREE,
    XGB_MODEL_N_ESTIMATORS,
    train_samples=5_000_000,
)
```

Dropping `XGB_MAX_BIN` from 128 to 64 is the other lever - it halves the histogram
working set at the cost of coarser split points - but it moves this section away from
the forest's bin count, so the two stop being directly comparable.

As in the Random Forest section both fits are saved under the same `get_xgb_name(...)`
file name, so a subsampled booster overwrites a full one trained with the same
hyperparameters.

In [167]:
XGB_MODEL_MAX_DEPTH = 6.0
XGB_MODEL_LEARNING_RATE = 0.1
XGB_MODEL_COLSAMPLE_BYTREE = 0.6
XGB_MODEL_N_ESTIMATORS = 343

In [104]:
def dump_xgb_model(model, name: str, subfolder: str, folder: str = PATH_FOLDER_MODEL):
    """`dump_trained_model` for XGBoost: the booster is written with `save_model` as a
    native `.json` archive instead of being pickled, so it survives an XGBoost upgrade."""
    target_folder = os.path.join(folder, subfolder)
    os.makedirs(target_folder, exist_ok=True)
    file_path = os.path.join(target_folder, name)
    model.save_model(file_path)
    return file_path

In [105]:
def train_xgb(
    train_x,
    train_y,
    max_depth: int,
    learning_rate: float,
    colsample_bytree: float,
    n_estimators: int,
    train_samples: int | None = XGB_TRAIN_SAMPLES,
    random_state: int = XGB_RANDOM_STATE,
    subfolder: str = "XGB",
) -> XGBClassifier:
    print(
        f"Training XGBoost: max_depth={max_depth}"
        f" learning_rate={learning_rate} colsample_bytree={colsample_bytree}"
        f" n_estimators={n_estimators}"
    )

    print("Preparing training data...")
    features, labels = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    n_rows, n_features = features.shape

    model = build_xgb(
        max_depth,
        learning_rate,
        colsample_bytree,
        n_estimators=n_estimators,
        random_state=random_state,
    )
    with step(
        f"boosting {n_estimators} rounds on {n_rows:,} rows x {n_features} features"
        f" ({XGB_DEVICE})"
    ):
        model.fit(features, labels, verbose=False)

    with step("saving model"):
        file_path = dump_xgb_model(
            model, get_xgb_name(max_depth, learning_rate, colsample_bytree), subfolder
        )
    print(f"Saved model to: {file_path}")
    return model

In [106]:
train_xgb(
    train_x,
    train_y,
    XGB_MODEL_MAX_DEPTH,
    XGB_MODEL_LEARNING_RATE,
    XGB_MODEL_COLSAMPLE_BYTREE,
    XGB_MODEL_N_ESTIMATORS,
)

Training XGBoost: max_depth=6.0 learning_rate=0.1 colsample_bytree=0.6 n_estimators=343
Preparing training data...
  boosting 343 rounds on 11,365,674 rows x 25 features (cuda) ... done
  saving model ... done
Saved model to: trained-model/XGB/xgb-d-6.0-lr-0.1-c-0.6.json


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6, device='cuda', early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=128,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=1.0, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=343, n_jobs=None,
              num_parallel_tree=None, ...)

## Evaluate XGBoost

In [168]:
def load_xgb_model(
    model_path: str, folder_path: str = PATH_FOLDER_MODEL
) -> XGBClassifier:
    """Rebuild the wrapper around a saved booster.

    `load_model` restores the trees, the objective and the class count, but the
    run-time settings come from the fresh `XGBClassifier()`, so `device` and
    `tree_method` are set again here - otherwise a booster trained on the card would be
    reloaded onto the CPU without saying so.
    """
    model = XGBClassifier()
    model.load_model(os.path.join(folder_path, "XGB", model_path))
    model.set_params(device=XGB_DEVICE, tree_method=XGB_TREE_METHOD)
    return model

In [169]:
def xgb_predict(
    model_path,
    x,
    chunk_size: int = PREDICT_BY_CHUNK,
    cache_dir: str = PATH_FOLDER_PREDICTION_RESULT,
    cache_dir_subfolder: str = "",
):
    cache_dir = os.path.join(cache_dir, "XGB")
    if cache_dir_subfolder:
        cache_dir = os.path.join(cache_dir, cache_dir_subfolder)
    os.makedirs(cache_dir, exist_ok=True)

    with step(f"loading model {model_path}"):
        model = load_xgb_model(model_path)

    total = len(x)
    if total == 0:
        print("Nothing to predict (0 rows)")
        return np.empty(0)

    n_chunks = math.ceil(total / chunk_size)
    print(f"Predicting {total:,} rows in {n_chunks:,} chunks of {chunk_size:,}")
    print(f"  cache: {cache_dir}")

    paths = []
    computed = 0
    cached = 0

    for i, start in enumerate(range(0, total, chunk_size), start=1):
        path = os.path.join(cache_dir, f"xgb_{i:05d}.npy")
        paths.append(path)

        if os.path.exists(path):
            cached += 1
        else:
            if isinstance(x, (pl.DataFrame, pl.Series)):
                chunk = x.slice(start, chunk_size)
            else:
                chunk = x[start : start + chunk_size]
            pred = to_numpy(model.predict(to_xgb_features(chunk))).astype(np.int32)

            tmp = path + ".tmp"
            with open(tmp, "wb") as f:
                np.save(f, pred)
            os.replace(tmp, path)
            computed += 1

        progress_bar(i, n_chunks, f"chunks ({computed:,} computed, {cached:,} cached)")

    del model
    free_gpu_memory()

    with step(f"loading {len(paths):,} prediction chunks"):
        result = np.concatenate([np.load(p) for p in paths])
    return result

In [109]:
xgb_predict(
    get_xgb_name(
        XGB_MODEL_MAX_DEPTH, XGB_MODEL_LEARNING_RATE, XGB_MODEL_COLSAMPLE_BYTREE
    ),
    dev_x,
    cache_dir_subfolder="dev",
)

  loading model xgb-d-6.0-lr-0.1-c-0.6.json ... done
Predicting 3,246,589 rows in 325 chunks of 10,000
  cache: prediction-result/XGB/dev
  [##############################] 325/325 chunks (325 computed, 0 cached)                          
  loading 325 prediction chunks ... done


array([11, 11, 11, ...,  0,  0,  0], shape=(3246589,), dtype=int32)

In [111]:
xgb_predict(
    get_xgb_name(
        XGB_MODEL_MAX_DEPTH, XGB_MODEL_LEARNING_RATE, XGB_MODEL_COLSAMPLE_BYTREE
    ),
    test_x,
    cache_dir_subfolder="test",
)

  loading model xgb-d-6.0-lr-0.1-c-0.6.json ... done
Predicting 3,246,594 rows in 325 chunks of 10,000
  cache: prediction-result/XGB/test
  [##############################] 325/325 chunks (325 computed, 0 cached)                          
  loading 325 prediction chunks ... done


array([11, 11, 11, ...,  0,  0,  0], shape=(3246594,), dtype=int32)

### Report

In [119]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "XGB", "dev"),
    "dev",
)

Found 168 dev files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.983919,0.777131,0.866008,0.77897


In [120]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "XGB", "test"),
    "test",
)

Found 168 test files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.983919,0.781424,0.868479,0.78063


In [121]:
xgb_test_pred, xgb_test_true = load_predictions_and_labels(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "XGB", "test"), "test"
)

Found 168 test files


In [122]:
get_classification_report(pred=xgb_test_pred, true=xgb_test_true)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,class,precision,recall,f1,support
0,Benign,0.989438,0.998089,0.993745,2696950
1,Bot,0.999948,0.999720,0.999834,57238
2,Brute Force -Web,0.144595,0.877049,0.248260,122
3,Brute Force -XSS,0.773585,0.891304,0.828283,46
4,DDOS attack-HOIC,1.000000,1.000000,1.000000,137203
5,DDOS attack-LOIC-UDP,0.723849,1.000000,0.839806,346
6,DDoS attacks-LOIC-HTTP,0.999557,0.998455,0.999006,115237
7,DoS attacks-GoldenEye,0.999398,0.999398,0.999398,8301
8,DoS attacks-Hulk,0.999978,0.999978,0.999978,92382
9,DoS attacks-SlowHTTPTest,0.795612,0.473050,0.593325,27978


In [123]:
cm = get_confusion_matrix(pred=xgb_test_pred, true=xgb_test_true)
display(cm)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Benign,Bot,Brute Force -Web,Brute Force -XSS,DDOS attack-HOIC,DDOS attack-LOIC-UDP,DDoS attacks-LOIC-HTTP,DoS attacks-GoldenEye,DoS attacks-Hulk,DoS attacks-SlowHTTPTest,DoS attacks-Slowloris,FTP-BruteForce,Infilteration,SQL Injection,SSH-Bruteforce
Benign,2691797,3,603,10,0,0,49,4,1,1,9,0,4387,73,13
Bot,16,57222,0,0,0,0,0,0,0,0,0,0,0,0,0
Brute Force -Web,11,0,107,1,0,0,1,0,0,0,0,0,0,2,0
Brute Force -XSS,0,0,1,41,0,0,0,0,0,0,0,0,0,4,0
DDOS attack-HOIC,0,0,0,0,137203,0,0,0,0,0,0,0,0,0,0
DDOS attack-LOIC-UDP,0,0,0,0,0,346,0,0,0,0,0,0,0,0,0
DDoS attacks-LOIC-HTTP,21,0,25,0,0,132,115059,0,0,0,0,0,0,0,0
DoS attacks-GoldenEye,4,0,0,0,0,0,0,8296,1,0,0,0,0,0,0
DoS attacks-Hulk,1,0,0,0,0,0,0,1,92380,0,0,0,0,0,0
DoS attacks-SlowHTTPTest,0,0,0,0,0,0,0,0,0,13235,0,14743,0,0,0


# 9. Feature Noise Injection (Robustness Evaluation)

Every number reported up to this point was measured on a **clean** test split. That is
the condition almost the whole ML-IDS literature reports under, and it is the gap this
study is built around: a detector that is excellent on pristine benchmark traffic tells
you nothing about what happens when the flow records it is fed have been distorted in
transit.

This section injects *feature noise* - a controlled corruption of the **input attribute
values**, never of the labels - into the test split and re-runs all five trained models
over the resulting test sets.

## What the percentages mean

**The intensity is a share of the test *records*, not the size of the perturbation.**
`gaussian-20` means one flow record in five arrives corrupted and the other four arrive
intact; how badly a corrupted record is damaged is a second, fixed number that does not
change between 10%, 20% and 30%. Those three scenarios differ in *how much* of the
traffic is bad, not in how bad it is.

That reading comes from Section 1.7 - "injeksi noise pada seluruh fitur numerik", every
numeric feature of an affected record is perturbed - so the percentage cannot be
selecting features. It is also the axis Al-Gethami et al. (2021) use, whose noise levels
are shares of mislabelled *instances*: Section 3.2 sets this study against them, and
keeping the same axis is what makes the two comparable. Theirs corrupts the output
label; this one corrupts the input values.

## Where the noise is injected

Noise is applied to `data-imputed/test`, i.e. to the **78 raw-scale flow attributes**
(`flow duration`, `pkt len mean`, `flow byts/s`, ...), *before* the scaler and before
IPCA. The noisy attributes are then pushed through the **frozen** `StandardScaler` and
`IncrementalPCA` transformers from Section 2:

```
data-imputed/test  ->  inject noise  ->  StandardScaler  ->  IPCA(25)  ->  data-noisy/<scenario>/test
                        (this section)      (frozen)          (frozen)
```

Two reasons for that placement rather than perturbing the 25 principal components
directly:

1. **It is the quantity that is actually corrupted in the field.** Packet-capture
   limits, transmission errors and adversarial shaping distort durations, packet
   counts and byte rates - measurable attributes of a flow. A principal component is
   an artefact of *this* pipeline; nothing in a network degrades one.
2. **It is the honest deployment scenario.** The scaler and the IPCA transformer were
   fitted on the clean training split and ship as part of the model. Re-fitting either
   on noisy data would quietly let the model adapt to the corruption, which is the
   opposite of what is being measured - the intrinsic robustness of a model trained on
   clean historical data and then handed degraded live traffic.

The label column is untouched throughout. This is a **feature-noise** study, which is
the distinction that separates it from Al-Gethami et al. (2021), where the injected
noise was mislabelled instances.

## Global Variables

In [185]:
PATH_FOLDER_NOISY = "data-noisy"

In [186]:
NOISE_TYPES = ("gaussian", "uniform", "multiplicative")

# The axis of the whole experiment: the share of test **records** that arrive corrupted. "20%" means
# one flow record in five is damaged and the other four are untouched - not that every record is
# nudged by 20% of something. This is Section 1.6 ("noise diberikan pada dataset ... dengan beberapa
# tingkat intensitas") read together with 1.7 ("injeksi noise pada seluruh fitur numerik"): every
# numeric feature of a selected record is disturbed, so the percentage cannot be choosing features -
# it chooses records. It is also the axis Al-Gethami et al. (2021) use, whose levels are shares of
# mislabelled *instances*; keeping it is what makes the comparison in 3.2 an honest one.
NOISE_INTENSITIES = (0.10, 0.20, 0.30)
NOISE_RANDOM_SEED = 42

# The five models, in the order they are reported and plotted. Kept here rather than
# derived from `get_noise_model_registry()` so that the whole comparison half of this
# section - collecting, ranking, plotting - runs from the cached predictions alone,
# without importing cuML/TensorFlow/XGBoost or needing the model files on disk.
NOISE_MODEL_ORDER = ("KNN", "SVM", "RF", "LOGREG", "XGB")

In [ ]:
# Which cells an intensity selects.
#
# "row"  - `p` of the records, every numeric feature within them. A capture that mis-measures a flow
#          mis-measures the whole record, and it matches "seluruh fitur numerik" in Section 1.7.
# "cell" - `p` of all values, drawn independently, for a per-field fault rather than a per-record one.
NOISE_SCOPE = "row"

In [ ]:
# How badly a *selected* record is damaged. Coverage on its own does not define the experiment -
# "10% of records get Gaussian noise" still has to say Gaussian noise of what size - so this is the
# second number, and it is **fixed**: it does not move between `gaussian-10` and `gaussian-30`, so
# the only thing those three scenarios vary is how much of the data is affected.
#
# The entries differ because the units differ. The additive types are measured in standard deviations
# of the clean training spread; the multiplicative one against each value's own magnitude. They are
# deliberately large: with coverage as the axis a selected record has to be decisively wrong, or the
# model classifies it correctly anyway and the scenario measures nothing.
NOISE_MAGNITUDES = {
    "gaussian": 1.0,        # sigma, in units of the feature's training standard deviation
    "uniform": 1.0,         # the same sigma; the range is widened to match it, see below
    "multiplicative": 0.5,  # standard deviation of the multiplier, which stays effectively positive
}

In [187]:
# Read literally, eq. 2.6 (`eta ~ U(-a, a)` with `a = m * sigma_j`) gives a standard deviation of
# only `a / sqrt(3)` ~ 0.58 `a`, so uniform noise would be a materially weaker disturbance than
# Gaussian noise at the same magnitude and any gap between the two curves would be measuring noise
# strength rather than noise shape. Setting the half-width to `sqrt(3) * m * sigma_j` equalises the
# variance of the two distributions so the only remaining difference is what this section wants to
# isolate: bounded, flat-tailed draws versus unbounded ones with a heavy centre.
NOISE_UNIFORM_MATCH_VARIANCE = True

In [188]:
# Section 1.6 scopes the injection to "seluruh fitur numerik", and all 78 surviving
# columns - `dst port` and `protocol` included - are numeric as far as the trained
# pipeline is concerned: they were standardised and fed into IPCA like any other
# attribute, so exempting them here would evaluate the models on a feature space they
# were never trained on. Listing a column name here restores its clean values after
# injection, for the variant where identifiers are held fixed.
NOISE_EXCLUDE_COLUMNS: tuple[str, ...] = ()

## Helper Functions

In [189]:
def get_noise_scenario_name(noise_type: str, intensity: float) -> str:
    """`("gaussian", 0.10)` -> `"gaussian-10"`.

    Zero-padded so the nine scenario folders sort in intensity order on disk rather
    than lexically (`gaussian-10`, `gaussian-20`, `gaussian-30`).
    """
    return f"{noise_type}-{round(intensity * 100):02d}"


def get_noise_scenarios(
    noise_types: tuple[str, ...] = NOISE_TYPES,
    intensities: tuple[float, ...] = NOISE_INTENSITIES,
) -> list[tuple[str, float, str]]:
    """The full grid as `(noise_type, intensity, scenario_name)` triples."""
    return [
        (noise_type, intensity, get_noise_scenario_name(noise_type, intensity))
        for noise_type in noise_types
        for intensity in intensities
    ]

In [190]:
def get_noise_reference_sigma(scaler: StandardScaler) -> tuple[np.ndarray, list[str]]:
    """Per-feature standard deviation that defines what "10% noise" means.

    Taken from the fitted `StandardScaler` instead of being recomputed on the test
    split. `scale_` was estimated on the **training** data only, so an intensity is
    always a fraction of the clean *training* dispersion and the test statistics are
    never consulted - the same discipline the rest of the pipeline follows, and it
    keeps the three noise types anchored to one common yardstick.
    """
    sigma = np.asarray(scaler.scale_, dtype=np.float64)
    columns = list(scaler.feature_names_in_)
    return sigma, columns


def get_noise_target_mask(
    columns: list[str], exclude: tuple[str, ...] = NOISE_EXCLUDE_COLUMNS
) -> np.ndarray:
    """Boolean mask over `columns`; `False` means "restore this column to clean"."""
    excluded = set(exclude)
    unknown = excluded.difference(columns)
    if unknown:
        raise ValueError(f"NOISE_EXCLUDE_COLUMNS names unknown columns: {sorted(unknown)}")
    return np.array([column not in excluded for column in columns], dtype=bool)

### Noise Distributions

The three injectors below implement equations 2.4-2.7. All of them take the clean
matrix `values` (rows x 78 raw-scale attributes), the intensity `p`, the per-feature
reference deviation `sigma`, and an explicit `rng`, and all of them return a new array
rather than perturbing in place - the clean values are needed again to restore any
excluded columns.

In [191]:
def inject_gaussian_noise(
    values: np.ndarray, magnitude: float, sigma: np.ndarray, rng: np.random.Generator
) -> np.ndarray:
    """`x' = x + eta`, `eta ~ N(0, (m * sigma_j)^2)` - equations 2.4 and 2.5.

    The mean is 0, so the perturbation adds no systematic bias to any attribute, only
    symmetric scatter around its true value. This is the measurement-error model:
    hardware precision limits on a timer or a byte counter.
    """
    return values + rng.standard_normal(values.shape) * (magnitude * sigma)

In [192]:
def inject_uniform_noise(
    values: np.ndarray,
    magnitude: float,
    sigma: np.ndarray,
    rng: np.random.Generator,
    match_variance: bool = NOISE_UNIFORM_MATCH_VARIANCE,
) -> np.ndarray:
    """`x' = x + eta`, `eta ~ U(-a, a)` - equations 2.4 and 2.6.

    The range is symmetric (`a = -b`) so, as with the Gaussian case, no direction is
    favoured. `match_variance` sets `a = sqrt(3) * m * sigma_j` rather than
    `a = m * sigma_j`; see the note on `NOISE_UNIFORM_MATCH_VARIANCE` above for why.
    This is the quantisation model: a value rounded to a fixed logging resolution is
    equally likely to land anywhere inside the bucket.
    """
    half_width = magnitude * sigma * (math.sqrt(3.0) if match_variance else 1.0)
    return values + rng.uniform(-1.0, 1.0, values.shape) * half_width

In [193]:
def inject_multiplicative_noise(
    values: np.ndarray, magnitude: float, sigma: np.ndarray, rng: np.random.Generator
) -> np.ndarray:
    """`x' = x * eta`, `eta ~ N(1, m^2)` - equation 2.7.

    `sigma` is accepted only to keep one signature across the three injectors and is
    deliberately unused: this perturbation is proportional to each value's *own*
    magnitude, so it needs no external yardstick. The mean is 1 rather than 0 so the
    overall scale of the data is preserved and only proportional fluctuation is added.

    Two consequences worth stating: a large attribute is displaced far more than a small
    one in absolute terms, and an attribute that is exactly 0 (an empty counter, an
    unset flag) is left at 0 - so a record picked for this kind of noise still keeps
    every zero it had.
    """
    del sigma
    return values * rng.normal(1.0, magnitude, values.shape)

In [ ]:
def draw_coverage_mask(
    shape: tuple[int, int],
    intensity: float,
    rng: np.random.Generator,
    scope: str = NOISE_SCOPE,
) -> np.ndarray:
    """The cells one intensity selects; `True` means "take the damaged value here".

    The count is exact rather than a coin toss per record: 10% is 10% of *this file*,
    not 10% on average, so the share reported by `check_noise_injection` is the share
    the scenario name claims.
    """
    n_rows, n_columns = shape

    if scope == "row":
        selected = np.zeros(n_rows, dtype=bool)
        selected[rng.choice(n_rows, round(intensity * n_rows), replace=False)] = True
        return np.repeat(selected[:, None], n_columns, axis=1)

    if scope == "cell":
        n_cells = n_rows * n_columns
        selected = np.zeros(n_cells, dtype=bool)
        selected[rng.choice(n_cells, round(intensity * n_cells), replace=False)] = True
        return selected.reshape(shape)

    raise ValueError(f"unknown NOISE_SCOPE {scope!r}; expected 'row' or 'cell'")

In [194]:
NOISE_INJECTORS = {
    "gaussian": inject_gaussian_noise,
    "uniform": inject_uniform_noise,
    "multiplicative": inject_multiplicative_noise,
}


def inject_feature_noise(
    values: np.ndarray,
    noise_type: str,
    intensity: float,
    sigma: np.ndarray,
    rng: np.random.Generator,
    target_mask: np.ndarray | None = None,
) -> np.ndarray:
    """Corrupt `intensity` of the records, then restore any column excluded by `target_mask`.

    The damaged matrix is built in full and then blended with the clean one through the
    coverage mask, rather than the injector being handed only the selected rows. It
    costs one extra array and keeps the three injectors free of any knowledge of how
    many records they are damaging.
    """
    if noise_type not in NOISE_INJECTORS:
        raise ValueError(
            f"unknown noise type {noise_type!r};"
            f" expected one of {sorted(NOISE_INJECTORS)}"
        )

    damaged = NOISE_INJECTORS[noise_type](
        values, NOISE_MAGNITUDES[noise_type], sigma, rng
    )
    noisy = np.where(draw_coverage_mask(values.shape, intensity, rng), damaged, values)

    if target_mask is not None and not target_mask.all():
        noisy[:, ~target_mask] = values[:, ~target_mask]
    return noisy

## Generate the Noisy Test Sets

One worker per Parquet file, mirroring `_scale_single_file` and
`_ipca_transform_single_file` in Section 2. Each worker writes the finished
25-component matrix straight to `data-noisy/<scenario>/test/`; the intermediate
78-column noisy matrix is never persisted, which keeps the nine scenarios to roughly
the size of one copy of `data-ipca/test` each instead of two.

In [195]:
def _make_noisy_single_file(
    file_path: str,
    output_split_folder: str,
    noise_type: str,
    intensity: float,
    sigma: np.ndarray,
    target_mask: np.ndarray,
    scaler: StandardScaler,
    ipca: IncrementalPCA,
    file_index: int,
    seed: int = NOISE_RANDOM_SEED,
):
    output_path = os.path.join(output_split_folder, os.path.basename(file_path))
    if os.path.exists(output_path):
        return

    df = pd.read_parquet(file_path)
    columns = df.columns.tolist()

    # A stream of its own per (scenario, file). The files are noised in parallel by
    # separate `loky` processes, so a generator seeded once in the parent would be
    # forked into every worker and hand all 168 files the *same* draws. Deriving the
    # seed from the coordinates instead makes each file independent and makes the whole
    # run reproducible - and resumable, since a file re-noised after an interrupt gets
    # the draws it would have got the first time.
    rng = np.random.default_rng(
        [seed, NOISE_TYPES.index(noise_type), round(intensity * 100), file_index]
    )
    noisy = inject_feature_noise(
        df.to_numpy(dtype=np.float64), noise_type, intensity, sigma, rng, target_mask
    )

    # Both transformers are handed DataFrames rather than bare arrays: each was fitted
    # on one and carries `feature_names_in_`, so an unnamed array would raise a feature
    # name warning on every one of the 1,512 (9 scenarios x 168 files) calls.
    scaled = scaler.transform(pd.DataFrame(noisy, columns=columns))
    reduced = ipca.transform(pd.DataFrame(scaled, columns=columns))
    reduced = reduced.astype(PARQUET_FLOAT_DTYPE, copy=False)

    pc_columns = [f"pc{i + 1}" for i in range(reduced.shape[1])]

    # Written via a temporary name so an interrupted run cannot leave a truncated
    # Parquet file behind that the `os.path.exists` check above would then trust.
    tmp = output_path + ".tmp"
    pd.DataFrame(reduced, columns=pc_columns).to_parquet(
        tmp, engine="pyarrow", compression="snappy", index=False
    )
    os.replace(tmp, output_path)

In [196]:
def make_noisy_test_sets(
    source_folder_name: str = PATH_FOLDER_IMPUTED,
    output_folder_name: str = PATH_FOLDER_NOISY,
    split_name: str = "test",
    scenarios: list[tuple[str, float, str]] | None = None,
    scaler_path: str = PATH_SCALER,
    pc: int = IPCA_PC_USED,
):
    """Build every `data-noisy/<scenario>/<split_name>/` folder.

    Already-written files are skipped, so an interrupted run resumes where it stopped
    and re-running the cell is cheap.
    """
    scenarios = get_noise_scenarios() if scenarios is None else scenarios

    scaler = load_standard_scaler(scaler_path)
    ipca = load_ipca(pc)
    sigma, columns = get_noise_reference_sigma(scaler)
    target_mask = get_noise_target_mask(columns)

    file_paths = get_split_parquet_files(source_folder_name, split_name)
    print(
        f"Generating {len(scenarios)} noisy '{split_name}' sets"
        f" from {len(file_paths)} files x {len(columns)} features"
        f" -> {ipca.n_components_} components"
    )
    if not target_mask.all():
        held = [c for c, keep in zip(columns, target_mask) if not keep]
        print(f"  holding {len(held)} column(s) clean: {held}")

    for noise_type, intensity, scenario in scenarios:
        output_split_folder = os.path.join(output_folder_name, scenario, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        print(f"\n[{scenario}] {noise_type} noise at {intensity:.0%} -> {output_split_folder}")

        with parallel_config(backend="loky", inner_max_num_threads=1):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_make_noisy_single_file)(
                    file_path,
                    output_split_folder,
                    noise_type,
                    intensity,
                    sigma,
                    target_mask,
                    scaler,
                    ipca,
                    file_index,
                )
                for file_index, file_path in enumerate(file_paths)
            )

        print(f"Finished {scenario}: {len(file_paths)} files.")

In [197]:
make_noisy_test_sets()

Found 168 test files
Generating 9 noisy 'test' sets from 168 files x 78 features -> 25 components

[gaussian-10] gaussian noise at 10% -> data-noisy/gaussian-10/test


/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator IncrementalPCA from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    4.4s
[Parallel(n_jobs=-1)]

Finished gaussian-10: 168 files.

[gaussian-20] gaussian noise at 20% -> data-noisy/gaussian-20/test
Finished gaussian-20: 168 files.


[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    0.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    0.0s



[gaussian-30] gaussian noise at 30% -> data-noisy/gaussian-30/test


...

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 155 out of 168 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    0.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 155 out of 168 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    0.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    0.0s


Finished multiplicative-10: 168 files.

[multiplicative-20] multiplicative noise at 20% -> data-noisy/multiplicative-20/test
Finished multiplicative-20: 168 files.

[multiplicative-30] multiplicative noise at 30% -> data-noisy/multiplicative-30/test


[Parallel(n_jobs=-1)]: Done 155 out of 168 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    0.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    0.1s


Finished multiplicative-30: 168 files.


[Parallel(n_jobs=-1)]: Done 155 out of 168 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    0.2s finished


### Verify the Injection

The written Parquet files hold 25 principal components, so the perturbation cannot be
read back out of them directly. This check instead re-runs each injector in-process on
one clean file and measures what it produced, which is what actually needs verifying.

`records_changed` is the reading that matters - it should land exactly on the intensity,
because that is what the intensity means. `cells_changed` should equal it for the two
additive types, since every numeric feature of a chosen record is perturbed;
`multiplicative` falls short because `x * eta` cannot move a zero and roughly three
quarters of this table is zeros. `magnitude` should sit on the fixed value from *Global
Variables* for that type - the same at 10% as at 30%, since the intensity moves how much
of the data is damaged, not how badly.

In [198]:
def check_noise_injection(
    source_folder_name: str = PATH_FOLDER_IMPUTED,
    split_name: str = "test",
    scenarios: list[tuple[str, float, str]] | None = None,
    file_index: int = 0,
    scaler_path: str = PATH_SCALER,
) -> pd.DataFrame:
    """Measure what every scenario did to a single clean file: how much, and how hard."""
    scenarios = get_noise_scenarios() if scenarios is None else scenarios

    scaler = load_standard_scaler(scaler_path)
    sigma, columns = get_noise_reference_sigma(scaler)
    target_mask = get_noise_target_mask(columns)

    file_paths = get_split_parquet_files(source_folder_name, split_name)
    clean = pd.read_parquet(file_paths[file_index]).to_numpy(dtype=np.float64)

    rows = []
    for noise_type, intensity, scenario in scenarios:
        rng = np.random.default_rng(
            [NOISE_RANDOM_SEED, NOISE_TYPES.index(noise_type), round(intensity * 100), file_index]
        )
        noisy = inject_feature_noise(clean, noise_type, intensity, sigma, rng, target_mask)
        changed = noisy != clean

        if noise_type == "multiplicative":
            # Against each value's own magnitude, and only where there was one to move.
            touched = changed & (clean != 0.0)
            magnitude = float(np.std((noisy[touched] - clean[touched]) / clean[touched]))
        else:
            # Against the column's spread. An additive perturbation is independent of the
            # value it lands on, so dividing by the value would only measure whichever
            # cells happen to sit nearest zero.
            magnitude = float(np.std(((noisy - clean) / sigma)[changed]))

        rows.append(
            {
                "scenario": scenario,
                "noise_type": noise_type,
                "intensity": intensity,
                "magnitude_set": NOISE_MAGNITUDES[noise_type],
                "records_changed": float(changed.any(axis=1).mean()),
                "cells_changed": float(changed.mean()),
                "magnitude": magnitude,
            }
        )
    return pd.DataFrame(rows)

In [199]:
check_noise_injection()

Found 168 test files


,scenario,noise_type,intensity,sd_in_sigma_units,sd_relative_to_value,mean_shift_in_sigma_units,unchanged_cells
0,gaussian-10,gaussian,0.1,0.099976,NaN,0.000086,0.000000
1,gaussian-20,gaussian,0.2,0.199861,NaN,0.000167,0.000000
2,gaussian-30,gaussian,0.3,0.300260,NaN,-0.000205,0.000000
3,uniform-10,uniform,0.1,0.100014,NaN,-0.000070,0.000000
4,uniform-20,uniform,0.2,0.200030,NaN,0.000084,0.000000
5,uniform-30,uniform,0.3,0.299990,NaN,-0.000065,0.000000
6,multiplicative-10,multiplicative,0.1,0.126545,0.099971,0.000172,0.756294
7,multiplicative-20,multiplicative,0.2,0.253321,0.199789,0.000144,0.756294
8,multiplicative-30,multiplicative,0.3,0.380495,0.300509,-0.000238,0.756294


## Predict the Noisy Test Sets

Each of the five trained models is replayed over all nine noisy test sets. Nothing is
retrained and no threshold is re-tuned - the models are exactly the artefacts evaluated
in Sections 4-8, which is the point: what is being measured is the robustness they
already have, not robustness they could be given by noise-augmented training.

**One cell is one pass: one model, one noise shape, one intensity.** The grid is 5 x 3
x 3 = 45 passes over 3.2M rows, and KNN and the RBF SVM account for most of the wall
clock - both score every query against a large stored set and neither has a shortcut.
Driving all 45 from a single loop means an interrupted session loses whichever pass was
in flight and offers no way to work through the grid in sittings, so it is laid out
below as 45 separate calls to `predict_one_noise_scenario(...)`, grouped by scenario.
Run as many as one sitting allows, close the notebook, come back and continue at the
first cell that has not been run.

Three things make stopping anywhere safe:

* **Per-chunk caching.** Each `*_predict` writes one `.npy` per 10,000 rows into
  `prediction-result/<MODEL>/noise/<scenario>/` and skips the chunks already on disk,
  so a cell killed halfway through resumes at the first missing chunk rather than at
  zero. This is the same cache the clean `test/` runs use.
* **A one-slot split cache.** The five cells of a scenario share the same ~3.2M x 25
  matrix. `load_noisy_split` holds the most recently used split in memory and drops it
  as soon as a different scenario is asked for, so a scenario read top to bottom loads
  its data once instead of five times - and only one split is ever resident.
* **Stored scores.** A finished pass writes its accuracy and macro
  precision/recall/F1 to `hyperparameter-search/noise-scores/<MODEL>__<scenario>.json`
  as soon as it completes. The comparison further down is then assembled from those 45
  small files, so the numbers of a pass survive the session that produced it and
  nothing has to be re-scored in one long block at the end.

`noise_prediction_status()` prints how far the grid has got at any point: chunks
written, chunks expected, and whether the score for that pair is on disk.

In [200]:
PATH_NOISE_PREDICTION_SUBFOLDER = "noise"


def get_noise_prediction_folder(
    model_key: str, scenario: str, folder: str = PATH_FOLDER_PREDICTION_RESULT
) -> str:
    """Where one (model, scenario) pair caches its prediction chunks."""
    return os.path.join(folder, model_key, PATH_NOISE_PREDICTION_SUBFOLDER, scenario)

In [201]:
def get_noise_model_registry() -> dict[str, tuple]:
    """`model key -> (predict function, saved model file name)` for all five models.

    Built on each call rather than held in a module-level dict so that it reads the
    current value of the `*_MODEL_*` constants - re-running one of the "Evaluate"
    sections above with a different winning combination is picked up here without
    having to remember to re-run this cell too.
    """
    registry = {
        "KNN": (
            knn_predict,
            get_knn_name(
                KNN_MODEL_N_COMPONENT,
                KNN_MODEL_DISTANCE_METRIC,
                KNN_MODEL_WEIGHING_METHOD,
            ),
        ),
        "SVM": (svm_predict, get_svm_name(SVM_MODEL_KERNEL, SVM_MODEL_C, SVM_MODEL_GAMMA)),
        "RF": (
            rf_predict,
            get_rf_name(RF_MODEL_N_ESTIMATORS, RF_MODEL_MAX_DEPTH, RF_MODEL_MAX_FEATURES),
        ),
        "LOGREG": (
            logreg_predict,
            get_logreg_name(
                LOGREG_MODEL_LEARNING_RATE, LOGREG_MODEL_BATCH_SIZE, LOGREG_MODEL_L2
            ),
        ),
        "XGB": (
            xgb_predict,
            get_xgb_name(
                XGB_MODEL_MAX_DEPTH, XGB_MODEL_LEARNING_RATE, XGB_MODEL_COLSAMPLE_BYTREE
            ),
        ),
    }

    if tuple(registry) != NOISE_MODEL_ORDER:
        raise ValueError(
            f"registry keys {tuple(registry)} disagree with"
            f" NOISE_MODEL_ORDER {NOISE_MODEL_ORDER}"
        )
    return registry

In [202]:
# One slot rather than a dict of nine: a noisy test split is ~3.2M x 25 float32, and
# holding several at once is the quickest way to run this section out of RAM. Asking
# for a different scenario evicts whichever one is currently held.
_NOISE_SPLIT_CACHE: dict[str, Any] = {}


def load_noisy_split(
    scenario: str,
    noisy_folder: str = PATH_FOLDER_NOISY,
    split_name: str = "test",
) -> pl.DataFrame:
    """The 25-component matrix of one scenario, shared by that scenario's five cells."""
    key = os.path.join(scenario, split_name)
    if _NOISE_SPLIT_CACHE.get("key") == key:
        held = cast(pl.DataFrame, _NOISE_SPLIT_CACHE["data"])
        print(f"Using the '{key}' split already in memory ({len(held):,} rows)")
        return held

    release_noisy_split()
    print(f"Loading '{key}'")
    data = get_polars_data_frame_without_label(
        os.path.join(noisy_folder, scenario), split_name
    )
    # Trailing blanks because the loader leaves the cursor on a `\r` progress line.
    print(f"Loaded {len(data):,} rows x {data.width} components" + " " * 20)
    _NOISE_SPLIT_CACHE["key"] = key
    _NOISE_SPLIT_CACHE["data"] = data
    return data


def release_noisy_split() -> None:
    """Drop the held split. Worth running before leaving the notebook open and idle."""
    key = _NOISE_SPLIT_CACHE.pop("key", None)
    if _NOISE_SPLIT_CACHE.pop("data", None) is not None:
        print(f"Released the '{key}' split")
    gc.collect()

In [203]:
PATH_FOLDER_NOISE_SCORES = os.path.join(PATH_FOLDER_SEARCH_RESULT, "noise-scores")


def get_noise_score_path(
    model_key: str, scenario: str, folder: str = PATH_FOLDER_NOISE_SCORES
) -> str:
    """One small JSON per finished (model, scenario) pass."""
    return os.path.join(folder, f"{model_key}__{scenario}.json")


def save_noise_score(record: dict, folder: str = PATH_FOLDER_NOISE_SCORES) -> str:
    """Write one score record, via a temporary name so a kill cannot truncate it."""
    os.makedirs(folder, exist_ok=True)
    path = get_noise_score_path(record["model"], record["scenario"], folder)
    tmp = path + ".tmp"
    with open(tmp, "w") as file:
        json.dump(record, file, indent=2)
    os.replace(tmp, path)
    return path


def load_noise_scores(folder: str = PATH_FOLDER_NOISE_SCORES) -> pd.DataFrame:
    """Every score stored so far, in one frame. Empty if none have been written yet."""
    records = []
    for path in sorted(glob.glob(os.path.join(folder, "*.json"))):
        with open(path) as file:
            records.append(json.load(file))
    return pd.DataFrame(records)


# The 3.2M true labels and the fixed class list are read once and reused by all 45
# scoring calls - see "Fixing the class list" below for why the list is pinned to the
# classes present in `true` instead of being re-derived per scenario.
_NOISE_LABEL_CACHE: dict[str, Any] = {}


def get_noise_true_labels(
    labels_folder: str = PATH_FOLDER_ENCODED_LABEL, split_name: str = "test"
) -> tuple[np.ndarray, list[int]]:
    """`(true labels, fixed class list)` for the split every scenario is scored on."""
    key = os.path.join(labels_folder, split_name)
    if _NOISE_LABEL_CACHE.get("key") != key:
        true = load_true_labels(labels_folder, split_name)
        _NOISE_LABEL_CACHE["key"] = key
        _NOISE_LABEL_CACHE["true"] = true
        _NOISE_LABEL_CACHE["labels"] = sorted({int(value) for value in np.unique(true)})
    return _NOISE_LABEL_CACHE["true"], _NOISE_LABEL_CACHE["labels"]

In [204]:
def score_noise_prediction(
    model_key: str,
    scenario: str,
    noise_type: str | None = None,
    intensity: float | None = None,
    pred: np.ndarray | None = None,
    folder: str | None = None,
    split_name: str = "test",
    labels_folder: str = PATH_FOLDER_ENCODED_LABEL,
    prediction_folder: str = PATH_FOLDER_PREDICTION_RESULT,
    scores_folder: str = PATH_FOLDER_NOISE_SCORES,
) -> pd.DataFrame:
    """Score one finished pass and store the row as JSON.

    `pred` is passed straight through by `predict_one_noise_scenario`, which already
    holds the concatenated predictions; left at `None` the chunks are re-read from
    disk, which is what makes a pass that was run in an earlier session scoreable
    without predicting it again.
    """
    if folder is None:
        folder = get_noise_prediction_folder(model_key, scenario, prediction_folder)
    if pred is None:
        pred = _load_chunks(folder)

    true, labels = get_noise_true_labels(labels_folder, split_name)
    if len(pred) != len(true):
        raise ValueError(
            f"length mismatch for {model_key}/{scenario}:"
            f" pred={len(pred):,}, true={len(true):,}"
        )

    if noise_type is None or intensity is None:
        # `"gaussian-10"` -> `("gaussian", 0.10)`; `"clean"` has no suffix to split off.
        head, _, tail = scenario.rpartition("-")
        noise_type = head or scenario
        intensity = int(tail) / 100 if tail.isdigit() else 0.0

    scores = evaluate(pred=pred, true=true, labels=labels).iloc[0].to_dict()
    record = {
        "model": model_key,
        "noise_type": noise_type,
        "intensity": intensity,
        "scenario": scenario,
        **{name: float(value) for name, value in scores.items()},
        "rows": int(len(pred)),
        "scored_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    path = save_noise_score(record, scores_folder)
    print(f"Stored {path}")
    return pd.DataFrame([record])

In [205]:
def predict_one_noise_scenario(
    model_key: str,
    noise_type: str,
    intensity: float,
    score: bool = True,
    noisy_folder: str = PATH_FOLDER_NOISY,
    split_name: str = "test",
    model_folder: str = PATH_FOLDER_MODEL,
    prediction_folder: str = PATH_FOLDER_PREDICTION_RESULT,
    scores_folder: str = PATH_FOLDER_NOISE_SCORES,
) -> pd.DataFrame | None:
    """Replay one model over one noisy test set, then store what came out.

    The unit of work this section is run in. Safe to re-run: every chunk it would
    recompute is already cached on disk, so a second call over a finished pass only
    re-reads and re-scores it. `score=False` skips the evaluation and leaves the
    predictions cached for later.
    """
    scenario = get_noise_scenario_name(noise_type, intensity)
    registry = get_noise_model_registry()
    if model_key not in registry:
        raise ValueError(
            f"unknown model key {model_key!r}; expected one of {tuple(registry)}"
        )

    predict_function, model_file = registry[model_key]
    model_path = os.path.join(model_folder, model_key, model_file)
    if not os.path.exists(model_path):
        print(f"Skipped {model_key} / {scenario}: no saved model at {model_path}")
        return None

    print(f"=== {model_key} / {scenario} ({noise_type} at {intensity:.0%}) ===")
    x = load_noisy_split(scenario, noisy_folder, split_name)

    pred = predict_function(
        model_file,
        x,
        cache_dir_subfolder=os.path.join(PATH_NOISE_PREDICTION_SUBFOLDER, scenario),
    )
    free_gpu_memory()

    if not score:
        print(f"Predictions cached in {get_noise_prediction_folder(model_key, scenario, prediction_folder)}")
        return None

    return score_noise_prediction(
        model_key,
        scenario,
        noise_type,
        intensity,
        pred=pred,
        split_name=split_name,
        prediction_folder=prediction_folder,
        scores_folder=scores_folder,
    )

In [206]:
# Row counts come from the Parquet footers, not from the data, so the status board is
# instant; cached because the nine counts never change once the splits are written.
_NOISE_ROW_COUNT_CACHE: dict[str, int] = {}


def get_noise_split_row_count(
    scenario: str, noisy_folder: str = PATH_FOLDER_NOISY, split_name: str = "test"
) -> int:
    key = os.path.join(noisy_folder, scenario, split_name)
    if key not in _NOISE_ROW_COUNT_CACHE:
        _NOISE_ROW_COUNT_CACHE[key] = int(
            pl.scan_parquet(os.path.join(key, "*.parquet"))
            .select(pl.len())
            .collect()
            .item()
        )
    return _NOISE_ROW_COUNT_CACHE[key]


def noise_prediction_status(
    scenarios: list[tuple[str, float, str]] | None = None,
    models: list[str] | None = None,
    chunk_size: int = PREDICT_BY_CHUNK,
    noisy_folder: str = PATH_FOLDER_NOISY,
    split_name: str = "test",
    prediction_folder: str = PATH_FOLDER_PREDICTION_RESULT,
    scores_folder: str = PATH_FOLDER_NOISE_SCORES,
    only_unfinished: bool = False,
) -> pd.DataFrame:
    """How far the 45-pass grid has got: chunks written, chunks expected, score stored.

    Reads the file system only, so it costs nothing and answers the question a run
    split across sittings keeps raising - which cell to start at next.
    """
    scenarios = get_noise_scenarios() if scenarios is None else scenarios
    models = list(NOISE_MODEL_ORDER) if models is None else models

    rows = []
    for _, _, scenario in scenarios:
        expected = math.ceil(
            get_noise_split_row_count(scenario, noisy_folder, split_name) / chunk_size
        )
        for model_key in models:
            folder = get_noise_prediction_folder(model_key, scenario, prediction_folder)
            written = len(glob.glob(os.path.join(folder, "*.npy")))
            rows.append(
                {
                    "model": model_key,
                    "scenario": scenario,
                    "chunks": written,
                    "of": expected,
                    "percent": round(100 * written / expected, 1) if expected else 0.0,
                    "predicted": written >= expected,
                    "scored": os.path.exists(
                        get_noise_score_path(model_key, scenario, scores_folder)
                    ),
                }
            )

    status = pd.DataFrame(rows)
    print(
        f"{int(status['predicted'].sum())}/{len(status)} passes predicted,"
        f" {int(status['scored'].sum())} scored"
    )
    if only_unfinished:
        status = status[~status["predicted"]].reset_index(drop=True)
    return status

In [207]:
def predict_noise_scenario(
    noise_type: str,
    intensity: float,
    models: list[str] | None = None,
    **kwargs,
) -> pd.DataFrame:
    """All five models over one noisy test set - the five cells below, in one call."""
    models = list(NOISE_MODEL_ORDER) if models is None else models
    rows = [
        predict_one_noise_scenario(model_key, noise_type, intensity, **kwargs)
        for model_key in models
    ]
    return pd.concat([row for row in rows if row is not None], ignore_index=True)


def predict_noisy_test_sets(
    scenarios: list[tuple[str, float, str]] | None = None,
    models: list[str] | None = None,
    **kwargs,
) -> pd.DataFrame:
    """The whole 45-pass grid in one call.

    Kept for an unattended run on a machine that can be left alone for it. The cells
    below are the same work one pass at a time, and are the better way to do it while
    watching, since each finishes and stores its score on its own.
    """
    scenarios = get_noise_scenarios() if scenarios is None else scenarios
    models = list(NOISE_MODEL_ORDER) if models is None else models

    rows = []
    for noise_type, intensity, _ in scenarios:
        for model_key in models:
            row = predict_one_noise_scenario(model_key, noise_type, intensity, **kwargs)
            if row is not None:
                rows.append(row)
    release_noisy_split()
    free_gpu_memory()
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

### Running the grid

`noise_prediction_status()` first: it is a file-system read, costs nothing, and says
which of the 45 passes are already done. Then work down the scenario blocks below, one
cell at a time, in any order - each cell is self-contained and re-running a finished
one just re-reads its cached chunks.

Keep the cells of one scenario together where possible. They share the loaded split, so
running `gaussian-10`'s five models back to back reads that split once, while hopping
between scenarios reloads it each time.

In [208]:
noise_prediction_status()

0/45 passes predicted, 0 scored


,model,scenario,chunks,of,percent,predicted,scored
0,KNN,gaussian-10,56,325,17.2,False,False
1,SVM,gaussian-10,0,325,0.0,False,False
2,RF,gaussian-10,0,325,0.0,False,False
3,LOGREG,gaussian-10,0,325,0.0,False,False
4,XGB,gaussian-10,0,325,0.0,False,False
5,KNN,gaussian-20,0,325,0.0,False,False
6,SVM,gaussian-20,0,325,0.0,False,False
7,RF,gaussian-20,0,325,0.0,False,False
8,LOGREG,gaussian-20,0,325,0.0,False,False
9,XGB,gaussian-20,0,325,0.0,False,False


#### Gaussian noise on 10% of the records

Unbounded, heavy-centre draws on every attribute of a corrupted record:
`x + eta`, `eta ~ N(0, sigma_j^2)`.

Scenario `gaussian-10`; predictions land in `prediction-result/<MODEL>/noise/gaussian-10/`.

In [209]:
predict_one_noise_scenario("KNN", "gaussian", 0.10)

=== KNN / gaussian-10 (gaussian at 10%) ===
Loading 'gaussian-10/test'
Found 168 test files
Loaded 3,246,594 rows x 25 components                    02-Friday_TrafficForML_CICFlowMeter_00011.parquet......
Predicted 3,246,594 rows                    s 


NameError: name 'load_true_labels' is not defined

In [210]:
predict_one_noise_scenario("SVM", "gaussian", 0.10)

=== SVM / gaussian-10 (gaussian at 10%) ===
Using the 'gaussian-10/test' split already in memory (3,246,594 rows)
  loading model svm-k-rbf-c-10.0-g-0.1.pkl ... done


NameError: name '_support_vector_count' is not defined

In [211]:
predict_one_noise_scenario("RF", "gaussian", 0.10)

=== RF / gaussian-10 (gaussian at 10%) ===
Using the 'gaussian-10/test' split already in memory (3,246,594 rows)
  loading model rf-n-300-d-24-f-sqrt.pkl ... failed


EOFError: 

In [ ]:
predict_one_noise_scenario("LOGREG", "gaussian", 0.10)

In [ ]:
predict_one_noise_scenario("XGB", "gaussian", 0.10)

#### Gaussian noise on 20% of the records

Unbounded, heavy-centre draws on every attribute of a corrupted record:
`x + eta`, `eta ~ N(0, sigma_j^2)`.

Scenario `gaussian-20`; predictions land in `prediction-result/<MODEL>/noise/gaussian-20/`.

In [ ]:
predict_one_noise_scenario("KNN", "gaussian", 0.20)

In [ ]:
predict_one_noise_scenario("SVM", "gaussian", 0.20)

In [ ]:
predict_one_noise_scenario("RF", "gaussian", 0.20)

In [ ]:
predict_one_noise_scenario("LOGREG", "gaussian", 0.20)

In [ ]:
predict_one_noise_scenario("XGB", "gaussian", 0.20)

#### Gaussian noise on 30% of the records

Unbounded, heavy-centre draws on every attribute of a corrupted record:
`x + eta`, `eta ~ N(0, sigma_j^2)`.

Scenario `gaussian-30`; predictions land in `prediction-result/<MODEL>/noise/gaussian-30/`.

In [ ]:
predict_one_noise_scenario("KNN", "gaussian", 0.30)

In [ ]:
predict_one_noise_scenario("SVM", "gaussian", 0.30)

In [ ]:
predict_one_noise_scenario("RF", "gaussian", 0.30)

In [ ]:
predict_one_noise_scenario("LOGREG", "gaussian", 0.30)

In [ ]:
predict_one_noise_scenario("XGB", "gaussian", 0.30)

#### Uniform noise on 10% of the records

Bounded, flat draws on every attribute of a corrupted record:
`x + eta`, `eta ~ U(-a, a)`, variance-matched to the Gaussian case.

Scenario `uniform-10`; predictions land in `prediction-result/<MODEL>/noise/uniform-10/`.

In [ ]:
predict_one_noise_scenario("KNN", "uniform", 0.10)

In [ ]:
predict_one_noise_scenario("SVM", "uniform", 0.10)

In [ ]:
predict_one_noise_scenario("RF", "uniform", 0.10)

In [ ]:
predict_one_noise_scenario("LOGREG", "uniform", 0.10)

In [ ]:
predict_one_noise_scenario("XGB", "uniform", 0.10)

#### Uniform noise on 20% of the records

Bounded, flat draws on every attribute of a corrupted record:
`x + eta`, `eta ~ U(-a, a)`, variance-matched to the Gaussian case.

Scenario `uniform-20`; predictions land in `prediction-result/<MODEL>/noise/uniform-20/`.

In [ ]:
predict_one_noise_scenario("KNN", "uniform", 0.20)

In [ ]:
predict_one_noise_scenario("SVM", "uniform", 0.20)

In [ ]:
predict_one_noise_scenario("RF", "uniform", 0.20)

In [ ]:
predict_one_noise_scenario("LOGREG", "uniform", 0.20)

In [ ]:
predict_one_noise_scenario("XGB", "uniform", 0.20)

#### Uniform noise on 30% of the records

Bounded, flat draws on every attribute of a corrupted record:
`x + eta`, `eta ~ U(-a, a)`, variance-matched to the Gaussian case.

Scenario `uniform-30`; predictions land in `prediction-result/<MODEL>/noise/uniform-30/`.

In [ ]:
predict_one_noise_scenario("KNN", "uniform", 0.30)

In [ ]:
predict_one_noise_scenario("SVM", "uniform", 0.30)

In [ ]:
predict_one_noise_scenario("RF", "uniform", 0.30)

In [ ]:
predict_one_noise_scenario("LOGREG", "uniform", 0.30)

In [ ]:
predict_one_noise_scenario("XGB", "uniform", 0.30)

#### Multiplicative noise on 10% of the records

Proportional distortion of every attribute of a corrupted record:
`x * eta`, `eta ~ N(1, 0.5^2)`. Zero-valued counters stay at zero.

Scenario `multiplicative-10`; predictions land in `prediction-result/<MODEL>/noise/multiplicative-10/`.

In [ ]:
predict_one_noise_scenario("KNN", "multiplicative", 0.10)

In [ ]:
predict_one_noise_scenario("SVM", "multiplicative", 0.10)

In [ ]:
predict_one_noise_scenario("RF", "multiplicative", 0.10)

In [ ]:
predict_one_noise_scenario("LOGREG", "multiplicative", 0.10)

In [ ]:
predict_one_noise_scenario("XGB", "multiplicative", 0.10)

#### Multiplicative noise on 20% of the records

Proportional distortion of every attribute of a corrupted record:
`x * eta`, `eta ~ N(1, 0.5^2)`. Zero-valued counters stay at zero.

Scenario `multiplicative-20`; predictions land in `prediction-result/<MODEL>/noise/multiplicative-20/`.

In [ ]:
predict_one_noise_scenario("KNN", "multiplicative", 0.20)

In [ ]:
predict_one_noise_scenario("SVM", "multiplicative", 0.20)

In [ ]:
predict_one_noise_scenario("RF", "multiplicative", 0.20)

In [ ]:
predict_one_noise_scenario("LOGREG", "multiplicative", 0.20)

In [ ]:
predict_one_noise_scenario("XGB", "multiplicative", 0.20)

#### Multiplicative noise on 30% of the records

Proportional distortion of every attribute of a corrupted record:
`x * eta`, `eta ~ N(1, 0.5^2)`. Zero-valued counters stay at zero.

Scenario `multiplicative-30`; predictions land in `prediction-result/<MODEL>/noise/multiplicative-30/`.

In [ ]:
predict_one_noise_scenario("KNN", "multiplicative", 0.30)

In [ ]:
predict_one_noise_scenario("SVM", "multiplicative", 0.30)

In [ ]:
predict_one_noise_scenario("RF", "multiplicative", 0.30)

In [ ]:
predict_one_noise_scenario("LOGREG", "multiplicative", 0.30)

In [ ]:
predict_one_noise_scenario("XGB", "multiplicative", 0.30)

### Once the grid is finished

`release_noisy_split()` hands back the ~320 MB the last scenario is still holding, and
the status board should show 45/45 predicted and 45 scored. Anything short of that is
listed by `noise_prediction_status(only_unfinished=True)`, which names the cells still
to run.

In [ ]:
release_noisy_split()
noise_prediction_status()

## Comparison

### Fixing the class list

`evaluate` defaults to scoring over the union of the classes that appear in `true` and
`pred`. That default is fine for a single run but wrong here: if noise pushes a model
into predicting a class it never predicted on clean data, the macro average silently
gains a denominator and the scenario stops being comparable to the baseline. Every
evaluation below is therefore scored over one fixed list - the classes present in the
**true** test labels - so all ten columns of a model's row are macro-averaged over
exactly the same classes.

In [ ]:
NOISE_CLEAN_SCENARIO = "clean"


def get_noise_evaluation_labels(
    labels_folder: str = PATH_FOLDER_ENCODED_LABEL, split_name: str = "test"
) -> list[int]:
    """The fixed class list every scenario is macro-averaged over.

    The very list each pass was already scored against up in "Predict the Noisy Test
    Sets" - one cached read of the true labels serves both halves of the section, so
    a score stored mid-grid and one computed here cannot drift apart.
    """
    _, labels = get_noise_true_labels(labels_folder, split_name)
    return labels

In [ ]:
def collect_noise_results(
    scenarios: list[tuple[str, float, str]] | None = None,
    models: list[str] | None = None,
    include_clean: bool = True,
    split_name: str = "test",
    labels_folder: str = PATH_FOLDER_ENCODED_LABEL,
    prediction_folder: str = PATH_FOLDER_PREDICTION_RESULT,
    scores_folder: str = PATH_FOLDER_NOISE_SCORES,
    prefer_stored: bool = True,
) -> pd.DataFrame:
    """One row per (model, scenario) with accuracy, macro precision/recall/F1.

    Assembled from the JSON each pass stored as it finished, so the table is a read of
    45 small files rather than a re-scoring of 45 x 3.2M predictions, and it survives
    however many sittings the grid was run over. A pair with predictions but no stored
    score is scored here and stored - which is also how the clean baseline gets in: it
    is read from the `test/` predictions the earlier sections wrote and folded in as
    intensity 0.0, so every degradation below is measured against the same artefact
    reported in Sections 4-8.

    Pairs that are still part-way through are reported and skipped, so the comparison
    can be looked at mid-grid. `prefer_stored=False` re-scores everything from the
    cached chunks, which is what to reach for after a model has been retrained and its
    predictions regenerated.
    """
    scenarios = get_noise_scenarios() if scenarios is None else scenarios
    models = list(NOISE_MODEL_ORDER) if models is None else models

    if include_clean:
        scenarios = [
            (NOISE_CLEAN_SCENARIO, 0.0, NOISE_CLEAN_SCENARIO)
        ] + list(scenarios)

    true, _ = get_noise_true_labels(labels_folder, split_name)

    rows = []
    for noise_type, intensity, scenario in scenarios:
        for model_key in models:
            score_path = get_noise_score_path(model_key, scenario, scores_folder)
            if prefer_stored and os.path.exists(score_path):
                with open(score_path) as file:
                    rows.append(json.load(file))
                continue

            folder = (
                os.path.join(prediction_folder, model_key, split_name)
                if scenario == NOISE_CLEAN_SCENARIO
                else get_noise_prediction_folder(model_key, scenario, prediction_folder)
            )
            if not os.path.isdir(folder):
                print(f"  missing predictions, skipped: {folder}")
                continue

            pred = _load_chunks(folder)
            if len(pred) != len(true):
                print(
                    f"  partial predictions, skipped: {model_key} / {scenario}"
                    f" ({len(pred):,} of {len(true):,} rows)"
                )
                continue

            print(f"  scoring {model_key} / {scenario}")
            rows.append(
                score_noise_prediction(
                    model_key,
                    scenario,
                    noise_type,
                    intensity,
                    pred=pred,
                    folder=folder,
                    split_name=split_name,
                    labels_folder=labels_folder,
                    prediction_folder=prediction_folder,
                    scores_folder=scores_folder,
                )
                .iloc[0]
                .to_dict()
            )

    return pd.DataFrame(rows)

In [ ]:
noise_results = collect_noise_results()
display(noise_results)

In [ ]:
noise_results.to_csv(
    os.path.join(PATH_FOLDER_SEARCH_RESULT, "noise-robustness-results.csv"), index=False
)
print("Saved: hyperparameter-search/noise-robustness-results.csv")

### Degradation

Absolute scores answer "which model is best under noise"; they do not answer the
question this study actually asks, which is **which model loses the least**. A model
that starts high can still be the most fragile. `retention` (noisy score / clean score)
separates the two: 1.00 means the noise cost the model nothing, 0.50 means it lost half
of what it had.

In [ ]:
def summarise_noise_degradation(
    results: pd.DataFrame, metric: str = "f1_macro"
) -> pd.DataFrame:
    """Attach each model's clean baseline and the drop/retention against it."""
    clean = (
        results[results["scenario"] == NOISE_CLEAN_SCENARIO]
        .set_index("model")[metric]
        .rename(f"{metric}_clean")
    )
    missing = set(results["model"]).difference(clean.index)
    if missing:
        raise ValueError(f"no clean baseline for: {sorted(missing)}")

    noisy = results[results["scenario"] != NOISE_CLEAN_SCENARIO].copy()
    noisy = noisy.join(clean, on="model")
    noisy[f"{metric}_drop"] = noisy[f"{metric}_clean"] - noisy[metric]
    noisy[f"{metric}_retention"] = noisy[metric] / noisy[f"{metric}_clean"]

    return noisy[
        [
            "model",
            "noise_type",
            "intensity",
            "scenario",
            f"{metric}_clean",
            metric,
            f"{metric}_drop",
            f"{metric}_retention",
        ]
    ].reset_index(drop=True)

In [ ]:
noise_degradation = summarise_noise_degradation(noise_results)
display(noise_degradation)

In [ ]:
def pivot_noise_metric(
    results: pd.DataFrame, metric: str = "f1_macro", include_clean: bool = True
) -> pd.DataFrame:
    """`model` x `scenario` view of one metric, scenarios in grid order."""
    order = [s for _, _, s in get_noise_scenarios()]
    if include_clean:
        order = [NOISE_CLEAN_SCENARIO] + order

    table = results.pivot_table(index="model", columns="scenario", values=metric)
    return table.reindex(columns=[s for s in order if s in table.columns])

In [ ]:
pivot_noise_metric(noise_results, "f1_macro").round(4)

### Plots

Five models means five series that have to stay distinguishable in one figure. The
colours below are the first five slots of a palette whose ordering was checked for
colour-vision separation rather than chosen by eye, taken in fixed order and keyed to
the model - so a model keeps its colour no matter which subset is plotted or how the
subset ranks. Marker shape repeats the same identity, so nothing depends on colour
alone.

In [ ]:
# Fixed order, never cycled: slot n of the palette always belongs to the same model,
# so a model keeps its colour whichever subset is plotted (see `NOISE_MODEL_ORDER`).
NOISE_MODEL_COLORS = {
    "KNN": "#2a78d6",     # blue
    "SVM": "#eb6834",     # orange
    "RF": "#1baf7a",      # aqua
    "LOGREG": "#eda100",  # yellow
    "XGB": "#e87ba4",     # magenta
}
# Secondary encoding, so the series stay separable in greyscale and for readers with
# colour-vision deficiency.
NOISE_MODEL_MARKERS = {
    "KNN": "o",
    "SVM": "s",
    "RF": "^",
    "LOGREG": "D",
    "XGB": "v",
}

# One-hue light->dark ramp for the magnitude (retention) heatmap. A single hue, never a
# rainbow: the quantity being shown is "how much survived", which has no midpoint and
# no two opposing directions.
NOISE_SEQUENTIAL_BLUE = (
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
    "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b",
)

NOISE_SURFACE = "#fcfcfb"
NOISE_INK = "#0b0b0b"
NOISE_INK_MUTED = "#52514e"
NOISE_GRID = "#e8e7e4"


NOISE_METRIC_LABELS = {
    "accuracy": "Accuracy",
    "precision_macro": "Macro precision",
    "recall_macro": "Macro recall",
    "f1_macro": "Macro F1",
}


def noise_metric_label(metric: str) -> str:
    return NOISE_METRIC_LABELS.get(metric, metric.replace("_", " "))

In [ ]:
def _style_noise_axes(ax) -> None:
    """Recessive grid and axes: the data marks should be the only strong thing."""
    ax.set_axisbelow(True)
    ax.grid(True, color=NOISE_GRID, linewidth=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(NOISE_GRID)
    ax.tick_params(colors=NOISE_INK_MUTED, length=0)


def _contrast_ratio(rgb: tuple[float, float, float], other: str) -> float:
    """WCAG contrast between an RGB triple and a hex colour."""
    def luminance(channels):
        c = np.asarray(channels, dtype=float)
        linear = np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
        return float(0.2126 * linear[0] + 0.7152 * linear[1] + 0.0722 * linear[2])

    other_rgb = [int(other.lstrip("#")[j : j + 2], 16) / 255 for j in (0, 2, 4)]
    high, low = sorted((luminance(rgb), luminance(other_rgb)), reverse=True)
    return (high + 0.05) / (low + 0.05)


def _ink_on(rgb: tuple[float, float, float]) -> str:
    """Pick the annotation ink with the most contrast against a filled cell.

    Measured rather than guessed from the cell's position in the ramp: the two are
    only loosely related, and the mid-ramp cells - where a fixed threshold is most
    likely to pick wrong - are exactly the ones where the margin is thinnest.
    """
    return max(
        (NOISE_INK, NOISE_SURFACE), key=lambda ink: _contrast_ratio(rgb, ink)
    )

In [ ]:
def plot_noise_degradation(
    results: pd.DataFrame,
    metric: str = "f1_macro",
    noise_types: tuple[str, ...] = NOISE_TYPES,
    models: list[str] | None = None,
):
    """One panel per noise type; one line per model; shared y-axis.

    Small multiples rather than one crowded axes, and a shared y-scale so the panels
    can be compared against each other by height. Every curve starts from the model's
    clean score at 0%, which is what makes the *shape* of the decline readable - the
    non-linearity Abdolazimi et al. (2024) describe would be invisible from a single
    noise level.
    """
    available = set(results["model"])
    models = (
        [m for m in NOISE_MODEL_ORDER if m in available] if models is None else models
    )
    clean = results[results["scenario"] == NOISE_CLEAN_SCENARIO].set_index("model")[metric]

    figure, axes = plt.subplots(
        1, len(noise_types), figsize=(5.2 * len(noise_types), 4.8), sharey=True
    )
    axes = np.atleast_1d(axes)

    for ax, noise_type in zip(axes, noise_types):
        subset = results[results["noise_type"] == noise_type]
        for model_key in models:
            model_rows = subset[subset["model"] == model_key].sort_values("intensity")
            x = [0.0] + (model_rows["intensity"] * 100).tolist()
            y = [clean.get(model_key, np.nan)] + model_rows[metric].tolist()
            ax.plot(
                x,
                y,
                label=model_key,
                color=NOISE_MODEL_COLORS[model_key],
                marker=NOISE_MODEL_MARKERS[model_key],
                markersize=8,
                markeredgecolor=NOISE_SURFACE,
                markeredgewidth=1.5,
                linewidth=2,
            )

        ax.set_title(noise_type.capitalize(), color=NOISE_INK)
        ax.set_xlabel("Corrupted records (%)", color=NOISE_INK_MUTED)
        ax.set_xticks([0] + [i * 100 for i in NOISE_INTENSITIES])
        _style_noise_axes(ax)

    axes[0].set_ylabel(noise_metric_label(metric), color=NOISE_INK_MUTED)

    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        loc="lower center",
        ncol=len(labels),
        frameon=False,
        labelcolor=NOISE_INK,
    )
    figure.suptitle(
        f"{noise_metric_label(metric)} under feature noise"
        " - CSE-CIC-IDS2018 test split",
        color=NOISE_INK,
    )
    figure.tight_layout(rect=(0, 0.07, 1, 1))
    plt.show()

In [ ]:
plot_noise_degradation(noise_results, "f1_macro")

In [ ]:
def plot_noise_retention_heatmap(
    degradation: pd.DataFrame, metric: str = "f1_macro", vmin: float | None = None
):
    """`model` x `scenario` grid of retained performance.

    `vmin` defaults to the smallest value in the data rather than to 0, so the ramp
    spends its whole range on the interval the results actually occupy; every cell is
    also printed, so the exact figure never depends on reading the colour.
    """
    from matplotlib.colors import LinearSegmentedColormap

    column = f"{metric}_retention"
    order = [scenario for _, _, scenario in get_noise_scenarios()]
    table = degradation.pivot_table(index="model", columns="scenario", values=column)
    table = table.reindex(
        index=[m for m in NOISE_MODEL_ORDER if m in table.index],
        columns=[s for s in order if s in table.columns],
    )

    values = table.to_numpy(dtype=float)
    low = float(np.nanmin(values)) if vmin is None else vmin
    high = float(np.nanmax(values))
    span = max(high - low, 1e-9)

    cmap = LinearSegmentedColormap.from_list("mlids_blue", NOISE_SEQUENTIAL_BLUE)

    figure, ax = plt.subplots(figsize=(1.15 * table.shape[1] + 3, 0.7 * table.shape[0] + 2.6))
    image = ax.imshow(values, cmap=cmap, vmin=low, vmax=high, aspect="auto")

    ax.set_xticks(range(table.shape[1]), table.columns, rotation=45, ha="right")
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.tick_params(colors=NOISE_INK_MUTED, length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # 2px surface gap between cells, drawn as a minor grid in the surface colour.
    ax.set_xticks(np.arange(-0.5, table.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, table.shape[0], 1), minor=True)
    ax.grid(which="minor", color=NOISE_SURFACE, linewidth=2)
    ax.tick_params(which="minor", length=0)

    for row in range(table.shape[0]):
        for col in range(table.shape[1]):
            value = values[row, col]
            if np.isnan(value):
                continue
            ax.text(
                col,
                row,
                f"{value:.3f}",
                ha="center",
                va="center",
                # Ink, not the ramp colour, and whichever of the two inks
                # actually has more contrast against this particular cell.
                color=_ink_on(cmap((value - low) / span)[:3]),
                fontsize=9,
            )

    bar = figure.colorbar(image, ax=ax)
    bar.set_label(
        f"retained {noise_metric_label(metric)} (noisy / clean)", color=NOISE_INK_MUTED
    )
    bar.ax.tick_params(colors=NOISE_INK_MUTED, length=0)
    bar.outline.set_visible(False)

    ax.set_title(
        f"Share of clean {noise_metric_label(metric)} retained under feature noise",
        color=NOISE_INK,
    )
    figure.tight_layout()
    plt.show()

In [ ]:
plot_noise_retention_heatmap(noise_degradation, "f1_macro")

### Robustness Ranking

This is the table that answers the research question in Section 1.3. It is ordered by
**mean retention**, not by absolute score: the model at the top is the one that gives
up the least of what it had, which is a different question from which model is most
accurate on clean traffic - and `f1_macro_clean` is kept in view precisely so the two
can be read against each other.

`worst_retention` and `worst_scenario` guard against a model that averages well by
coping with two noise types and collapsing under the third; for an operational
recommendation the floor usually matters more than the mean.

In [ ]:
def rank_noise_robustness(
    degradation: pd.DataFrame, metric: str = "f1_macro"
) -> pd.DataFrame:
    """Per-model robustness summary, most robust first."""
    column = f"{metric}_retention"

    ranking = degradation.groupby("model").agg(
        **{
            f"{metric}_clean": (f"{metric}_clean", "first"),
            "mean_retention": (column, "mean"),
            "worst_retention": (column, "min"),
            "mean_drop": (f"{metric}_drop", "mean"),
            "worst_drop": (f"{metric}_drop", "max"),
        }
    )

    worst_rows = degradation.loc[degradation.groupby("model")[column].idxmin()]
    ranking["worst_scenario"] = worst_rows.set_index("model")["scenario"]

    ranking = ranking.sort_values("mean_retention", ascending=False)
    ranking.insert(0, "rank", range(1, len(ranking) + 1))
    return ranking

In [ ]:
noise_ranking = rank_noise_robustness(noise_degradation)
display(noise_ranking.round(4))
noise_ranking.to_csv(
    os.path.join(PATH_FOLDER_SEARCH_RESULT, "noise-robustness-ranking.csv")
)
print("Saved: hyperparameter-search/noise-robustness-ranking.csv")

### Per-class behaviour under the harshest scenario

The macro F1 above is an average over 14 classes that are wildly unequal in size, so a
drop in it does not say *where* the model broke. This pulls the full per-class report
and confusion matrix for one (model, scenario) pair, the same way each model's clean
"Report" subsection does - the usual thing to look for is whether the loss is spread
evenly or concentrated in the rare attack classes, which are the ones an IDS is
deployed to catch.

In [ ]:
def get_noise_classification_report(
    model_key: str,
    scenario: str,
    split_name: str = "test",
    labels_folder: str = PATH_FOLDER_ENCODED_LABEL,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """`(classification report, confusion matrix)` for one (model, scenario) pair."""
    folder = (
        os.path.join(PATH_FOLDER_PREDICTION_RESULT, model_key, split_name)
        if scenario == NOISE_CLEAN_SCENARIO
        else get_noise_prediction_folder(model_key, scenario)
    )
    pred, true = load_predictions_and_labels(folder, split_name, labels_folder)
    return (
        get_classification_report(pred=pred, true=true),
        get_confusion_matrix(pred=pred, true=true),
    )

In [ ]:
report, cm = get_noise_classification_report("RF", "multiplicative-30")
display(report)
display(cm)